<a href="https://colab.research.google.com/github/DianeLaourou/5th_hsaf_user_Workshop/blob/main/Bulletin_HEBDO_IBF_VF_corrige.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌦 MÉTÉO BÉNIN — Cartes de Vigilance
## Notebook unique · IBF/OMM · ECMWF Dissémination ECPDS (0.1°) — repli Open Data

### ⚠️ Prérequis : deux shapefiles
> **Communes/départements** (obligatoire, colonnes commune+dept) : `ben_admin2.shp/.dbf/.shx/.prj/.cpg`
> **Arrondissements** (optionnel, contours de repère uniquement) : ZIP geoBoundaries ADM3 — demandé séparément en C4

| Cellule | Rôle |
|---------|------|
| **C1** | ⚙️ CONFIG — **seule cellule à modifier chaque matin** |
| C2 | 📦 Installation packages |
| C3 | 📚 Imports + référentiel OMM/IBF |
| **C4** | 🗺️ Chargement shapefile OCHA (77 communes) + arrondissements + logo |
| **C5** | 🌐 ECMWF Dissémination ECPDS (0.1°, compte wmo_bj) — repli Open Data|
| **C6bis** | 📐 Vigilance sur grille (raster 0.1°) — repli commune si grille indisponible |
| **C7** | 📊 Calcul vigilance IBF + 🖱️ Ajustement interactif par département |
| **C8** | 🗺️📊 Cartes vigilance + Météorogrammes éditables |
| **C9** | 🌍 Carte de synthèse nationale (grille jour par jour + commentaire auto) |
| **C9bis** | 🌧️ Carte de cumul pluviométrique attendu (légende continue mm) |
| **C9quater** | 🛰️ Cumul observé IMERG, 7 derniers jours (automatisé, `earthaccess`) |
| **C9ter** | 🔎 Cartes zoom par commune ≥ seuil configurable (arrondissements en évidence) |
| **C10** | 💾 Téléchargement (ZIP) |
| **C11** | 📊 Remplissage du canevas PowerPoint validé (Météo-Bénin & ABPC) — nécessite le fichier CANEVAS |
| C3bis | 📈 Comparaison observations AWS vs prévisions (log de biais) — outil annexe, en fin de notebook, sans dépendance avec les cellules ci-dessus |

**Workflow :** Upload shapefiles → C1 → `Ctrl+F9` (C1 à C11 dans l'ordre) → ajuster C7 si besoin → C10/C11. C3bis est indépendante, à lancer quand tu veux comparer aux observations.

---
🟢 **V1** Faible risque· 🟡 **V2** Soyez attentif · 🟠 **V3** Prenez vos précautions · 🔴 **V4** Exécutez les consignes


In [1]:
import datetime

# ║═════════════════════════════════════
# C1 ─ CONFIGURATION PRINCIPALE
# Modifier uniquement cette cellule chaque matin.
# ║═══════════════════════════════════════

global CONFIG # Ensure CONFIG is a global variable
CONFIG = {
    # ─ Période du produit ══════════════════════════
    # La fenêtre de validité (7 jours) démarre toujours au prochain
    # 'jour_reference' à 00h ─ que tu lances le script ce jour-là ou
    # la veille (le calcul avance automatiquement, rien à changer).
    # 'vendredi' = bulletin hebdomadaire principal
    # 'mercredi' = actualisation mi-semaine
    'jour_reference': 'jeudi',
    'heure_debut': '00:00',
    'nb_jours_cartes': 7,

    # ─ ECMWF Open Data (complément de queue + repli total uniquement) ─
    # Ne s'applique PAS à ECPDS (source prioritaire, 0.1°) qui détecte
    # toujours seul son dernier run disponible par listing du serveur.
    # Laisser 'auto' pour prendre le dernier run Open Data disponible
    # (avec restriction automatique aux runs 00Z/12Z si des échéances
    # longues, >48h, sont nécessaires ─ seuls ces runs vont jusqu'à 240h).
    # Mettre '20260601' et '00', '06', '12' ou '18' si tu veux forcer un run précis.
    'ecmwf_date': '20260924',         # 'auto' ou YYYYMMDD
    'ecmwf_run': 'auto',          # 'auto', '00', '06', '12' ou '18'
    'pas_h': 3,                   # pas de temps : 3h conseillé
    'horizon_h': 192,             # horizon téléchargé ; marge pour couvrir un lancement la veille + les 7 jours de la période (192h = 8 jours)

    # ─ Sorties ═══════════════════════════
    'output_dir': '/content/cartes_vigilance',
    'icones_dir': '/content',          # dossier des icone_*.png uploadés
    'organisme': 'MÉTÉO BÉNIN',

    # ─ Cartes de vigilance ════════════════════
    'afficher_noms_villes': True,          # noms des 12 villes légèrement sur les cartes
    'afficher_pictogrammes': True,         # symboles météo noirs sur les cartes
    'generer_version_sans_pictos': False,  # mettre True pour produire aussi une carte sans symbole
    'taille_picto': 0.105,                 # taille des symboles sur carte ; ajuster entre 0.09 et 0.16
    'cartes_sans_titre_legende': True,     # True = carte seule, mieux pour insertion dans Word
    'carte_dpi': 300,                      # résolution élevée pour Word
    'taille_police_villes': 4.9,          # police des noms de villes sur la carte
    'taille_police_dept': 5.3,            # police des noms de départements
    'logo_zoom_general': 0.035,           # Zoom du logo sur les cartes pleine page (C8, C9bis, C9quater)
    'logo_zoom_synthese': 0.028,           # Zoom du logo sur les mini-cartes de C9 (plus petites : zoom réduit pour garder les proportions)
    'logo_zoom_focus': 0.03,              # Zoom du logo sur les cartes focus commune (C9ter)

    # ─ Pages focus commune (C9ter) ═════════════════
    # Niveau de vigilance mini pour qu'une commune ait sa propre page zoom
    # dans le PPT. 2 = Modéré, 3 = Élevé/Grave, 4 = Très sévère. Relevé à 3
    # par défaut (trop de pages générées à 2 sur une semaine chargée).
    'seuil_focus_commune': 3,

    # ─ Météorogrammes ════════════════════
    'heure_locale_offset_h': 1,            # Bénin = UTC+1 pour l'affichage ECMWF
    'meteogramme_pas_h': 6,                # affichage de 6h en 6h
    'meteogramme_fin_jour_offset': 4,      # 0=lundi, 1=mardi, 2=mercredi, 3=jeudi, 4=vendredi (for 5 days total)
    'meteogramme_fin_heure_locale': 19,    # fin : jeudi 19h locale

    # ─ Villes de référence : 1 ville par département ─════════════
    # Format : (Ville, Département, latitude, longitude)
    # Aplahoué est conservée pour le Couffo, car c'est le centre de lancement.
    'villes': [
        ('Kandi',         'Alibori',     11.13,  2.94),
        ('Natitingou',    'Atacora',     10.30,  1.38),
        ('Abomey-Calavi', 'Atlantique',   6.45,  2.35),
        ('Parakou',       'Borgou',       9.35,  2.63),
        ('Dassa-Zoumè',   'Collines',     7.75,  2.18),
        ('Aplahoué',      'Couffo',       6.93,  1.67),
        ('Djougou',       'Donga',        9.71,  1.67),
        ('Cotonou',       'Littoral',     6.37,  2.42),
        ('Lokossa',       'Mono',         6.64,  1.72),
        ('Porto-Novo',    'Ouémé',        6.50,  2.61),
        ('Pobè',          'Plateau',      6.98,  2.66),
        ('Bohicon',       'Zou',          7.18,  2.07),
    ],

    # ─ Villes supplémentaires pour l'affichage cartographique ─══
    # Deux villes par département, sauf le Littoral qui garde seulement Cotonou.
    # Format : (ville, département, latitude, longitude)
    'villes_carte_supplementaires': [
        ('Malanville',    'Alibori',     11.87,  3.38),
        ('Tanguiéta',     'Atacora',     10.62,  1.27),
        ('Allada',        'Atlantique',   6.67,  2.15),
        ('Nikki',         'Borgou',       9.94,  3.21),
        ('Savè',          'Collines',     8.03,  2.49),
        ('Dogbo',         'Couffo',       6.80,  1.78),
        ('Bassila',       'Donga',        9.01,  1.67),
        ('Comè',          'Mono',         6.41,  1.88),
        ('Adjarra',       'Ouémé',        6.53,  2.66),
        ('Kétou',         'Plateau',      7.36,  2.61),
        ('Abomey',        'Zou',          7.18,  1.99),

        # ─ Communes nord/centre ajoutées pour lisibilité (repère demandé) ─
        # Coordonnées approximatives (chef-lieu de commune) ─ à corriger si
        # une position plus précise est disponible.
        ('Banikoara',     'Alibori',     11.30,  2.44),
        ('Ségbana',       'Alibori',     11.20,  3.42),
        ('Cobly',         'Atacora',     10.35,  0.83),
        ('Porga',         'Atacora',     11.05,  0.93),
        ('Kérou',         'Atacora',     10.80,  2.02),
        ('Bembéréké',     'Borgou',      10.23,  2.67),
        ('Banté',         'Collines',     8.42,  1.83),
    ],
}

# ─ Corrections expertisées par le prévisionniste ─════════════════
# Ces ajustements remplacent ou corrigent le calcul automatique ECMWF.
# Format : 'Département' : [J1, J2, J3, J4, J5]
# Valeurs de vigilance :
#   0 = aucune couleur / pas de risque significatif
#   1 = faible risque / vert léger
#   2 = risque modéré / jaune léger
#   3 = risque élevé / orange léger
#   4 = risque très élevé / rouge léger
# Mettre None pour ne pas modifier une journée.
AJUST_DEPT = {
    # Exemple : 'Littoral': [2, 3, None, 1, 0],
}

# Même logique au niveau commune, si nécessaire.
AJUST_COMMUNE = {
    # Exemple : 'Cotonou': [3, 3, 2, 1, 0],
}

# ─ Ajustements des symboles météo par le prévisionniste ─════
# Symboles autorisés : 'pluie', 'orage', 'chaleur', 'vent', 'aucun', 'auto'
#   None    = ne pas modifier la proposition automatique
#   'auto'  = revenir à la proposition automatique
#   'aucun' = forcer l'absence de symbole
# Format : 'Département' : [J1, J2, J3, J4, J5]
AJUST_PICTO_DEPT = {
    # Exemple : 'Couffo': ['orage', None, 'pluie', 'aucun', 'auto'],
}

# Même logique au niveau commune, si nécessaire.
AJUST_PICTO_COMMUNE = {
    # Exemple : 'Aplahoué': ['orage', 'pluie', None, 'aucun', 'auto'],
}


# ─ Ajustement des valeurs de météogrammes par les prévisionnistes ─═
# Les dates ci-dessous sont en HEURE LOCALE DU BÉNIN.
# Format : 'Ville': {'YYYY-MM-DD HH:MM': {'temperature_c': ..., 'pluie_mm': ..., 'humidite_pct': ...}}
# Les clés disponibles sont : temperature_c, pluie_mm, humidite_pct.
# Si une valeur n'est pas indiquée, la valeur ECMWF est conservée.
AJUST_METEOGRAMMES = {
    # Exemple :
    # 'Cotonou': {
    #     '2026-06-01 13:00': {'temperature_c': 31.0, 'pluie_mm': 8.0, 'humidite_pct': 82},
    #     '2026-06-01 19:00': {'pluie_mm': 15.0},
    # },
}

print('✅ Configuration chargée')
print(f"   Période : prochain {CONFIG['jour_reference']} {CONFIG['heure_debut']} → +{CONFIG['nb_jours_cartes']} jours")
print(f"   ECMWF   : date={CONFIG['ecmwf_date']} | run={CONFIG['ecmwf_run']} | pas={CONFIG['pas_h']}h")
print(f"   Villes  : {len(CONFIG['villes'])} villes de référence, une par département")

✅ Configuration chargée
   Période : prochain jeudi 00:00 → +7 jours
   ECMWF   : date=20260924 | run=auto | pas=3h
   Villes  : 12 villes de référence, une par département


In [2]:
# ════════════════════════════════════════════════════════════════
# C2 — INSTALLATION DES PACKAGES
# ════════════════════════════════════════════════════════════════

import subprocess, sys

pkgs = [
    'ecmwf-opendata',
    'cfgrib',
    'xarray',
    'geopandas',
    'shapely',
    'ipywidgets'
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('✅ Packages installés')


✅ Packages installés


In [3]:
import os, re, io, zipfile, shutil, warnings, datetime, requests, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from shapely.geometry import Polygon, MultiPolygon

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'savefig.dpi': 200,
})

# ── Période officielle du produit : alignée automatiquement sur C1 (ecmwf_date) ──
_date_str = str(CONFIG['ecmwf_date'])
if _date_str == 'auto':
    _date_parsed = datetime.datetime.now().date()
    _date_str = _date_parsed.strftime('%Y%m%d')
else:
    _date_parsed = datetime.datetime.strptime(_date_str, '%Y%m%d').date()

LOCAL_UTC_OFFSET = pd.Timedelta(hours=CONFIG.get('heure_locale_offset_h', 1))
PERIODE_DEBUT = pd.Timestamp(f"{_date_parsed} {CONFIG['heure_debut']}")
PERIODE_FIN = PERIODE_DEBUT + pd.Timedelta(days=CONFIG['nb_jours_cartes'])
PERIODE_DEBUT_UTC = PERIODE_DEBUT - LOCAL_UTC_OFFSET
PERIODE_FIN_UTC = PERIODE_FIN - LOCAL_UTC_OFFSET

print(f"📅 Période locale : {PERIODE_DEBUT:%d/%m/%Y %Hh} → {PERIODE_FIN:%d/%m/%Y %Hh}")
print(f"   Équivalent UTC : {PERIODE_DEBUT_UTC:%d/%m/%Y %Hh} → {PERIODE_FIN_UTC:%d/%m/%Y %Hh}")

JOURS = []
JOURS_FR = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
MOIS_FR = ['janvier','février','mars','avril','mai','juin','juillet','août','septembre','octobre','novembre','décembre']
for i in range(CONFIG['nb_jours_cartes']):
    d0 = PERIODE_DEBUT + pd.Timedelta(days=i)
    d1 = d0 + pd.Timedelta(days=1)
    label = f"{JOURS_FR[d0.weekday()]} {d0.day} {MOIS_FR[d0.month-1]}"
    JOURS.append({'idx': i, 'debut': d0, 'fin': d1, 'label': label})

# ── Couleurs de vigilance légères ──
VIG = {
    0: {'code': 'Aucun',    'label': 'Pas de risque',        'hex': '#FFFFFF', 'bord': '#B0BEC5'},
    1: {'code': 'Faible',    'label': 'Faible risque',        'hex': '#4DD0E1', 'bord': '#00838F'},
    2: {'code': 'Modéré',    'label': 'Risque modéré',        'hex': '#FFE800', 'bord': '#C9A600'},
    3: {'code': 'Élevé',     'label': 'Risque élevé (grave)', 'hex': '#FF9800', 'bord': '#E65100'},
    4: {'code': 'Très élevé','label': 'Très sévère',          'hex': '#E53935', 'bord': '#B71C1C'},
}
VIG_HEX = {k: v['hex'] for k, v in VIG.items()}
VIG_BORD = {k: v['bord'] for k, v in VIG.items()}

# ── Palette cumul unique (mm) ──
CUMUL_BORNES_MM = [0, 10, 25, 50, 75, 100, 125, 150, 200]
CUMUL_COULEURS  = ['#FFFFFF', '#BFE3F7', '#6FB8E8', '#2E7BC4',
                   '#1B4C8C', '#F2A354', '#D9622B', '#B0201F', '#4A0000']

def cumul_cmap_norm():
    from matplotlib.colors import ListedColormap, BoundaryNorm
    cmap = ListedColormap(CUMUL_COULEURS)
    bornes = CUMUL_BORNES_MM + [CUMUL_BORNES_MM[-1] + 1000]
    norm = BoundaryNorm(bornes, cmap.N)
    return cmap, norm

PICTO_LABELS = {
    None: 'Aucun symbole',
    'aucun': 'Aucun symbole',
    'pluie': 'Pluie',
    'orage': 'Orage',
    'chaleur': 'Chaleur',
    'vent': 'Vent',
}

# ── Normalisation des noms ──
def norm_txt(x):
    if x is None:
        return ''
    s = str(x).lower().replace('-', '').replace(' ', '').replace('_', '').replace("'", '')
    return ''.join(ch for ch in unicodedata.normalize('NFD', s)
                   if unicodedata.category(ch) != 'Mn')

DEPT_ALIASES = {
    'oueme': 'Ouémé', 'couffo': 'Couffo', 'kouffo': 'Couffo',
    'atacora': 'Atacora', 'atakora': 'Atacora', 'plateau': 'Plateau',
    'collines': 'Collines', 'borgou': 'Borgou', 'alibori': 'Alibori',
    'donga': 'Donga', 'zou': 'Zou', 'mono': 'Mono',
    'atlantique': 'Atlantique', 'littoral': 'Littoral',
}

def canon_dept(x):
    return DEPT_ALIASES.get(norm_txt(x), str(x).strip())

def normaliser_picto(val):
    if val is None:
        return None
    v = norm_txt(val)
    if v in ['', 'none', 'nan']:
        return None
    if v in ['aucun', 'sans', 'pas de symbole', 'pas symbole', '0']:
        return 'aucun'
    if v in ['auto', 'automatique']:
        return 'auto'
    if v in ['pluie', 'rain', 'averse', 'averses']:
        return 'pluie'
    if v in ['orage', 'orages', 'thunderstorm', 'ts']:
        return 'orage'
    if v in ['chaleur', 'temperature', 'température', 'heat', 'chaud']:
        return 'chaleur'
    if v in ['vent', 'wind', 'rafale', 'rafales']:
        return 'vent'
    return None

# ── Seuils pluie, température, vent ──
def niveau_pluie(pluie_24h):
    pluie_24h = 0.0 if pd.isna(pluie_24h) else max(float(pluie_24h), 0.0)
    if pluie_24h < 10.0: return 0
    elif pluie_24h < 25.0: return 1
    elif pluie_24h < 50.0: return 2
    elif pluie_24h < 75.0: return 3
    else: return 4

SEUILS_CHALEUR = [33.0, 35.0, 38.0, 40.0]

def niveau_temperature(temp_ressenti):
    if pd.isna(temp_ressenti): return 0
    s1, s2, s3, s4 = SEUILS_CHALEUR
    temp_ressenti = float(temp_ressenti)
    if temp_ressenti < s1: return 0
    elif temp_ressenti < s2: return 1
    elif temp_ressenti < s3: return 2
    elif temp_ressenti < s4: return 3
    else: return 4

def niveau_vent(vent_kmh):
    vent_kmh = 0.0 if pd.isna(vent_kmh) else max(float(vent_kmh), 0.0)
    if vent_kmh <= 30.0: return 0
    elif vent_kmh < 45.0: return 1
    elif vent_kmh < 60.0: return 2
    elif vent_kmh < 75.0: return 3
    else: return 4

def niveau_vigilance(pluie_24h, temp_ressenti, vent_kmh=None):
    niveaux = [niveau_pluie(pluie_24h), niveau_temperature(temp_ressenti)]
    if vent_kmh is not None:
        niveaux.append(niveau_vent(vent_kmh))
    return max(niveaux)

def picto_auto_jour(df_jour, pluie_24h, tmax, departement):
    if df_jour is None or df_jour.empty:
        return None
    pluie_max_pas = float(df_jour['tp_mm'].max()) if 'tp_mm' in df_jour else 0.0
    vent_max = float(df_jour['wind_kmh'].max()) if 'wind_kmh' in df_jour and not df_jour['wind_kmh'].isna().all() else 0.0
    niv_temp = niveau_temperature(tmax)

    if pluie_24h >= 15.0 or pluie_max_pas >= 7.5: return 'orage'
    if pluie_24h > 5.0: return 'pluie'
    if vent_max >= 35.0: return 'vent'
    if niv_temp >= 1: return 'chaleur'
    return None

# ── UPLOAD DES PICTOGRAMMES ──
print("📂 Veuillez choisir les fichiers PNG des pictogrammes...")
from google.colab import files
uploaded_pictograms = files.upload()
if uploaded_pictograms:
    for fname, content in uploaded_pictograms.items():
        with open(Path('/content') / fname, 'wb') as f:
            f.write(content)
    print("✅ Pictogrammes chargés avec succès !\n")
else:
    print("⚠️ Aucun pictogramme n'a été chargé.\n")

os.makedirs(CONFIG['output_dir'], exist_ok=True)
print('✅ Imports, période, seuils et pictogrammes prêts')
print(f'   Fenêtre météogrammes : {PERIODE_DEBUT} → {PERIODE_FIN}')
for j in JOURS:
    print(f"   J{j['idx']+1} : {j['label']} | {j['debut']} → {j['fin']}")

📅 Période locale : 24/09/2026 00h → 01/10/2026 00h
   Équivalent UTC : 23/09/2026 23h → 30/09/2026 23h
📂 Veuillez choisir les fichiers PNG des pictogrammes...


Saving icone-chaleur.png to icone-chaleur.png
Saving icone-pluie.png to icone-pluie.png
Saving icone-vent.png to icone-vent.png
✅ Pictogrammes chargés avec succès !

✅ Imports, période, seuils et pictogrammes prêts
   Fenêtre météogrammes : 2026-09-24 00:00:00 → 2026-10-01 00:00:00
   J1 : Jeudi 24 septembre | 2026-09-24 00:00:00 → 2026-09-25 00:00:00
   J2 : Vendredi 25 septembre | 2026-09-25 00:00:00 → 2026-09-26 00:00:00
   J3 : Samedi 26 septembre | 2026-09-26 00:00:00 → 2026-09-27 00:00:00
   J4 : Dimanche 27 septembre | 2026-09-27 00:00:00 → 2026-09-28 00:00:00
   J5 : Lundi 28 septembre | 2026-09-28 00:00:00 → 2026-09-29 00:00:00
   J6 : Mardi 29 septembre | 2026-09-29 00:00:00 → 2026-09-30 00:00:00
   J7 : Mercredi 30 septembre | 2026-09-30 00:00:00 → 2026-10-01 00:00:00


In [5]:
import os
import shutil
from pathlib import Path
import geopandas as gpd
import unicodedata
import re
import zipfile
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from google.colab import files
from shapely.geometry import Polygon, MultiPolygon

# ================================================================
# 0. ÉTAPE PRÉALABLE : CRÉATION DU CSV DE RÉFÉRENCE (546 ARRONDISSEMENTS)
# ================================================================

DATA_BENIN = [
    ("Alibori", "Banikoara", ["Banikoara", "Founougo", "Gomparou", "Goumori", "Kokey", "Kokiborou", "Ounet", "Sompérékou", "Soroko", "Toura"]),
    ("Alibori", "Gogounou", ["Bagou", "Gogounou", "Gounarou", "Ouara", "Sori", "Zoungou-Pantrossi"]),
    ("Alibori", "Kandi", ["Angaradébou", "Bensékou", "Donwari", "Kandi I", "Kandi II", "Kandi III", "Kassakou", "Saah", "Sam", "Sonsoro"]),
    ("Alibori", "Karimama", ["Birni Lafia", "Bogo-Bogo", "Karimama", "Kompa", "Monsey"]),
    ("Alibori", "Malanville", ["Garou", "Guéné", "Malanville", "Madécali", "Toumboutou"]),
    ("Alibori", "Ségbana", ["Libantè", "Liboussou", "Lougou", "Ségbana", "Sokotindji"]),
    ("Atacora", "Boukoumbé", ["Boukoumbé", "Dipoli", "Korontière", "Kossoucoingou", "Manta", "Natta", "Tabota"]),
    ("Atacora", "Cobly", ["Cobly", "Datori", "Kountori", "Tapoga"]),
    ("Atacora", "Kérou", ["Brignamaro", "Firou", "Kérou", "Koabagou"]),
    ("Atacora", "Kouandé", ["Birni", "Chabi-Couma", "Fô-Tancé", "Guilmaro", "Kouandé", "Oroukayo"]),
    ("Atacora", "Matéri", ["Dassari", "Gouandé", "Matéri", "Nodi", "Tantéga", "Tchianhoun-Cossi"]),
    ("Atacora", "Natitingou", ["Kotopounga", "Kouaba", "Koundata", "Natitingou I", "Natitingou II", "Natitingou III", "Natitingou IV", "Perma", "Tchoumi-Tchoumi"]),
    ("Atacora", "Péhunco", ["Gnémasson", "Péhunco", "Tobré"]),
    ("Atacora", "Tanguiéta", ["Cotiakou", "N'Dahonta", "Taiakou", "Tanguiéta", "Tanongou"]),
    ("Atacora", "Toucountouna", ["Kouarfa", "Tampégré", "Toucountouna"]),
    ("Atlantique", "Abomey-Calavi", ["Abomey-Calavi", "Akassato", "Godomey", "Glo-Djigbé", "Hêvié", "Kpanroun", "Ouèdo", "Togba", "Zinvié"]),
    ("Atlantique", "Allada", ["Agbanou", "Ahouannonzoun", "Allada", "Attogon", "Avakpa", "Ayou", "Hinvi", "Lissègazoun", "Lon-Agonmey", "Sékou", "Togoudo", "Tokpa-Avagoudo"]),
    ("Atlantique", "Kpomassè", ["Aganmalomè", "Agbanto", "Agonkanmè", "Dédomè", "Dékanmè", "Kpomassè", "Sègbèya", "Sègbohouè", "Tokpa-Domè"]),
    ("Atlantique", "Ouidah", ["Avlékété", "Djègbadji", "Gakpé", "Houakpè-Daho", "Ouidah I", "Ouidah II", "Ouidah III", "Ouidah IV", "Pahou", "Savi"]),
    ("Atlantique", "Sô-Ava", ["Ahomey-Lokpo", "Dékanmey", "Ganvié I", "Ganvié II", "Houédo-Aguékon", "Sô-Ava", "Vekky"]),
    ("Atlantique", "Toffo", ["Agué", "Colli-Agbamè", "Coussi", "Damè", "Djanglanmè", "Houègbo", "Kpomè", "Sè", "Sèhouè", "Toffo-Agué"]),
    ("Atlantique", "Tori-Bossito", ["Avamè", "Azohouè-Aliho", "Azohouè-Cada", "Tori-Bossito", "Tori-Cada", "Tori-Gare", "Tori-Aïdohouè", "Tori-Acadjamè"]),
    ("Atlantique", "Zè", ["Adjan", "Dawè", "Djigbé", "Dodji-Bata", "Hèkanmé", "Koundokpoé", "Sèdjè-Dénou", "Sèdjè-Houégoudo", "Tangbo-Djèvié", "Yokpo", "Zè"]),
    ("Borgou", "Bembèrèkè", ["Bembèrèkè", "Béroubouay", "Bouanri", "Gomia", "Ina"]),
    ("Borgou", "Kalalé", ["Basso", "Bouka", "Dèrassi", "Dunkassa", "Kalalé", "Péonga"]),
    ("Borgou", "N'Dali", ["Bori", "Gbégourou", "N'Dali", "Ouénou", "Sirarou"]),
    ("Borgou", "Nikki", ["Biro", "Gnonkourakali", "Nikki", "Ouénou", "Sérékalé", "Suya", "Tasso"]),
    ("Borgou", "Parakou", ["1er arrondissement de Parakou", "2e arrondissement de Parakou", "3e arrondissement de Parakou"]),
    ("Borgou", "Pèrèrè", ["Gninsy", "Guinagourou", "Kpané", "Pébié", "Pèrèrè", "Sontou"]),
    ("Borgou", "Sinendé", ["Fô-Bourè", "Sèkèrè", "Sikki", "Sinendé"]),
    ("Borgou", "Tchaourou", ["Alafiarou", "Bétérou", "Goro", "Kika", "Sanson", "Tchaourou", "Tchatchou"]),
    ("Collines", "Bantè", ["Agoua", "Akpassi", "Atokoligbé", "Bantè", "Bobè", "Gouka", "Kloué", "Kpassa", "Pira"]),
    ("Collines", "Dassa-Zoume", ["Akofodjoulè", "Dassa I", "Dassa II", "Gbaffo", "Kpingni", "Lèma", "Paouignan", "Soclogbo", "Tre"]),
    ("Collines", "Glazoué", ["Assanté", "Glazoué", "Gomè", "Kpakpaza", "Magoumi", "Ouèdèmè", "Sokponta", "Tchatchagou", "Zaffé"]),
    ("Collines", "Ouèssè", ["Challa-Ogo", "Djègbè", "Gbanlin", "Ikèmon", "Kilibo", "Laminou", "Odouba", "Ouèssè", "Toui"]),
    ("Collines", "Savalou", ["Djaloukou", "Doumè", "Gobada", "Kpakpassa", "Lahotan", "Lissa", "Monkpa", "Ottola", "Ouèssè", "Savalou-Agbado", "Savalou-Aga", "Savalou-Attakè", "Tchetti", "Tchitchagou"]),
    ("Collines", "Savè", ["Adido", "Bessé", "Bori", "Kaboua", "Offè", "Okpara", "Plateau", "Savè"]),
    ("Couffo", "Aplahoue", ["Aplahoué", "Atomè", "Azovè", "Dekpo", "Godohou", "Kissamey", "Lonkly"]),
    ("Couffo", "Djakotomey", ["Adjintimey", "Bétoumey", "Djakotomey I", "Djakotomey II", "Gohomey", "Houégamey", "Kpoba", "Koudo", "Sokouhoué", "Stefany"]),
    ("Couffo", "Dogbo", ["Ayomi", "Clamé", "Dévé", "Honton", "Lokogba", "Tota", "Tota-Lissèzoun"]),
    ("Couffo", "Klouékanmè", ["Ahogbèya", "Aya-Hohoué", "Djotto", "Hondoun", "Klouékanmè", "Lanta"]),
    ("Couffo", "Lalo", ["Adingnigon", "Ahodjinnako", "Ahomadégbé", "Banigbé", "Gnathoungnan", "Hlassamezoun", "Lalo", "Lokogba", "Tchito", "Zalli"]),
    ("Couffo", "Toviklin", ["Agbodji", "Avédjin", "Doko", "Houédogli", "Tannou-Gola", "Toviklin"]),
    ("Donga", "Bassila", ["Alédjo", "Bassila", "Manigri", "Pénéssoulou"]),
    ("Donga", "Copargo", ["Anandana", "Copargo", "Pabégou", "Singré"]),
    ("Donga", "Djougou", ["Barei", "Bariénou", "Bélléfoungou", "Bougou", "Djougou I", "Djougou II", "Djougou III", "Kolokondé", "Onklou", "Patargo", "Pélébina", "Sérou"]),
    ("Donga", "Ouaké", ["Badjoudè", "Kondé", "Ouaké", "Sèmèrè I", "Sèmèrè II", "Tchalinga"]),
    ("Littoral", "Cotonou", [f"{i}er arrondissement de Cotonou" if i == 1 else f"{i}e arrondissement de Cotonou" for i in range(1, 14)]),
    ("Mono", "Athiémé", ["Adohoun", "Atchannou", "Athiémé", "Dédékpoé", "Kpinnou"]),
    ("Mono", "Bopa", ["Agbodji", "Badazoui", "Bopa", "Gbakpodji", "Lobogo", "Possotomè", "Yégodoé"]),
    ("Mono", "Comè", ["Agatogbo", "Akodéha", "Comè", "Ouèdèmè-Pédah", "Oumako"]),
    ("Mono", "Grand-Popo", ["Adjaha", "Agoué", "Avloh", "Djanglanmey", "Gbéhoué", "Grand-Popo", "Sazoué"]),
    ("Mono", "Houéyogbé", ["Dahé", "Doutou", "Honhoué", "Houéyogbé", "Sè", "Zoungbonou"]),
    ("Mono", "Lokossa", ["Agamé", "Houin", "Koudo", "Lokossa", "Ouèdèmè"]),
    ("Ouémé", "Adjarra", ["Adjarra I", "Adjarra II", "Aglogbé", "Honvié", "Malanhoui", "Médédjonou"]),
    ("Ouémé", "Adjohoun", ["Adjohoun", "Akpadanou", "Awonou", "Azowlissè", "Dèmè", "Gangban", "Kodè", "Togbota"]),
    ("Ouémé", "Aguégués", ["Avagbodji", "Houédomè", "Zoungamè"]),
    ("Ouémé", "Akpro-Missérété", ["Akpro-Missérété", "Gomè-Sota", "Katagon", "Vakon", "Zodogbomey"]),
    ("Ouémé", "Avrankou", ["Atchoukpa", "Avrankou", "Djomon", "Gbozounmè", "Kouty", "Ouanho", "Sado"]),
    ("Ouémé", "Bonou", ["Affamè", "Atchonsa", "Bonou", "Damè-Wogon", "Houinviguè"]),
    ("Ouémé", "Dangbo", ["Dangbo", "Dèkin", "Gbéko", "Houédomey", "Hozin", "Késsounou", "Zounguè"]),
    ("Ouémé", "Porto-Novo", ["1er arrondissement de Porto-Novo", "2e arrondissement de Porto-Novo", "3e arrondissement de Porto-Novo", "4e arrondissement de Porto-Novo", "5e arrondissement de Porto-Novo"]),
    ("Ouémé", "Sèmè-Kpodji", ["Agblangandan", "Aholouyèmè", "Djèrègbè", "Ekpè", "Sèmè-Kpodji", "Tohouè"]),
    ("Plateau", "Adja-Ouèrè", ["Adja-Ouèrè", "Ikpinlè", "Kpoulou", "Massè", "Oko-Akarè", "Totonnoukon"]),
    ("Plateau", "Ifangni", ["Banigbé", "Daagbé", "Ifangni", "Ko-Koumolou", "Lagbé", "Tchaada"]),
    ("Plateau", "Kétou", ["Adakplamé", "Idigny", "Kpankou", "Kétou", "Odometa", "Okpometa"]),
    ("Plateau", "Pobè", ["Ahoyéyé", "Igana", "Issaba", "Pobè", "Towé"]),
    ("Plateau", "Sakété", ["Aguidi", "Ita-Djèbou", "Sakété I", "Sakété II", "Takon", "Yoko"]),
    ("Zou", "Abomey", ["Agblodjo", "Agbokpa", "Djidja I", "Djidja II", "Hounli", "Sèhoun", "Vidolè", "Zounzonmè"]),
    ("Zou", "Agbangnizoun", ["Adahondjigon", "Agbangnizoun", "Kinta", "Lissazoun", "Sahé", "Sinwé", "Tanwé-Hessou", "Zoungoudo"]),
    ("Zou", "Bohicon", ["Agongointo", "Avogbana", "Bohicon I", "Bohicon II", "Gnidjazoun", "Lissèzoun", "Ouassaho", "Passagon", "Saclo", "Sodohomè"]),
    ("Zou", "Covè", ["Adogbé", "Gounli", "Houéko", "Lainta-Cogbé", "Naogon", "Solizoun", "Zogba"]),
    ("Zou", "Djidja", ["Agouna", "Dan", "Djidja-Mansè", "Dohouimè", "Gobè", "Outo", "Setto", "Zoukon"]),
    ("Zou", "Ouinhi", ["Dasso", "Sèhouè", "Za-Kpota", "Zagnanado"]),
    ("Zou", "Za-Kpota", ["Allahé", "Assalin", "Houngomey", "Kpakpame", "Koudo", "Za-Kpota", "Za-Tanta", "Zèko"]),
    ("Zou", "Zagnanado", ["Agonli-Houégbo", "Banamè", "Dontan", "Dovi", "N-Tan", "Zagnanado"]),
    ("Zou", "Zogbodomey", ["Cana I", "Cana II", "Dêmè", "Koussoukpa", "Kpanroun", "Massi", "Tanwé-Hessou", "Zogbodomey", "Zoukou"])
]

rows = []
for dept, com, arr_list in DATA_BENIN:
    for arr in arr_list:
        rows.append({'departement': dept, 'commune': com, 'arrondissement': arr})

df_ref_arr = pd.DataFrame(rows)

def norm_txt(x):
    x = '' if x is None else str(x)
    x = unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', x.strip().lower())

df_ref_arr['arrondissement_norm'] = df_ref_arr['arrondissement'].apply(norm_txt)
MAPPING_ARR = dict(zip(df_ref_arr['arrondissement_norm'], df_ref_arr['arrondissement']))

def corriger_nom_via_ref(nom_brut):
    if not isinstance(nom_brut, str):
        return nom_brut
    nom_propre_temp = nom_brut.replace('\ufffd', '?')
    cle_norm = norm_txt(nom_propre_temp)
    if cle_norm in MAPPING_ARR:
        return MAPPING_ARR[cle_norm]
    cle_regex = re.sub(r'[\?]+', '.*', cle_norm)
    for k, v in MAPPING_ARR.items():
        if re.fullmatch(cle_regex, k):
            return v
    return nom_brut.strip().title()

# ================================================================
# 1. FONCTIONS ET ALIASES DE BASE
# ================================================================

DEPT_ALIASES = {
    'oueme': 'Ouémé', 'couffo': 'Couffo', 'kouffo': 'Couffo', 'atacora': 'Atacora',
    'atakora': 'Atacora', 'plateau': 'Plateau', 'collines': 'Collines', 'borgou': 'Borgou',
    'alibori': 'Alibori', 'donga': 'Donga', 'zou': 'Zou', 'mono': 'Mono',
    'atlantique': 'Atlantique', 'littoral': 'Littoral',
}

def canon_dept(x):
    return DEPT_ALIASES.get(norm_txt(x), str(x).strip())

def _lire_shapefile_encodage_auto(path):
    encodages = ['UTF-8', 'CP1252', 'ISO-8859-1', 'LATIN1']
    meilleur, meilleur_score, meilleur_enc = None, None, None
    ancien = os.environ.get('SHAPE_ENCODING')
    for enc in encodages:
        os.environ['SHAPE_ENCODING'] = enc
        try:
            gdf_test = gpd.read_file(path)
        except Exception:
            continue
        finally:
            if ancien is None:
                os.environ.pop('SHAPE_ENCODING', None)
            else:
                os.environ['SHAPE_ENCODING'] = ancien
        score = 0
        for col in gdf_test.select_dtypes(include='object').columns:
            if col == 'geometry': continue
            score += gdf_test[col].astype(str).str.count(r'[\?u\ufffd]').sum()
        if meilleur_score is None or score < meilleur_score:
            meilleur, meilleur_score, meilleur_enc = gdf_test, score, enc
        if meilleur_score == 0: break
    return meilleur if meilleur is not None else gpd.read_file(path)

# ================================================================
# 2. CHARGEMENT SÉQUENTIEL DES FICHIERS
# ================================================================

target_shp_dir = Path('/content/shapefile_benin')
ARR_DIR = Path(CONFIG.get('shapefile_arrondissements_dir', '/content/shapefile_arrondissements'))
GENERIC_ZIP_DIR = Path('/content/generic_zips')

target_shp_dir.mkdir(exist_ok=True)
ARR_DIR.mkdir(exist_ok=True)
GENERIC_ZIP_DIR.mkdir(exist_ok=True)

def extraire_et_deplacer(uploaded_dict, destination_dir):
    for fname, content in uploaded_dict.items():
        fpath = Path('/content') / fname
        with open(fpath, 'wb') as f:
            f.write(content)
        if fpath.suffix.lower() == '.zip':
            with zipfile.ZipFile(fpath, 'r') as z:
                z.extractall(destination_dir)
            fpath.unlink(missing_ok=True)
        else:
            shutil.move(str(fpath), destination_dir / fpath.name)

# 1. Charger ADMIN 1 / 2
print("📂 Veuillez choisir le premier fichier ZIP ou SHP (ADMIN 1 / ADMIN 2)...")
uploaded_admin12 = files.upload()
if uploaded_admin12:
    extraire_et_deplacer(uploaded_admin12, target_shp_dir)
    print("✅ Shapefile ADMIN 1 / 2 chargé avec succès !\n")

# 2. Charger ADMIN 3
print("📂 Veuillez choisir le deuxième fichier ZIP ou SHP (ADMIN 3 / Arrondissements)...")
uploaded_admin3 = files.upload()
if uploaded_admin3:
    extraire_et_deplacer(uploaded_admin3, ARR_DIR)
    print("✅ Calque ADMIN 3 chargé avec succès !\n")

# 3. Charger ZIP générique
print("📂 Veuillez choisir le troisième fichier ZIP (générique, optionnel)...")
uploaded_generic_zip = files.upload()
if uploaded_generic_zip:
    extraire_et_deplacer(uploaded_generic_zip, GENERIC_ZIP_DIR)
    print("✅ Fichier ZIP générique chargé avec succès !\n")

# 4. Charger le Logo
print("📂 Veuillez choisir le fichier PNG ou JPG pour le logo...")
uploaded_logo = files.upload()
if uploaded_logo:
    for fname, content in uploaded_logo.items():
        with open(Path('/content') / fname, 'wb') as f:
            f.write(content)
    print("✅ Logo chargé avec succès !\n")

# ================================================================
# 3. TRAITEMENT DU SHAPEFILE PRINCIPAL (ADMIN 1 / 2)
# ================================================================

shp_files = list(target_shp_dir.rglob('*.shp'))
admin2_shp_path = next((p for p in shp_files if 'ben_admin2.shp' in str(p)), None)
SHP_PATH = admin2_shp_path if admin2_shp_path else (shp_files[0] if shp_files else None)

if SHP_PATH and SHP_PATH.exists():
    CONFIG['shapefile_path'] = str(SHP_PATH)
    print(f'\n🗺️ Chargement : {SHP_PATH}')
    gdf_raw = _lire_shapefile_encodage_auto(SHP_PATH)
    display(gdf_raw.head())

    cols = list(gdf_raw.columns)
    cols_norm = {norm_txt(c): c for c in cols}

    # Liste étendue de candidats pour éviter le KeyError
    candidats_commune = ['shapename', 'adm2_name', 'adm2_fr', 'adm2name', 'name_2', 'commune', 'communes', 'adm2', 'adm_2', 'name']
    candidats_dept = ['name_1', 'adm1_name', 'adm1_fr', 'adm1name', 'departement', 'département', 'dept', 'adm1', 'adm_1', 'shapegroup', 'country']

    def find_col(candidats, min_unique=1):
        def _ok(col):
            return min_unique <= 1 or gdf_raw[col].nunique(dropna=True) >= min_unique

        # 1. Matching exact
        for cand in candidats:
            key = norm_txt(cand)
            if key in cols_norm and _ok(cols_norm[key]):
                return cols_norm[key]

        # 2. Matching partiel
        for key, original in cols_norm.items():
            if any(norm_txt(c) in key for c in candidats) and _ok(original):
                return original
        return None

    # Détection de la commune et du département
    col_commune = find_col(candidats_commune, min_unique=1)
    col_dept = find_col(candidats_dept, min_unique=2)

    # Secours si col_dept renvoie None (ex: shapeGroup avec 1 seule valeur)
    if col_dept is None:
        col_dept = find_col(candidats_dept, min_unique=1)

    # Si la colonne commune n'a pas été trouvée, on prend la 1ère colonne texte disponible
    if col_commune is None:
        obj_cols = [c for c in cols if c != 'geometry']
        col_commune = obj_cols[0] if obj_cols else cols[0]

    # Secours ultime : si département non trouvé, on réutilise la commune ou une chaîne fixe pour éviter None
    if col_dept is None:
        col_dept = col_commune

    print(f"   Colonne commune sélectionnée     : {col_commune}")
    print(f"   Colonne département sélectionnée : {col_dept}")

    gdf_com = gdf_raw[[col_commune, col_dept, 'geometry']].copy()
    gdf_com = gdf_com.rename(columns={col_commune: 'commune', col_dept: 'dept'})
    gdf_com['commune'] = gdf_com['commune'].astype(str).str.strip()
    gdf_com['dept'] = gdf_com['dept'].apply(canon_dept)

    if gdf_com.crs is None:
        gdf_com = gdf_com.set_crs(epsg=4326)
    elif gdf_com.crs.to_epsg() != 4326:
        gdf_com = gdf_com.to_crs(epsg=4326)

    gdf_com['geometry'] = gdf_com.geometry.buffer(0)

    def keep_largest(geom, seuil=0.05):
        if isinstance(geom, Polygon): return geom
        if isinstance(geom, MultiPolygon):
            polys = [p for p in geom.geoms if isinstance(p, Polygon) and not p.is_empty]
            if not polys: return geom
            max_a = max(p.area for p in polys)
            ok = [p for p in polys if p.area >= max_a * seuil]
            return ok[0] if len(ok)==1 else MultiPolygon(ok)
        return geom

    gdf_com_clean = gdf_com.copy()
    gdf_com_clean['geometry'] = gdf_com_clean.geometry.apply(keep_largest)
    gdf_dept = gdf_com[['dept', 'geometry']].rename(columns={'dept': 'name'}).dissolve(by='name').reset_index()

    print(f'✅ Shapefile chargé : {len(gdf_com)} communes · {len(gdf_dept)} départements')

# ================================================================
# 4. CALQUE ARRONDISSEMENTS (ADM3) AVEC CORRECTION AUTOMATIQUE
# ================================================================

gdf_arr = None
arr_shp_existant = list(ARR_DIR.rglob('*.shp')) if ARR_DIR.exists() else []

if arr_shp_existant:
    try:
        admin3_shp_path = next((p for p in arr_shp_existant if 'geoBoundaries-BEN-ADM3_simplified.shp' in str(p)), None)
        selected_arr_shp = admin3_shp_path if admin3_shp_path else arr_shp_existant[0]

        gdf_arr = _lire_shapefile_encodage_auto(selected_arr_shp)
        if gdf_arr.crs is None:
            gdf_arr = gdf_arr.set_crs(epsg=4326)
        elif gdf_arr.crs.to_epsg() != 4326:
            gdf_arr = gdf_arr.to_crs(epsg=4326)
        gdf_arr['geometry'] = gdf_arr.geometry.buffer(0)

        # Recherche flexible de la colonne d'arrondissement
        col_arr = None
        for c in ['shapeName', 'adm3_name', 'adm3_fr', 'name_3', 'arrondissement']:
            for col_real in gdf_arr.columns:
                if norm_txt(col_real) == norm_txt(c):
                    col_arr = col_real
                    break
            if col_arr: break

        if col_arr:
            gdf_arr['shapeName'] = gdf_arr[col_arr].apply(corriger_nom_via_ref)

        print(f'✅ Calque arrondissements corrigé et chargé : {len(gdf_arr)} unités ({selected_arr_shp.name}).')
    except Exception as e:
        print(f'⚠️ Calque arrondissements non exploitable ({e})')
        gdf_arr = None

# ================================================================
# 5. LOGO MÉTÉO-BÉNIN
# ================================================================

LOGO_PATH = None
_logo_candidats = list(Path('/content').glob('logo*.png')) + list(Path('/content').glob('logo*.jpg'))

if _logo_candidats:
    LOGO_PATH = str(_logo_candidats[0])
    CONFIG['logo_path'] = LOGO_PATH
    print(f'✅ Logo chargé : {LOGO_PATH}')

📂 Veuillez choisir le premier fichier ZIP ou SHP (ADMIN 1 / ADMIN 2)...


Saving ben_admin_boundaries.shp (1).zip to ben_admin_boundaries.shp (1).zip
✅ Shapefile ADMIN 1 / 2 chargé avec succès !

📂 Veuillez choisir le deuxième fichier ZIP ou SHP (ADMIN 3 / Arrondissements)...


Saving geoBoundaries-BEN-ADM3-all.zip to geoBoundaries-BEN-ADM3-all.zip
✅ Calque ADMIN 3 chargé avec succès !

📂 Veuillez choisir le troisième fichier ZIP (générique, optionnel)...


Saving geoBoundaries-BEN-ADM3-all.zip to geoBoundaries-BEN-ADM3-all.zip
✅ Fichier ZIP générique chargé avec succès !

📂 Veuillez choisir le fichier PNG ou JPG pour le logo...


Saving logo.png to logo.png
✅ Logo chargé avec succès !


🗺️ Chargement : /content/shapefile_benin/ben_admin2.shp


,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,...,version,lang,lang1,lang2,lang3,adm0_en,adm2_ref_n,center_lat,center_lon,geometry
0,Abomey,None,None,None,BJ1201,Zou,None,None,None,BJ12,...,v01,fr,None,None,None,Benin,Abomey,7.180098,1.972786,"POLYGON ((1.90458 7.17512, 1.90633 7.18243, 1...."
1,Abomey-Calavi,None,None,None,BJ0301,Atlantique,None,None,None,BJ03,...,v01,fr,None,None,None,Benin,Abomey-Calavi,6.521738,2.325246,"POLYGON ((2.21874 6.45584, 2.22242 6.46285, 2...."
2,Adja-Ouere,None,None,None,BJ1101,Plateau,None,None,None,BJ11,...,v01,fr,None,None,None,Benin,Adja-Ouere,7.014533,2.595494,"POLYGON ((2.53742 7.187, 2.55926 7.18418, 2.57..."
3,Adjarra,None,None,None,BJ1001,Oueme,None,None,None,BJ10,...,v01,fr,None,None,None,Benin,Adjarra,6.485182,2.687731,"POLYGON ((2.65862 6.4436, 2.66037 6.44912, 2.6..."
4,Adjohoun,None,None,None,BJ1002,Oueme,None,None,None,BJ10,...,v01,fr,None,None,None,Benin,Adjohoun,6.705048,2.495928,"POLYGON ((2.4154 6.6334, 2.41107 6.64024, 2.40..."


   Colonne commune sélectionnée     : adm2_name
   Colonne département sélectionnée : adm1_name
✅ Shapefile chargé : 77 communes · 12 départements
✅ Calque arrondissements corrigé et chargé : 546 unités (geoBoundaries-BEN-ADM3_simplified.shp).
✅ Logo chargé : /content/logo.png


In [7]:
# ════════════════════════════════════════════════════════════════
# C5 — DONNÉES ECMWF DISSEMINATION (ECPDS, compte WMO-BJ)
# Remplace l'Open Data (0.25°) par le flux de dissémination réel
# (0.1°, paquet surface FSD) accessible via le compte wmo_bj.
# Repli sur Open Data si ECPDS est indisponible ; aucune donnée simulée.
# ════════════════════════════════════════════════════════════════

import getpass
import re as _re

def calc_rh_from_t_td(t_c, td_c):
    """Calcule l'humidité relative (%) à partir de T2m et du point de rosée 2m.
    Formule de Magnus, suffisante pour les météogrammes opérationnels."""
    if np.isnan(t_c) or np.isnan(td_c):
        return np.nan
    es_td = 6.112 * np.exp((17.67 * td_c) / (td_c + 243.5))
    es_t = 6.112 * np.exp((17.67 * t_c) / (t_c + 243.5))
    return float(np.clip(100.0 * es_td / es_t, 0, 100))


def calc_temp_ressentie(t_c, td_c, wind_kmh):
    """Température ressentie (apparent temperature), formule du Bureau of
    Meteorology australien — bien adaptée aux climats humides/tropicaux
    comme le Bénin. Utilisée pour la vigilance chaleur (demande explicite :
    'pour la température, il s'agit du ressenti', pas la température brute).
    AT = Ta + 0.33*e - 0.70*ws - 4.00
    où e = pression de vapeur (hPa, à partir du point de rosée),
       ws = vitesse du vent en m/s.
    """
    if np.isnan(t_c) or np.isnan(td_c):
        return t_c if not np.isnan(t_c) else np.nan
    e = 6.112 * np.exp((17.67 * td_c) / (td_c + 243.5))
    ws_ms = 0.0 if np.isnan(wind_kmh) else wind_kmh / 3.6
    return float(t_c + 0.33 * e - 0.70 * ws_ms - 4.00)



# ── Règles uniques de temps et de précipitation ─────────────────────
def utc_to_local_naive(ts):
    return pd.Timestamp(ts) + LOCAL_UTC_OFFSET


def deaccumuler_tp(df):
    """Convertit TP cumulé (mm) en pluie par pas (mm) avant filtrage temporel."""
    out = df.sort_values('time').reset_index(drop=True).copy()
    if out.empty:
        out['tp_mm'] = pd.Series(dtype=float)
        return out
    cumul = pd.to_numeric(out['tp_cumul_mm'], errors='coerce').clip(lower=0)
    diff = cumul.diff()
    tp = diff.where(diff >= 0, cumul).fillna(cumul)
    out['tp_mm'] = tp.clip(lower=0).round(1)
    return out


def filtrer_periode_locale(df):
    return df[(df['time'] >= PERIODE_DEBUT) & (df['time'] < PERIODE_FIN)].copy()

# ── Décodage/encodage des noms de fichiers ECPDS ──────────────────
_ECPDS_RE = _re.compile(
    r'^(?P<prefix>[A-Z]{2,3})'
    r'(?P<run_md>\d{4})(?P<run_hm>\d{4})'
    r'(?P<valid_md>\d{4})(?P<valid_h>\d{2})'
    r'(?P<seq>\d{3})$'
)

def _ecpds_parse_filename(fname, year):
    m = _ECPDS_RE.match(fname)
    if not m:
        return None, None, None
    d = m.groupdict()
    run_dt = datetime.datetime.strptime(f"{year}{d['run_md']}{d['run_hm']}", "%Y%m%d%H%M")
    valid_dt = datetime.datetime.strptime(f"{year}{d['valid_md']}{d['valid_h']}", "%Y%m%d%H")
    return d['prefix'], run_dt, valid_dt


def _ecpds_list_dir(session, date_str):
    """Liste les fichiers disponibles pour une date (YYYYMMDD) sur ECPDS.
    Le serveur renvoie du HTML par défaut ; il faut demander explicitement
    le JSON via l'en-tête Accept pour obtenir la liste structurée
    (confirmé le 2026-07-17 : réponse = liste plate de
    {"name","size","time","directory"})."""
    url = f'https://diss.ecmwf.int/ecpds/data/list/{date_str}/'
    r = session.get(url, headers={'Accept': 'application/json'}, timeout=30)
    r.raise_for_status()
    payload = r.json()
    return [entry['name'] for entry in payload
            if isinstance(entry, dict) and not entry.get('directory', False) and entry.get('name')]


def _ecpds_download(session, date_str, filename, dest_path):
    url = f'https://diss.ecmwf.int/ecpds/data/file/{date_str}/{filename}'
    with session.get(url, timeout=120, stream=True) as r:
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)


# ── Recadrage de la grille sur le Bénin (mémoire raisonnable) ─────
# Bénin : ~6.0°N à 12.6°N, ~0.5°E à 3.9°E (marge incluse).
BENIN_LAT_MIN, BENIN_LAT_MAX = 6.0, 12.6
BENIN_LON_MIN, BENIN_LON_MAX = 0.5, 3.9


def _crop_indices(lat0=28.97, lon0=-26.34, dlat=0.1):
    """Indices [row0:row1, col0:col1] correspondant au Bénin sur la grille 0.1°."""
    row1 = int(np.ceil((lat0 - BENIN_LAT_MIN) / dlat)) + 1
    row0 = int(np.floor((lat0 - BENIN_LAT_MAX) / dlat))
    col0 = int(np.floor((BENIN_LON_MIN - lon0) / dlat))
    col1 = int(np.ceil((BENIN_LON_MAX - lon0) / dlat)) + 1
    return max(row0, 0), row1, max(col0, 0), col1


def _grille_bilineaire(vals, lat0, lon0, dlat, lat, lon):
    """Interpolation bilinéaire sur une grille régulière (0.1°) dont l'origine
    est en haut à gauche (lat décroissante vers le sud)."""
    nj, ni = vals.shape
    fy = (lat0 - lat) / dlat
    fx = (lon - lon0) / dlat
    y0, x0 = int(np.floor(fy)), int(np.floor(fx))
    y1, x1 = min(y0 + 1, nj - 1), min(x0 + 1, ni - 1)
    y0, x0 = max(y0, 0), max(x0, 0)
    wy, wx = fy - y0, fx - x0
    return (vals[y0, x0] * (1 - wy) * (1 - wx) + vals[y0, x1] * (1 - wy) * wx +
            vals[y1, x0] * wy * (1 - wx) + vals[y1, x1] * wy * wx)


def _lire_fsd(fsd_path, communes_geodata, lat0=28.97, lon0=-26.34, dlat=0.1,
              params=('2t', '2d', '10u', '10v', 'tp'), garder_grille=True):
    """Lit un fichier FSD et renvoie :
      - {ville: {param: valeur}} interpolé (bilinéaire) par ville,
      - si garder_grille=True, la grille complète recadrée sur le Bénin
        pour chaque paramètre (utilisée pour la carte raster en C6bis).
    """
    grilles = {}
    with open(fsd_path, 'rb') as f:
        while True:
            gid = ec.codes_grib_new_from_file(f)
            if gid is None:
                break
            sn = ec.codes_get(gid, 'shortName')
            if sn in params and sn not in grilles:
                ni = ec.codes_get(gid, 'Ni')
                nj = ec.codes_get(gid, 'Nj')
                grilles[sn] = np.array(ec.codes_get_values(gid)).reshape(nj, ni)
            ec.codes_release(gid)

    out = {}
    for nom, dept, lat, lon in communes_geodata:
        out[nom] = {}
        for p, vals in grilles.items():
            out[nom][p] = _grille_bilineaire(vals, lat0, lon0, dlat, lat, lon)

    grille_benin = None
    if garder_grille and grilles:
        row0, row1, col0, col1 = _crop_indices(lat0, lon0, dlat)
        grille_benin = {p: v[row0:row1, col0:col1] for p, v in grilles.items()}

    return out, grille_benin


def charger_donnees_ecpds():
    """Télécharge et assemble les données de dissémination ECPDS (0.1°, FSD)
    pour toutes les échéances de CONFIG['horizon_h'], au pas CONFIG['pas_h']."""
    try:
        import eccodes as ec  # noqa: F401 (rendu global ci-dessous)
        globals()['ec'] = ec
    except ModuleNotFoundError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'eccodes'], check=True)
        import eccodes as ec
        globals()['ec'] = ec

    identifiant = CONFIG.get('ecpds_user', 'wmo_bj')
    mot_de_passe = CONFIG.get('ecpds_password')  # laisser None -> demande interactive
    if not mot_de_passe:
        mot_de_passe = getpass.getpass(f"Mot de passe ECPDS pour '{identifiant}' : ")

    session = requests.Session()
    session.auth = (identifiant, mot_de_passe)

    # Naïf (sans fuseau) : les dates extraites des noms de fichiers ECPDS
    # (_ecpds_parse_filename) n'ont pas de fuseau, elles seraient sinon
    # incomparables ("can't compare offset-naive and offset-aware datetimes").
    maintenant = datetime.datetime.utcnow()
    dossier_local = CONFIG.get('dossier_fsd', os.path.join(CONFIG['output_dir'], 'ecpds_fsd'))
    os.makedirs(dossier_local, exist_ok=True)
    CONFIG['dossier_fsd'] = dossier_local  # pour que C3bis retrouve les mêmes fichiers

    # ── Trouver le dernier run disponible (teste aujourd'hui puis hier) ──
    run_choisi = None
    fichiers_disponibles = {}
    for delta in range(0, 2):
        d = maintenant - datetime.timedelta(days=delta)
        date_str = d.strftime('%Y%m%d')
        try:
            noms = _ecpds_list_dir(session, date_str)
        except Exception as e:
            print(f'⚠️ Listing ECPDS impossible pour {date_str} : {e}')
            continue

        runs_presents = {}
        for nom in noms:
            prefix, run_dt, valid_dt = _ecpds_parse_filename(nom, d.year)
            if prefix != 'FSD' or run_dt is None:
                continue
            runs_presents.setdefault(run_dt, []).append((valid_dt, nom))

        runs_valides = [r for r in runs_presents if r <= maintenant]
        if runs_valides:
            run_choisi = max(runs_valides)
            fichiers_disponibles = {v: n for v, n in runs_presents[run_choisi]}
            break

    if run_choisi is None:
        raise RuntimeError("Aucun run FSD trouvé sur ECPDS pour aujourd'hui/hier.")

    print(f'✅ Run ECPDS utilisé : {run_choisi:%Y-%m-%d %H}Z ({len(fichiers_disponibles)} échéances FSD trouvées)')

    # ── Sélectionner/télécharger les échéances voulues ────────────────
    steps = list(range(0, CONFIG['horizon_h'] + CONFIG['pas_h'], CONFIG['pas_h']))
    echeances_valid = sorted(fichiers_disponibles.keys())

    data_brute = {}   # {valid_time: {ville: {param: val}}}
    grid_series = {}  # {valid_time: {param: grille 2D recadrée Bénin}}
    for step in steps:
        cible = run_choisi + datetime.timedelta(hours=step)
        # Échéance disponible la plus proche (tolérance 90 min)
        plus_proche = min(echeances_valid, key=lambda v: abs((v - cible).total_seconds()))
        if abs((plus_proche - cible).total_seconds()) > 90 * 60:
            continue

        nom_fichier = fichiers_disponibles[plus_proche]
        dest = os.path.join(dossier_local, nom_fichier)
        if not os.path.exists(dest):
            _ecpds_download(session, run_choisi.strftime('%Y%m%d'), nom_fichier, dest)

        # Use ALL_COMMUNES_GEODATA here
        par_ville, grille_benin = _lire_fsd(dest, ALL_COMMUNES_GEODATA)
        data_brute[plus_proche] = par_ville
        if grille_benin is not None:
            grid_series[utc_to_local_naive(plus_proche)] = grille_benin

    if not data_brute:
        raise RuntimeError('Aucune échéance FSD exploitable après téléchargement.')

    # Grille brute exposée globalement pour C6bis (calcul de vigilance sur grille)
    global GRID_SERIES, GRID_META
    GRID_SERIES = grid_series
    row0, row1, col0, col1 = _crop_indices()
    GRID_META = {
        'lat0': 28.97, 'lon0': -26.34, 'dlat': 0.1,
        'row0': row0, 'row1': row1, 'col0': col0, 'col1': col1,
        'lats': 28.97 - 0.1 * np.arange(row0, row1),
        'lons': -26.34 + 0.1 * np.arange(col0, col1),
    }

    # ── Assemblage par ville (mêmes colonnes qu'auparavant) ───────────
    # Use ALL_COMMUNES_GEODATA here
    data = {}
    for nom, dept, lat, lon in ALL_COMMUNES_GEODATA:
        rows = []
        for valid_time in sorted(data_brute):
            v = data_brute[valid_time].get(nom, {})
            t2 = v.get('2t', np.nan) - 273.15
            td2 = v.get('2d', np.nan) - 273.15 if '2d' in v else np.nan
            u10, v10 = v.get('10u', np.nan), v.get('10v', np.nan)
            wind = np.sqrt(u10**2 + v10**2) * 3.6 if not (np.isnan(u10) or np.isnan(v10)) else np.nan
            rh = calc_rh_from_t_td(t2, td2)
            tp_acc = max(v.get('tp', 0.0) * 1000, 0)  # m -> mm, cumulé depuis le début du run

            at_ressenti = calc_temp_ressentie(t2, td2, wind)

            rows.append({'time': utc_to_local_naive(valid_time), 't2m_C': round(t2, 1),
                         'td2m_C': round(td2, 1) if not np.isnan(td2) else np.nan,
                         'at_C': round(at_ressenti, 1) if not np.isnan(at_ressenti) else np.nan,
                         'tp_cumul_mm': round(tp_acc, 2),
                         'wind_kmh': round(wind, 1) if not np.isnan(wind) else np.nan,
                         'rh_pct': round(rh, 1) if not np.isnan(rh) else np.nan})

        df = pd.DataFrame(rows).sort_values('time').reset_index(drop=True)
        df = deaccumuler_tp(df)
        df_window = filtrer_periode_locale(df)
        data[nom] = df_window.reset_index(drop=True)

    # ── Garde-fou couverture ────────────────────────────────────────
    # La dissémination ECPDS est progressive : le court terme arrive vite
    # après le cutoff, le moyen/long terme (jusqu'à horizon_h) peut prendre
    # plusieurs heures de plus. Si l'échéance la plus lointaine trouvée est
    # trop courte par rapport à la fenêtre du bulletin (PERIODE_FIN), on lève
    # une exception : le bloc appelant (plus bas) retombe alors automatique-
    # ment sur Open Data plutôt que de publier un bulletin silencieusement
    # tronqué (cartes J2-J7 vides, météorogrammes coupés).
    derniere_echeance = max(echeances_valid)
    source_label = f'ECMWF Dissémination ECPDS ({run_choisi:%Y-%m-%d %HZ}, 0.1°)'
    if derniere_echeance < PERIODE_FIN_UTC:
        manque_h = (PERIODE_FIN_UTC - derniere_echeance).total_seconds() / 3600
        print(f"⚠️ Couverture ECPDS incomplète : dernière échéance {derniere_echeance:%Y-%m-%d %HZ}, "
              f"il manque {manque_h:.0f}h pour couvrir le bulletin jusqu'au {PERIODE_FIN:%Y-%m-%d %HZ} "
              f"(dissémination probablement encore en cours pour ce run).")
        pas = datetime.timedelta(hours=CONFIG['pas_h'])
        valid_times_manquants = []
        t = derniere_echeance + pas
        while t <= PERIODE_FIN_UTC:
            valid_times_manquants.append(t)
            t += pas
        try:
            print(f"   🔁 Complément Open Data pour {len(valid_times_manquants)} échéance(s) manquante(s)...")
            _completer_avec_open_data(data, valid_times_manquants, ALL_COMMUNES_GEODATA)
            source_label += f' + Open Data 0.25° (repli pour {derniere_echeance:%d/%m %Hh}\u2192{PERIODE_FIN:%d/%m %Hh})'
            print("   ✅ Complément Open Data appliqué.")
        except Exception as e:
            print(f"   ⚠️ Complément Open Data impossible ({e}) — le bulletin restera tronqué "
                  f"au-delà du {derniere_echeance:%d/%m %Hh} pour cette exécution.")

    return data, source_label


# ── Repli : Open Data (0.25°), inchangé par rapport à l'ancienne C5 ──
def _open_data_dernier_run(besoin_echeance_longue=False):
    """Dernier run Open Data (0.25°) publié — factorisé pour être réutilisé
    par charger_donnees_open_data() (repli total) ET par
    _completer_avec_open_data() (complément partiel de queue ECPDS).

    besoin_echeance_longue=True restreint la recherche aux runs 00Z/12Z :
    dans le cycle opérationnel ECMWF, seuls ces deux runs "principaux" sont
    disséminés jusqu'à 240h ; les runs 06Z/18Z ("secondaires") s'arrêtent
    bien avant (~90h), même quand ils sont plus récents. Sans cette
    restriction, on peut sélectionner un run 06Z/18Z frais qui 404 sur
    toute échéance longue, alors qu'un run 00Z/12Z plus ancien l'aurait
    fournie (cause du 404 observé sur l'échéance 150h).

    Respecte aussi CONFIG['ecmwf_date']/['ecmwf_run'] quand ils sont
    renseignés (différents de 'auto') : permet de forcer un run précis
    au lieu de l'auto-détection.
    """
    forcer_date = CONFIG.get('ecmwf_date')
    forcer_run = CONFIG.get('ecmwf_run')
    if forcer_date not in (None, 'auto') and forcer_run not in (None, 'auto'):
        return forcer_date, forcer_run

    base = 'https://data.ecmwf.int/forecasts'
    now = datetime.datetime.now(datetime.timezone.utc)
    runs_h = ['12', '00'] if besoin_echeance_longue else ['18', '12', '06', '00']
    candidats = []
    for delta in range(0, 4):
        d = now - datetime.timedelta(days=delta)
        ds = d.strftime('%Y%m%d')
        for run_h in runs_h:
            run_dt = datetime.datetime.strptime(ds + run_h, '%Y%m%d%H').replace(tzinfo=datetime.timezone.utc)
            if run_dt > now:
                continue
            candidats.append((run_dt, ds, run_h))
    for _, ds, run_h in sorted(candidats, reverse=True):
        url = f'{base}/{ds}/{run_h}z/ifs/0p25/oper/{ds}{run_h}0000-0h-oper-fc.index'
        try:
            r = requests.head(url, timeout=10, allow_redirects=True)
            if r.status_code == 200:
                return ds, run_h
        except Exception:
            pass
    return None, None


def _completer_avec_open_data(data, valid_times_manquants, communes_geodata):
    """Complète `data` (par ville) ET GRID_SERIES (par maille, sur la grille
    Bénin déjà en place via GRID_META) pour les échéances que ECPDS n'a pas
    encore disséminées, en interpolant le dernier run Open Data (0.25°)
    disponible sur cette même grille cible (xarray .interp, pas de bricolage
    manuel). Ne casse jamais le pipeline : toute erreur ici est rattrapée par
    l'appelant, qui garde alors la portion ECPDS déjà obtenue telle quelle.

    Point d'attention assumé : Open Data et ECPDS ne sont pas forcément le
    même run (init time différent) — mélange de deux runs pour couvrir une
    même semaine, préférable à des jours vides mais pas parfaitement homogène.
    Le cumul de pluie (tp) est dé-accumulé séparément pour la portion Open
    Data (son compteur repart de zéro à SON propre run, pas de continuité
    possible avec le compteur ECPDS).
    """
    from ecmwf.opendata import Client as ECMWFClient
    import cfgrib
    import xarray as xr

    # Échéance la plus lointaine à couvrir (approx. par rapport à "now" ; la
    # différence d'âge entre runs candidats se compte en heures, ce qui ne
    # change pas la conclusion sur le seuil des ~90h des runs secondaires).
    lead_max_h = max(
        (vt.replace(tzinfo=datetime.timezone.utc) - datetime.datetime.now(datetime.timezone.utc)).total_seconds() / 3600
        for vt in valid_times_manquants
    )
    besoin_echeance_longue = lead_max_h > 48

    ecmwf_date, ecmwf_run = _open_data_dernier_run(besoin_echeance_longue)
    if ecmwf_date is None:
        raise RuntimeError('Aucun run Open Data disponible pour compléter ECPDS.')
    open_run_dt = datetime.datetime.strptime(ecmwf_date + ecmwf_run, '%Y%m%d%H').replace(tzinfo=datetime.timezone.utc)

    # Grille réelle des échéances publiées par Open Data ("oper", 0.25°) :
    # pas de 3h jusqu'à 144h, puis seulement 6h de 144h à 240h. Demander une
    # échéance hors grille (ex. 147h) renvoie un 404 qui fait échouer TOUT le
    # lot — on cale donc chaque échéance manquante sur l'échéance valide la
    # plus proche plutôt que de supposer un pas de 3h uniforme partout.
    GRILLE_STEPS_OPEN_DATA = list(range(0, 144 + 1, 3)) + list(range(150, 240 + 1, 6))

    steps_bruts = sorted({
        int(round((vt.replace(tzinfo=datetime.timezone.utc) - open_run_dt).total_seconds() / 3600))
        for vt in valid_times_manquants
    })
    steps_bruts = [s for s in steps_bruts if s >= 0]

    def _step_valide_le_plus_proche(s):
        return min(GRILLE_STEPS_OPEN_DATA, key=lambda g: abs(g - s))

    steps_needed = sorted({_step_valide_le_plus_proche(s) for s in steps_bruts
                            if 0 <= s <= 240})
    if not steps_needed:
        raise RuntimeError('Aucune échéance Open Data exploitable pour la période manquante.')
    if any(s not in GRILLE_STEPS_OPEN_DATA for s in steps_bruts):
        print(f"   ℹ️ Échéances Open Data recalées sur la grille disponible (3h jusqu'à 144h, "
              f"puis 6h) : {steps_bruts} -> {steps_needed}")

    client = ECMWFClient('ecmwf')
    date_e = datetime.datetime.strptime(ecmwf_date, '%Y%m%d').strftime('%Y-%m-%d')
    grib = f"/content/ecmwf_completement_{ecmwf_date}_{ecmwf_run}z.grib2"

    # Même un run 00Z/12Z "long" peut ne pas encore avoir disséminé sa toute
    # dernière échéance demandée (dissémination progressive) : plutôt que de
    # faire échouer tout le lot sur un seul 404, on retire l'échéance la plus
    # lointaine et on retente, jusqu'à obtenir un lot qui passe.
    steps_tentative = list(steps_needed)
    if not os.path.exists(grib):
        while steps_tentative:
            try:
                client.retrieve(date=date_e, time=int(ecmwf_run), step=steps_tentative, type='fc',
                                 param=['2t', '2d', 'tp', '10u', '10v'], target=grib)
                break
            except Exception as e:
                if '404' not in str(e) or len(steps_tentative) == 1:
                    raise
                echeance_abandonnee = steps_tentative.pop()
                print(f"   ⚠️ Échéance {echeance_abandonnee}h indisponible sur le run "
                      f"{(ecmwf_date)} {ecmwf_run}Z (404) — nouvelle tentative sans cette échéance.")
        if not steps_tentative:
            raise RuntimeError('Aucune échéance Open Data récupérable sur le run sélectionné.')

    ds_list = cfgrib.open_datasets(grib)
    ds_t = ds_d = ds_u = ds_tp = None
    for ds in ds_list:
        vars_ds = list(ds.data_vars)
        if 't2m' in vars_ds: ds_t = ds
        if 'd2m' in vars_ds: ds_d = ds
        if 'u10' in vars_ds and 'v10' in vars_ds: ds_u = ds
        if 'tp' in vars_ds: ds_tp = ds
    if ds_t is None:
        raise ValueError('Variable t2m introuvable dans le GRIB de complément Open Data.')

    valid_times_od = pd.to_datetime(ds_t.valid_time.values)

    # ── GRID_SERIES : régrillage 0.25° -> grille Bénin (GRID_META) via xarray ──
    global GRID_SERIES
    if GRID_SERIES is not None and GRID_META is not None:
        lat_cible = xr.DataArray(GRID_META['lats'], dims='lat_cible')
        lon_cible = xr.DataArray(GRID_META['lons'], dims='lon_cible')
        for i, vt in enumerate(valid_times_od):
            champ = {}
            for code, ds_src, var in [('2t', ds_t, 't2m'), ('2d', ds_d, 'd2m'),
                                       ('10u', ds_u, 'u10'), ('10v', ds_u, 'v10'),
                                       ('tp', ds_tp, 'tp')]:
                if ds_src is None:
                    continue
                champ[code] = ds_src[var].isel(step=i).interp(
                    latitude=lat_cible, longitude=lon_cible, method='linear'
                ).values
            GRID_SERIES[utc_to_local_naive(vt)] = champ
        print(f'   ✏️  GRID_SERIES complété avec {len(valid_times_od)} échéance(s) Open Data '
              f'régrillées sur la grille Bénin (0.25° -> 0.1°, run {ecmwf_date} {ecmwf_run}Z).')

    # ── Par ville : mêmes colonnes que le reste de data[nom] ──────────
    for nom, dept, lat, lon in communes_geodata:
        rows = []
        for i, valid_time in enumerate(valid_times_od):
            def val(ds_src, var, offset=0.0):
                if ds_src is None:
                    return np.nan
                return float(ds_src[var].isel(step=i).interp(latitude=lat, longitude=lon, method='linear').values) - offset
            t2 = val(ds_t, 't2m', 273.15)
            td2 = val(ds_d, 'd2m', 273.15) if ds_d is not None else np.nan
            u10 = val(ds_u, 'u10') if ds_u is not None else np.nan
            v10 = val(ds_u, 'v10') if ds_u is not None else np.nan
            wind = np.sqrt(u10**2 + v10**2) * 3.6 if not (np.isnan(u10) or np.isnan(v10)) else np.nan
            rh = calc_rh_from_t_td(t2, td2)
            tp_acc = max(val(ds_tp, 'tp') * 1000, 0) if ds_tp is not None else 0.0
            at_ressenti = calc_temp_ressentie(t2, td2, wind)
            rows.append({'time': utc_to_local_naive(valid_time), 't2m_C': round(t2, 1),
                         'td2m_C': round(td2, 1) if not np.isnan(td2) else np.nan,
                         'at_C': round(at_ressenti, 1) if not np.isnan(at_ressenti) else np.nan,
                         'tp_cumul_mm': round(tp_acc, 2),
                         'wind_kmh': round(wind, 1) if not np.isnan(wind) else np.nan,
                         'rh_pct': round(rh, 1) if not np.isnan(rh) else np.nan})
        if not rows:
            continue
        df_od = pd.DataFrame(rows).sort_values('time').reset_index(drop=True)
        # Dé-accumulation propre à CE run Open Data (compteur indépendant d'ECPDS)
        df_od = deaccumuler_tp(df_od)

        if nom in data and not data[nom].empty:
            data[nom] = pd.concat([data[nom], df_od], ignore_index=True) \
                          .drop_duplicates(subset='time').sort_values('time').reset_index(drop=True)
        else:
            data[nom] = df_od

    return len(valid_times_od)  # nombre d'échéances effectivement complétées


def charger_donnees_open_data():
    from ecmwf.opendata import Client as ECMWFClient

    # Repli total (ECPDS ET complément Open Data indisponibles) : on a besoin
    # de tout l'horizon (CONFIG['horizon_h'], généralement > 90h), donc on
    # restreint d'emblée aux runs 00Z/12Z (seuls disséminés au-delà de ~90h).
    # Réutilise _open_data_dernier_run() plutôt qu'une copie locale (l'ancienne
    # fonction get_latest_run() imbriquée ici était identique à celle-ci).
    ecmwf_date, ecmwf_run = _open_data_dernier_run(besoin_echeance_longue=CONFIG['horizon_h'] > 90)
    if ecmwf_date is None:
        raise RuntimeError('Aucun run Open Data disponible.')

    client = ECMWFClient('ecmwf')
    steps = list(range(0, CONFIG['horizon_h'] + CONFIG['pas_h'], CONFIG['pas_h']))
    date_e = datetime.datetime.strptime(ecmwf_date, '%Y%m%d').strftime('%Y-%m-%d')
    grib = f"/content/ecmwf_{ecmwf_date}_{ecmwf_run}z_{CONFIG['horizon_h']}h_t2_d2_tp_u10_v10.grib2"

    if not os.path.exists(grib):
        client.retrieve(date=date_e, time=int(ecmwf_run), step=steps, type='fc',
                         param=['2t', '2d', 'tp', '10u', '10v'], target=grib)

    import cfgrib
    ds_list = cfgrib.open_datasets(grib)
    ds_t = ds_d = ds_u = ds_tp = None
    for ds in ds_list:
        vars_ds = list(ds.data_vars)
        if 't2m' in vars_ds:
            ds_t = ds
        if 'd2m' in vars_ds:
            ds_d = ds
        if 'u10' in vars_ds and 'v10' in vars_ds:
            ds_u = ds
        if 'tp' in vars_ds:
            ds_tp = ds
    if ds_t is None:
        raise ValueError('Variable t2m introuvable dans le fichier GRIB.')

    data = {}
    # Use ALL_COMMUNES_GEODATA here
    for nom, dept, lat, lon in ALL_COMMUNES_GEODATA:
        rows = []
        valid_times = pd.to_datetime(ds_t.valid_time.values)
        for i, valid_time in enumerate(valid_times):
            def val(ds, var, offset=0.0):
                if ds is None:
                    return np.nan
                return float(ds[var].isel(step=i).interp(latitude=lat, longitude=lon, method='linear').values) - offset
            t2 = val(ds_t, 't2m', 273.15)
            td2 = val(ds_d, 'd2m', 273.15) if ds_d is not None else np.nan
            u10 = val(ds_u, 'u10') if ds_u is not None else np.nan
            v10 = val(ds_u, 'v10') if ds_u is not None else np.nan
            wind = np.sqrt(u10**2 + v10**2) * 3.6 if not np.isnan(u10) and not np.isnan(v10) else np.nan
            rh = calc_rh_from_t_td(t2, td2)
            tp_acc = max(val(ds_tp, 'tp') * 1000, 0) if ds_tp is not None else 0
            at_ressenti = calc_temp_ressentie(t2, td2, wind)
            rows.append({'time': utc_to_local_naive(valid_time), 't2m_C': round(t2, 1),
                         'td2m_C': round(td2, 1) if not np.isnan(td2) else np.nan,
                         'at_C': round(at_ressenti, 1) if not np.isnan(at_ressenti) else np.nan,
                         'tp_cumul_mm': round(tp_acc, 2),
                         'wind_kmh': round(wind, 1) if not np.isnan(wind) else np.nan,
                         'rh_pct': round(rh, 1) if not np.isnan(rh) else np.nan})
        df = pd.DataFrame(rows).sort_values('time').reset_index(drop=True)
        df = deaccumuler_tp(df)
        df_window = filtrer_periode_locale(df)
        data[nom] = df_window.reset_index(drop=True)

    return data, 'ECMWF Open Data (0.25°, repli)'


# Prepare the list of all communes with their centroid coordinates
ALL_COMMUNES_GEODATA = []
for idx, row in gdf_com_clean.iterrows():
    try:
        # Ensure geometry is valid and extract centroid
        if row.geometry and row.geometry.is_valid and not row.geometry.is_empty:
            centroid = row.geometry.centroid
            ALL_COMMUNES_GEODATA.append((row['commune'], row['dept'], centroid.y, centroid.x))
        else:
            print(f"⚠️ Géométrie invalide ou vide pour la commune {row['commune']}. Ignorée pour l'interpolation.")
    except Exception as e:
        print(f"⚠️ Erreur lors du calcul du centroïde pour la commune {row['commune']}: {e}. Ignorée pour l'interpolation.")

print(f"Préparation des données pour {len(ALL_COMMUNES_GEODATA)} communes.")

# ── Exécution : ECPDS prioritaire, Open Data en repli ──
print('🌐 Chargement des données ECMWF (source prioritaire : dissémination ECPDS 0.1°)...')

GRID_SERIES = None  # rempli uniquement par charger_donnees_ecpds() ; sinon,
GRID_META = None    # C6bis (carte raster) repasse en mode commune classique.

try:
    data, SOURCE_DONNEES = charger_donnees_ecpds()
except Exception as e:
    print(f'⚠️ ECPDS indisponible ({e}) — repli sur Open Data.')
    print('   ⚠️ La carte raster (grille) ne sera pas disponible avec cette source ;')
    print('      C6bis repassera automatiquement en rendu par commune.')
    try:
        data, SOURCE_DONNEES = charger_donnees_open_data()
    except Exception as e2:
        raise RuntimeError(
            f"ECPDS et Open Data ECMWF sont indisponibles. Bulletin non généré.\nECPDS : {e}\nOpen Data : {e2}"
        ) from e2

print(f'✅ Source utilisée : {SOURCE_DONNEES}')
for ville, df in data.items():
    if not df.empty:
        print(f"   {ville:<12} {df['time'].min()} → {df['time'].max()} | {len(df)} pas")

Préparation des données pour 77 communes.
🌐 Chargement des données ECMWF (source prioritaire : dissémination ECPDS 0.1°)...
Mot de passe ECPDS pour 'wmo_bj' : ··········
✅ Run ECPDS utilisé : 2026-09-24 00Z (31 échéances FSD trouvées)
⚠️ Couverture ECPDS incomplète : dernière échéance 2026-09-27 18Z, il manque 77h pour couvrir le bulletin jusqu'au 2026-10-01 00Z (dissémination probablement encore en cours pour ce run).
   🔁 Complément Open Data pour 25 échéance(s) manquante(s)...
   ℹ️ Échéances Open Data recalées sur la grille disponible (3h jusqu'à 144h, puis 6h) : [129, 132, 135, 138, 141, 144, 147, 150, 153, 156, 159, 162, 165, 168, 171, 174, 177, 180, 183, 186, 189, 192, 195, 198, 201] -> [129, 132, 135, 138, 141, 144, 150, 156, 162, 168, 174, 180, 186, 192, 198]


<multiple>:   0%|          | 0.00/55.1M [00:00<?, ?B/s]

By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.
   ✏️  GRID_SERIES complété avec 15 échéance(s) Open Data régrillées sur la grille Bénin (0.25° -> 0.1°, run 20260922 12Z).
   ✅ Complément Open Data appliqué.
✅ Source utilisée : ECMWF Dissémination ECPDS (2026-09-24 00Z, 0.1°) + Open Data 0.25° (repli pour 27/09 18h→01/10 00h)
   Abomey       2026-09-24 01:00:00 → 2026-09-30 19:00:00 | 46 pas
   Abomey-Calavi 2026-09-24 01:00:00 → 2026-09-30 19:00:00 | 46 pas
   Adja-Ouere   2026-09-24 01:00:00 → 2026-09-30 19:00:00 | 46 pas
   Adjarra      2026-09-24 01:00:00 → 2026-09-30 19:00:00 | 46 pas
   Adjohoun     2026-09-24 01:00:00 → 2026-09-30 19:00:00 | 46 pas
   Agbangnizoun 2026-09-24 01:00:00 → 2026-09-30 19:00:00 | 46 pas
   Aguegues     2026-09-24 01:00:00 → 2026-09-30 19:00:00 | 46 pas
   Akpo-Misserete 2026-09-24 01:00:00 → 2026-09-30 19:00:00 | 46 pas
   Allada

In [8]:
# CONTRÔLE OPÉRATIONNEL DES DONNÉES
print('🔎 Contrôle temporel')
for nom, df in data.items():
    if df.empty:
        print(f'⚠️ {nom}: aucune donnée')
        continue
    debut, fin = df['time'].min(), df['time'].max()
    if debut > PERIODE_DEBUT or fin < PERIODE_FIN - pd.Timedelta(hours=CONFIG['pas_h']):
        print(f'⚠️ {nom}: couverture {debut} → {fin}')
print('ℹ️ Timestamps normalisés en heure locale du Bénin (UTC+1).')


🔎 Contrôle temporel
⚠️ Abomey: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Abomey-Calavi: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Adja-Ouere: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Adjarra: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Adjohoun: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Agbangnizoun: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Aguegues: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Akpo-Misserete: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Allada: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Aplahoue: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Athieme: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Avrankou: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Banikoara: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Bante: couverture 2026-09-24 01:00:00 → 2026-09-30 19:00:00
⚠️ Bassila: couverture 2026-09-24 01:00:00 → 20

# C6bis — Vigilance sur grille (raster 0.1°, Bénin)

S'exécute après C4 (`gdf_dept`) et C5 (`GRID_SERIES`/`GRID_META`). Calcule le niveau de vigilance IBF/OMM maille par maille plutôt qu'une seule valeur par commune. Repli automatique sur le rendu par commune si la grille n'est pas disponible (source Open Data).

In [10]:
# ════════════════════════════════════════════════════════════════
# C6bis — VIGILANCE SUR GRILLE (raster 0.1°, Bénin)
#
# S'exécute après C4 (gdf_dept) et C5 (GRID_SERIES/GRID_META).
# Calcule, pour chaque maille de la grille ECPDS recadrée sur le
# Bénin, le niveau de vigilance IBF/OMM (0-4) par jour — au lieu
# d'une seule valeur par commune issue d'une ville de référence.
#
# Si GRID_SERIES est indisponible (repli Open Data en C5),
# cette cellule ne fait rien : C8/C9 repassent automatiquement en
# rendu par commune (comportement précédent, inchangé).
# ════════════════════════════════════════════════════════════════

from matplotlib.path import Path as _MplPath
from shapely.geometry import MultiPolygon as _MultiPolygon, Polygon as _Polygon

GRID_VIG = None

if GRID_SERIES is None or GRID_META is None:
    print('⚠️ Pas de grille ECPDS disponible (repli Open Data en C5).')
    print('   → C8/C9 utiliseront le rendu par commune (comportement précédent).')
else:
    print('📐 Calcul de la vigilance sur grille (0.1°)...')

    lats = GRID_META['lats']            # 1D, décroissant (nord -> sud)
    lons = GRID_META['lons']            # 1D, croissant (ouest -> est)
    LON2D, LAT2D = np.meshgrid(lons, lats)   # (n_lat, n_lon)
    n_lat, n_lon = LAT2D.shape
    points_grille = np.column_stack([LON2D.ravel(), LAT2D.ravel()])

    # ── 1. Département de chaque maille (test point-dans-polygone) ──
    def _mask_polygone(geom, points):
        """Renvoie un masque booléen (points dans geom), gère Polygon/MultiPolygon."""
        polys = list(geom.geoms) if isinstance(geom, _MultiPolygon) else [geom]
        masque = np.zeros(len(points), dtype=bool)
        for poly in polys:
            if not isinstance(poly, _Polygon) or poly.is_empty:
                continue
            chemin = _MplPath(np.asarray(poly.exterior.coords))
            masque |= chemin.contains_points(points)
        return masque

    GRID_DEPT = np.full((n_lat, n_lon), None, dtype=object)
    for _, drow in gdf_dept.iterrows():
        m = _mask_polygone(drow.geometry, points_grille).reshape(n_lat, n_lon)
        GRID_DEPT[m] = drow['name']

    n_hors_pays = int((GRID_DEPT == None).sum())
    print(f'   {n_lat*n_lon - n_hors_pays} mailles dans le territoire national '
          f'({n_hors_pays} hors frontière, masquées).')

    # ── 2. Seuil de température unique national (cf. C3, SEUILS_CHALEUR) ──
    # Un seul seuil pour tout le pays est correct : l'entrée est le Tmax
    # RESSENTI (at_stack plus bas), qui intègre déjà humidité et vent —
    # pas besoin de re-varier le seuil par région par-dessus.
    S1, S2, S3, S4 = SEUILS_CHALEUR

    # ── 3. Déaccumulation de tp par maille, sur toute la série ────────
    temps_tries = sorted(GRID_SERIES.keys())
    tp_cumul_stack = np.stack(
        [np.maximum(GRID_SERIES[t].get('tp', np.zeros((n_lat, n_lon))) * 1000, 0)
         for t in temps_tries], axis=0
    )  # m -> mm, cumulé depuis le début du run
    tp_pas = np.diff(tp_cumul_stack, axis=0, prepend=tp_cumul_stack[[0]])
    tp_pas[tp_pas < 0] = tp_cumul_stack[np.where(tp_pas < 0)]  # garde-fou identique à C5
    tp_pas = np.clip(tp_pas, 0, None)

    t2m_stack = np.stack(
        [GRID_SERIES[t]['2t'] - 273.15 for t in temps_tries], axis=0
    )

    # ── Température ressentie (comme en C5/C7) : Ta + 0.33e - 0.70*ws - 4.00
    # (formule BOM australien), calculée sur toute la grille pour la vigilance
    # chaleur — demande explicite : "pour la température, il s'agit du ressenti".
    d2m_stack = np.stack(
        [GRID_SERIES[t]['2d'] - 273.15 for t in temps_tries], axis=0
    )
    u10_stack = np.stack([GRID_SERIES[t]['10u'] for t in temps_tries], axis=0)
    v10_stack = np.stack([GRID_SERIES[t]['10v'] for t in temps_tries], axis=0)
    ws_stack = np.sqrt(u10_stack**2 + v10_stack**2)  # m/s
    e_stack = 6.112 * np.exp((17.67 * d2m_stack) / (d2m_stack + 243.5))
    at_stack = t2m_stack + 0.33 * e_stack - 0.70 * ws_stack - 4.00

    # ── 4. Agrégation journalière par maille, en heure locale ─────────
    GRID_VIG = []
    for j_idx in range(CONFIG['nb_jours_cartes']):
        day_start = PERIODE_DEBUT + pd.Timedelta(days=j_idx)
        day_end = PERIODE_DEBUT + pd.Timedelta(days=j_idx + 1)
        sel = [i for i, t in enumerate(temps_tries) if day_start <= t < day_end]

        if not sel:
            GRID_VIG.append(np.full((n_lat, n_lon), np.nan))
            print(f'⚠️ J{j_idx+1} : aucune échéance disponible dans la fenêtre locale.')
            continue

        pluie_24h = tp_pas[sel].sum(axis=0)
        tmax = at_stack[sel].max(axis=0)  # ressenti, pas la température brute

        # niveau_pluie vectorisé (nouveaux seuils : <10 / 10-<25 / 25-<50 / 50-<75 / >=75 mm)
        niv_pluie = np.zeros((n_lat, n_lon), dtype=int)
        niv_pluie[pluie_24h >= 10.0] = 1
        niv_pluie[pluie_24h >= 25.0] = 2
        niv_pluie[pluie_24h >= 50.0] = 3
        niv_pluie[pluie_24h >= 75.0] = 4

        # niveau_temperature vectorisé (seuil national unique, cf. C3)
        niv_temp = np.zeros((n_lat, n_lon), dtype=int)
        niv_temp[tmax >= S1] = 1
        niv_temp[tmax >= S2] = 2
        niv_temp[tmax >= S3] = 3
        niv_temp[tmax >= S4] = 4

        # niveau_vent vectorisé (mêmes seuils que C3 : 30 / 45 / 60 / 75 km/h)
        vent_kmh_jour = (ws_stack[sel] * 3.6).max(axis=0)
        niv_vent = np.zeros((n_lat, n_lon), dtype=int)
        niv_vent[vent_kmh_jour > 30.0] = 1
        niv_vent[vent_kmh_jour >= 45.0] = 2
        niv_vent[vent_kmh_jour >= 60.0] = 3
        niv_vent[vent_kmh_jour >= 75.0] = 4

        niv = np.maximum(np.maximum(niv_pluie, niv_temp), niv_vent).astype(float)
        niv[GRID_DEPT == None] = np.nan  # hors territoire national -> transparent

        GRID_VIG.append(niv)

    # ── 5. Étendue pour affichage (imshow) ─────────────────────────────
    demi = GRID_META['dlat'] / 2
    GRID_EXTENT = [lons.min() - demi, lons.max() + demi,
                   lats.min() - demi, lats.max() + demi]  # [ouest, est, sud, nord]

    # ── 6. Cumul pluviométrique sur toute la période (mm) — pour la carte de synthèse ──
    GRID_CUMUL_PLUIE = tp_pas.sum(axis=0)
    GRID_CUMUL_PLUIE[GRID_DEPT == None] = np.nan

    # ── 7. Vigilance maximale sur toute la période — pour la carte de synthèse ──
    GRID_VIG_MAX = np.nanmax(np.stack(GRID_VIG, axis=0), axis=0)
    GRID_VIG_MAX[GRID_DEPT == None] = np.nan

    print(f'✅ Vigilance calculée sur {len(GRID_VIG)} jours, grille {n_lat}×{n_lon}.')
    print(f'✅ Cumul pluviométrique période : min {np.nanmin(GRID_CUMUL_PLUIE):.0f} mm, '
          f'max {np.nanmax(GRID_CUMUL_PLUIE):.0f} mm.')

📐 Calcul de la vigilance sur grille (0.1°)...
   952 mailles dans le territoire national (1496 hors frontière, masquées).
✅ Vigilance calculée sur 7 jours, grille 68×36.
✅ Cumul pluviométrique période : min 19 mm, max 125 mm.


In [11]:
import copy
import unicodedata
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Dictionnaires d'ajustement global
if 'AJUST_DEPT' not in globals(): AJUST_DEPT = {}
if 'AJUST_COMMUNE' not in globals(): AJUST_COMMUNE = {}
if 'AJUST_PICTO_DEPT' not in globals(): AJUST_PICTO_DEPT = {}
if 'AJUST_PICTO_COMMUNE' not in globals(): AJUST_PICTO_COMMUNE = {}

# __ Ajout pour la conversion du Niveau Proposé en entier __
_REVERSE_SEUIL_LABEL = {v: k for k, v in SEUIL_LABEL.items()} if 'SEUIL_LABEL' in globals() else {}

# ── 1. PRÉ-CALCUL DES CUMULS ET DÉTAILS PAR VILLE DE RÉFÉRENCE ─────
VILLE_DETAILS = {}
for nom, dept, lat, lon in CONFIG['villes']:
    # Recherche souple dans data
    data_key = next((k for k in data.keys() if norm_txt(k) == norm_txt(nom)), None) if 'data' in globals() else None

    if not data_key:
        print(f"⚠️ Données ECMWF manquantes pour la ville de référence : {nom}.")
        continue

    df_ville = data[data_key]
    daily_details = []

    for j_idx in range(CONFIG['nb_jours_cartes']):
        day_start = PERIODE_DEBUT + pd.Timedelta(days=j_idx)
        day_end = PERIODE_DEBUT + pd.Timedelta(days=j_idx + 1)

        df_jour = df_ville[(df_ville['time'] >= day_start) & (df_ville['time'] < day_end)].copy()

        pluie_24h = float(df_jour['tp_mm'].sum()) if (not df_jour.empty and 'tp_mm' in df_jour.columns) else np.nan

        if not df_jour.empty and 'at_C' in df_jour.columns and not df_jour['at_C'].dropna().empty:
            tmax = float(df_jour['at_C'].max())
        elif not df_jour.empty and 't2m_C' in df_jour.columns and not df_jour['t2m_C'].dropna().empty:
            tmax = float(df_jour['t2m_C'].max())
        else:
            tmax = np.nan

        tmin = float(df_jour['t2m_C'].min()) if (not df_jour.empty and 't2m_C' in df_jour.columns and not df_jour['t2m_C'].dropna().empty) else np.nan
        wind_max = float(df_jour['wind_kmh'].max()) if (not df_jour.empty and 'wind_kmh' in df_jour.columns) else np.nan

        daily_details.append({
            'pluie_24h': round(pluie_24h, 1),
            'tmax': round(tmax, 1) if not np.isnan(tmax) else np.nan,
            'tmin': round(tmin, 1) if not np.isnan(tmin) else np.nan,
            'wind_max': round(wind_max, 1),
            'df_jour_hourly': df_jour
        })
    VILLE_DETAILS[nom] = daily_details

# ── 2. ASSOCIATION COMMUNE -> DETAILS DE LA VILLE DE RÉFÉRENCE ────
COMMUNE_REF_CITY_DETAILS = {}
for _, row in gdf_com.iterrows():
    commune_name = row['commune']
    dept_name = canon_dept(row['dept'])

    ref_city_name = None
    for v_nom, v_dept, _, _ in CONFIG['villes']:
        if canon_dept(v_dept) == dept_name:
            ref_city_name = v_nom
            break

    if ref_city_name and ref_city_name in VILLE_DETAILS:
        COMMUNE_REF_CITY_DETAILS[commune_name] = VILLE_DETAILS[ref_city_name]
    else:
        # Fallback de sécurité
        ref_trouvee = next((v for k, v in VILLE_DETAILS.items() if ref_city_name and norm_txt(k) == norm_txt(ref_city_name)), None)
        if ref_trouvee:
            COMMUNE_REF_CITY_DETAILS[commune_name] = ref_trouvee
        else:
            COMMUNE_REF_CITY_DETAILS[commune_name] = [
                {'pluie_24h': np.nan, 'tmax': np.nan, 'tmin': np.nan, 'wind_max': np.nan, 'df_jour_hourly': pd.DataFrame()}
                for _ in range(CONFIG['nb_jours_cartes'])
            ]

COM_VIG = {}
COM_PICTO = {}
COM_PICTO_AUTO = {}

# ── 3. GRILLE DE VIGILANCE COMMUNE 0.1° ────────────────────────────
GRID_NIV_COMMUNE_JOUR = None
if 'GRID_VIG' in globals() and GRID_VIG is not None and 'GRID_META' in globals() and GRID_META is not None:
    from matplotlib.path import Path as _MplPath
    from shapely.geometry import MultiPolygon as _MultiPolygon, Polygon as _Polygon

    def _mask_polygone_c7(geom, points):
        polys = list(geom.geoms) if isinstance(geom, _MultiPolygon) else [geom]
        masque = np.zeros(len(points), dtype=bool)
        for poly in polys:
            if not isinstance(poly, _Polygon) or poly.is_empty:
                continue
            masque |= _MplPath(np.asarray(poly.exterior.coords)).contains_points(points)
        return masque

    _lats_c7 = GRID_META['lats']; _lons_c7 = GRID_META['lons']
    _LON2D_c7, _LAT2D_c7 = np.meshgrid(_lons_c7, _lats_c7)
    _n_lat_c7, _n_lon_c7 = _LAT2D_c7.shape
    _points_grille_c7 = np.column_stack([_LON2D_c7.ravel(), _LAT2D_c7.ravel()])

    GRID_COMMUNE_C7 = np.full((_n_lat_c7, _n_lon_c7), None, dtype=object)
    for _, _crow in gdf_com_clean.iterrows():
        _m = _mask_polygone_c7(_crow.geometry, _points_grille_c7).reshape(_n_lat_c7, _n_lon_c7)
        GRID_COMMUNE_C7[_m] = _crow['commune']

    GRID_NIV_COMMUNE_JOUR = {}
    for _, _crow in gdf_com_clean.iterrows():
        _nom_com = _crow['commune']
        _m = (GRID_COMMUNE_C7 == _nom_com)
        if _m.any():
            GRID_NIV_COMMUNE_JOUR[_nom_com] = [
                int(np.nanmax(GRID_VIG[_j][_m])) if np.isfinite(GRID_VIG[_j][_m]).any() else 0
                for _j in range(CONFIG['nb_jours_cartes'])
            ]

def recalculer_com_vig_picto():
    for _, row in gdf_com.iterrows():
        commune = row['commune']
        dept = canon_dept(row['dept'])

        com_vig_list = [0] * CONFIG['nb_jours_cartes']
        com_picto_list = [None] * CONFIG['nb_jours_cartes']
        com_picto_auto_list = [None] * CONFIG['nb_jours_cartes']

        daily_details_for_commune = COMMUNE_REF_CITY_DETAILS.get(commune,
            [{'pluie_24h': np.nan, 'tmax': np.nan, 'tmin': np.nan, 'wind_max': np.nan, 'df_jour_hourly': pd.DataFrame()}
             for _ in range(CONFIG['nb_jours_cartes'])]
        )

        for j_idx in range(CONFIG['nb_jours_cartes']):
            details = daily_details_for_commune[j_idx]

            pluie_24h = details['pluie_24h']
            tmax = details['tmax']
            df_jour_hourly = details['df_jour_hourly']

            if GRID_NIV_COMMUNE_JOUR is not None and commune in GRID_NIV_COMMUNE_JOUR:
                vig_auto = GRID_NIV_COMMUNE_JOUR[commune][j_idx]
            else:
                vig_auto = niveau_vigilance(pluie_24h, tmax, details.get('wind_max')) if 'niveau_vigilance' in globals() else 0

            picto_auto = picto_auto_jour(df_jour_hourly, pluie_24h, tmax, dept) if 'picto_auto_jour' in globals() else None

            vig_final = vig_auto
            if dept in AJUST_DEPT and j_idx < len(AJUST_DEPT[dept]) and AJUST_DEPT[dept][j_idx] is not None:
                vig_final = AJUST_DEPT[dept][j_idx]

            picto_final = picto_auto
            if dept in AJUST_PICTO_DEPT and j_idx < len(AJUST_PICTO_DEPT[dept]) and AJUST_PICTO_DEPT[dept][j_idx] is not None:
                picto_adj = normaliser_picto(AJUST_PICTO_DEPT[dept][j_idx]) if 'normaliser_picto' in globals() else AJUST_PICTO_DEPT[dept][j_idx]
                picto_final = picto_auto if picto_adj == 'auto' else (None if picto_adj == 'aucun' else picto_adj)

            if commune in AJUST_COMMUNE and j_idx < len(AJUST_COMMUNE[commune]) and AJUST_COMMUNE[commune][j_idx] is not None:
                vig_final = AJUST_COMMUNE[commune][j_idx]

            # --- APPLICATION SYSTÉMATIQUE DU NIVEAU PROPOSÉ (MATRICE DE RISQUE) ---
            # Si un 'Niveau Proposé' est disponible pour cette commune et ce jour, il a la priorité.
            if 'NIVEAU_MATRICE_RISQUE' in globals():
                key_matrix = (commune, j_idx)
                if key_matrix in NIVEAU_MATRICE_RISQUE:
                    niveau_prop_str = NIVEAU_MATRICE_RISQUE[key_matrix]
                    # Convertir le libellé en niveau entier (ex: 'modéré' -> 2)
                    niveau_prop_int = _REVERSE_SEUIL_LABEL.get(niveau_prop_str)
                    if niveau_prop_int is not None:
                        vig_final = niveau_prop_int
            # --- FIN DE L'APPLICATION SYSTÉMATIQUE ---

            com_vig_list[j_idx] = vig_final
            com_picto_list[j_idx] = picto_final
            com_picto_auto_list[j_idx] = picto_auto

        COM_VIG[commune] = com_vig_list
        COM_PICTO[commune] = com_picto_list
        COM_PICTO_AUTO[commune] = com_picto_auto_list

recalculer_com_vig_picto()

def grille_vig_ajustee(j_idx):
    if GRID_VIG is None: return None
    grille = GRID_VIG[j_idx].copy()
    if 'GRID_DEPT' in globals() and GRID_DEPT is not None:
        for _dept, _vals in AJUST_DEPT.items():
            if j_idx < len(_vals) and _vals[j_idx] is not None:
                grille[GRID_DEPT == _dept] = _vals[j_idx]
    if 'GRID_COMMUNE_C7' in globals() and GRID_COMMUNE_C7 is not None:
        for _com, _vals in AJUST_COMMUNE.items():
            if j_idx < len(_vals) and _vals[j_idx] is not None:
                grille[GRID_COMMUNE_C7 == _com] = _vals[j_idx]
    return grille

print('✅ C7 prêt : Données de cumul, vigilance et pictogrammes pré-calculés.')

✅ C7 prêt : Données de cumul, vigilance et pictogrammes pré-calculés.


In [12]:
# ════════════════════════════════════════════════════════════════
# C6ter — VRAISEMBLANCE DE LA PRÉVISION (AVEC SAUVEGARDE DU DASHBOARD SUR DRIVE)
# ════════════════════════════════════════════════════════════════

CONFIG.setdefault('activer_vraisemblance', False)
CONFIG.setdefault('archive_drive_dir', '/content/drive/MyDrive/Vigilance_Meteo_Benin/archive_vraisemblance')

VRAISEMBLANCE_PAR_COMMUNE = {}
NIVEAU_MATRICE_RISQUE = {}

if 'COM_VIG' not in globals() or not COM_VIG:
    print("⚠️ COM_VIG indisponible — C6ter (vraisemblance) sauté, aucun impact sur le reste du notebook.")
else:
    import os as _os
    import pandas as _pd
    import datetime as _dt
    from IPython.display import display, HTML

    # ── 1. Montage Drive (une seule fois par session) ──────────────────
    _drive_ok = False
    try:
        if not _os.path.isdir('/content/drive/MyDrive'):
            from google.colab import drive as _gdrive
            _gdrive.mount('/content/drive')
        _drive_ok = _os.path.isdir('/content/drive/MyDrive')
    except Exception as _e:
        print(f"⚠️ Montage Drive impossible ({_e}) — archive vraisemblance désactivée pour cette exécution.")

    if _drive_ok:
        _os.makedirs(CONFIG['archive_drive_dir'], exist_ok=True)
        ARCHIVE_PATH = _os.path.join(CONFIG['archive_drive_dir'], 'archive_previsions.csv')
        ARCHIVE_TMP_PATH = ARCHIVE_PATH + '.tmp'
        ARCHIVE_COLS = ['date_run', 'date_cible', 'commune', 'niveau_prevu']

        # ── 2. Date du run du jour ──────────────────────────────────────
        _date_run = CONFIG.get('ecmwf_date', 'auto')
        if _date_run == 'auto':
            _date_run = _dt.date.today().strftime('%Y%m%d')

        # ── 3. Lecture / Initialisation de l'archive ────────────────────
        if _os.path.exists(ARCHIVE_PATH):
            try:
                archive_df = _pd.read_csv(ARCHIVE_PATH, dtype={'date_run': str, 'date_cible': str})
            except Exception as _e:
                archive_df = _pd.DataFrame(columns=ARCHIVE_COLS)
        else:
            archive_df = _pd.DataFrame(columns=ARCHIVE_COLS)

        # ── 4. Lignes du run d'aujourd'hui ──────────────────────────────
        _lignes_du_jour = []
        for commune, niveaux_jours in COM_VIG.items():
            for j_idx, niveau in enumerate(niveaux_jours):
                date_cible = (PERIODE_DEBUT + _pd.Timedelta(days=j_idx)).strftime('%Y%m%d')
                _lignes_du_jour.append({
                    'date_run': _date_run, 'date_cible': date_cible,
                    'commune': commune, 'niveau_prevu': int(niveau),
                })
        nouvelles_lignes_df = _pd.DataFrame(_lignes_du_jour, columns=ARCHIVE_COLS)

        # ── 5. Fusion & Écriture atomique ───────────────────────────────
        archive_df = _pd.concat([archive_df, nouvelles_lignes_df], ignore_index=True)
        archive_df = archive_df.drop_duplicates(subset=['date_run', 'date_cible', 'commune'], keep='last')

        try:
            archive_df.to_csv(ARCHIVE_TMP_PATH, index=False)
            _os.replace(ARCHIVE_TMP_PATH, ARCHIVE_PATH)
        except Exception as _e:
            pass

        # ── 6. Matrice de Risque & Couleurs Charte ──────────────────────
        SEUIL_LABEL = {0: 'néant', 1: 'faible', 2: 'modéré', 3: 'grave', 4: 'sévère'}

        COLOR_MAP = {
            'néant':  {'bg': '#FFFFFF', 'txt': '#000000', 'border': '#ccc'},
            'faible': {'bg': '#52BFCB', 'txt': '#000000', 'border': '#3da2ae'},
            'modéré': {'bg': '#ECE12B', 'txt': '#000000', 'border': '#c8bd17'},
            'grave':  {'bg': '#FF9800', 'txt': '#000000', 'border': '#e08600'},
            'sévère': {'bg': '#E53935', 'txt': '#FFFFFF', 'border': '#b71c1c'},
        }

        RISK_MATRIX_PLAFOND = {
            'faible':  {'faible': 'faible', 'modéré': 'faible', 'grave': 'modéré', 'sévère': 'grave'},
            'moyenne': {'faible': 'faible', 'modéré': 'modéré', 'grave': 'grave',  'sévère': 'sévère'},
            'élevée':  {'faible': 'faible', 'modéré': 'modéré', 'grave': 'grave',  'sévère': 'sévère'},
        }

        commune_dept_map = {}
        if 'gdf_com_clean' in globals():
            commune_dept_map = {row['commune']: canon_dept(row['dept']) for _, row in gdf_com_clean.iterrows()}

        rapport_rows = []

        # Remplissage trié par JOUR (j_idx) -> DÉPARTEMENT -> COMMUNE
        for j_idx in range(CONFIG.get('nb_jours_cartes', 7)):
            echeance_j = j_idx + 1
            date_cible_str = (PERIODE_DEBUT + _pd.Timedelta(days=j_idx)).strftime('%d/%m')

            communes_triees = sorted(
                COM_VIG.keys(),
                key=lambda c: (commune_dept_map.get(c, 'Z_Inconnu'), c)
            )

            for commune in communes_triees:
                niveaux_jours = COM_VIG[commune]
                if j_idx >= len(niveaux_jours): continue

                niveau = niveaux_jours[j_idx]
                if niveau < 2:
                    continue

                sev = SEUIL_LABEL[niveau]
                dept = commune_dept_map.get(commune, 'Bénin')

                # Échéance
                if echeance_j <= 3: classe_echeance = 'élevée'
                elif echeance_j <= 5: classe_echeance = 'moyenne'
                else: classe_echeance = 'faible'

                # Cohérence inter-runs
                date_cible = (PERIODE_DEBUT + _pd.Timedelta(days=j_idx)).strftime('%Y%m%d')
                runs_prec = archive_df[
                    (archive_df['commune'] == commune) &
                    (archive_df['date_cible'] == date_cible) &
                    (archive_df['date_run'] != _date_run)
                ].sort_values('date_run')

                _froid = len(runs_prec) == 0
                if _froid:
                    classe_coherence = 'moyenne'
                else:
                    niveaux_prec = runs_prec['niveau_prevu'].tolist()[-2:]
                    if len(niveaux_prec) >= 2 and all(n >= 2 for n in niveaux_prec):
                        classe_coherence = 'élevée'
                    elif any(n >= 2 for n in niveaux_prec):
                        classe_coherence = 'moyenne'
                    else:
                        classe_coherence = 'faible'

                # Marge au seuil
                classe_marge = 'élevée' if niveau == 4 else ('moyenne' if niveau == 3 else 'faible')

                # Synthèse Vraisemblance
                classes = [classe_echeance, classe_coherence, classe_marge]
                ordre = {'faible': 0, 'moyenne': 1, 'élevée': 2}
                vraisemblance = min(classes, key=lambda c: ordre[c])

                niveau_final = RISK_MATRIX_PLAFOND[vraisemblance][sev]
                VRAISEMBLANCE_PAR_COMMUNE[(commune, j_idx)] = vraisemblance
                NIVEAU_MATRICE_RISQUE[(commune, j_idx)] = niveau_final

                est_divergent = (sev != niveau_final)
                rapport_rows.append({
                    'j_idx': j_idx,
                    'Jour': f"J+{echeance_j} ({date_cible_str})",
                    'Département': dept,
                    'Commune': commune,
                    'Brut ECMWF': sev,
                    'Vraisemblance': vraisemblance.capitalize(),
                    'Échéance': classe_echeance,
                    'Cohérence': classe_coherence + (" (Froid)" if _froid else ""),
                    'Niveau Proposé': niveau_final,
                    'Action Matrice': 'Plafonné' if est_divergent else 'Conforme',
                    '_divergent': est_divergent
                })

        # ── 7. SAUVEGARDE SUR DRIVE & AFFICHAGE DYNAMIQUE ──────────────
        if rapport_rows:
            df_export = _pd.DataFrame(rapport_rows).drop(columns=['j_idx', '_divergent'])

            # Sauvegarde des tableaux CSV sur Google Drive
            path_latest = _os.path.join(CONFIG['archive_drive_dir'], 'diagnostic_vraisemblance_LATEST.csv')
            path_run = _os.path.join(CONFIG['archive_drive_dir'], f'diagnostic_vraisemblance_{_date_run}.csv')

            try:
                df_export.to_csv(path_latest, index=False, encoding='utf-8-sig')
                df_export.to_csv(path_run, index=False, encoding='utf-8-sig')
                print(f"💾 Dashboard sauvegardé sur Drive : diagnostic_vraisemblance_{_date_run}.csv")
            except Exception as _e:
                print(f"⚠️ Échec de la sauvegarde du dashboard sur Drive : {_e}")

            tot_cas = len(rapport_rows)
            tot_div = sum(1 for r in rapport_rows if r['_divergent'])
            tot_conf = tot_cas - tot_div

            # Agrégation par Département
            df_all = _pd.DataFrame(rapport_rows)
            dept_rows = []
            ordre_seuil = {'néant':0, 'faible':1, 'modéré':2, 'grave':3, 'sévère':4}

            for (j_idx, dept), group in df_all.groupby(['j_idx', 'Département'], sort=False):
                max_brut = max(group['Brut ECMWF'], key=lambda x: ordre_seuil[x])
                max_prop = max(group['Niveau Proposé'], key=lambda x: ordre_seuil[x])
                div = any(group['_divergent'])
                communes_list = ", ".join(group['Commune'].unique())

                dept_rows.append({
                    'Jour': group['Jour'].iloc[0],
                    'Département': dept,
                    'Nb Communes': len(group),
                    'Communes': communes_list,
                    'Brut Max': max_brut,
                    'Niveau Proposé Max': max_prop,
                    'Action Matrice': '⚠️ Plafonné' if div else 'Conforme',
                    '_divergent': div
                })

            html_code = f"""
            <style>
                .kpi-container {{ display: flex; gap: 12px; margin-bottom: 12px; font-family: Arial, sans-serif; }}
                .kpi-card {{ flex: 1; padding: 10px; border-radius: 6px; text-align: center; color: white; font-weight: bold; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
                .kpi-blue {{ background-color: #0C447C; }}
                .kpi-orange {{ background-color: #FF9800; color: #000; }}
                .kpi-green {{ background-color: #2E7D32; }}
                .vraisem-table {{ width: 100%; border-collapse: collapse; font-family: Arial, sans-serif; font-size: 13px; margin-top: 8px; }}
                .vraisem-table th {{ background-color: #0C447C; color: white; padding: 8px; text-align: center; position: sticky; top: 0; z-index: 2; }}
                .vraisem-table td {{ padding: 6px 8px; text-align: center; border: 1px solid #ddd; }}
                .badge-alert {{ background-color: #ffebee; color: #c62828; font-weight: bold; padding: 3px 6px; border-radius: 4px; border: 1px solid #ef9a9a; }}
                .badge-ok {{ background-color: #e8f5e9; color: #2e7d32; padding: 3px 6px; border-radius: 4px; border: 1px solid #a5d6a7; }}
                .cell-vig {{ font-weight: bold; padding: 4px 8px; border-radius: 3px; display: inline-block; width: 80px; }}
                .tab-btn {{ padding: 7px 14px; margin-right: 6px; border: 1px solid #0C447C; background: #e3f2fd; cursor: pointer; border-radius: 4px; font-weight: bold; color: #0C447C; }}
                .tab-btn.active {{ background: #0C447C; color: white; }}
            </style>

            <!-- BOÎTES KPI -->
            <div class="kpi-container">
                <div class="kpi-card kpi-blue">
                    <span style="font-size: 20px;">{tot_cas}</span><br><span style="font-size: 11px; opacity:0.9;">CAS ANALYSÉS (≥ Modéré)</span>
                </div>
                <div class="kpi-card kpi-orange">
                    <span style="font-size: 20px;">{tot_div}</span><br><span style="font-size: 11px; opacity:0.9;">⚠️ PLAFONNÉS PAR MATRICE</span>
                </div>
                <div class="kpi-card kpi-green">
                    <span style="font-size: 20px;">{tot_conf}</span><br><span style="font-size: 11px; opacity:0.9;">✅ CONFORMES MAINTENUS</span>
                </div>
            </div>

            <!-- CHOIX DE VUE -->
            <div style="margin: 10px 0; font-family: Arial, sans-serif;">
                <b>Niveau de visualisation :</b>
                <button class="tab-btn active" id="btnVueDept" onclick="switchVue('dept')">🏢 Vue par Département (Groupée)</button>
                <button class="tab-btn" id="btnVueCom" onclick="switchVue('commune')">📍 Vue Détaillée (Jour ➔ Dept ➔ Commune)</button>
            </div>

            <!-- TABLEAU 1 : VUE DÉPARTEMENT -->
            <div id="boxVueDept" style="max-height: 380px; overflow-y: auto; border: 1px solid #0C447C; border-radius: 6px; background: white;">
            <table class="vraisem-table">
                <thead>
                    <tr>
                        <th>Jour</th>
                        <th>Département</th>
                        <th>Communes Concernées</th>
                        <th>Brut Max</th>
                        <th>Niveau Proposé Max</th>
                        <th>Diagnostic Matrice</th>
                    </tr>
                </thead>
                <tbody>
            """

            prev_jour_dept = None
            for r in dept_rows:
                border_top = "border-top: 2px solid #0C447C;" if (prev_jour_dept and prev_jour_dept != r['Jour']) else ""
                prev_jour_dept = r['Jour']

                tr_style = f"background-color: #fff8e1; {border_top}" if r['_divergent'] else f"{border_top}"
                diag_badge = f"<span class='badge-alert'>{r['Action Matrice']}</span>" if r['_divergent'] else f"<span class='badge-ok'>{r['Action Matrice']}</span>"
                cfg_brut = COLOR_MAP.get(r['Brut Max'], {'bg':'#fff','txt':'#000','border':'#ccc'})
                cfg_ajust = COLOR_MAP.get(r['Niveau Proposé Max'], {'bg':'#fff','txt':'#000','border':'#ccc'})

                html_code += f"""
                <tr style="{tr_style}">
                    <td><b>{r['Jour']}</b></td>
                    <td><b style="color:#0C447C;">{r['Département']}</b></td>
                    <td style="text-align:left; font-size:12px;"><b>{r['Nb Communes']}</b> ({r['Communes']})</td>
                    <td><span class="cell-vig" style="background-color: {cfg_brut['bg']}; color: {cfg_brut['txt']}; border: 1px solid {cfg_brut['border']}">{r['Brut Max'].capitalize()}</span></td>
                    <td><span class="cell-vig" style="background-color: {cfg_ajust['bg']}; color: {cfg_ajust['txt']}; border: 1px solid {cfg_ajust['border']}">{r['Niveau Proposé Max'].capitalize()}</span></td>
                    <td>{diag_badge}</td>
                </tr>
                """

            html_code += """
                </tbody>
            </table>
            </div>

            <!-- TABLEAU 2 : VUE DÉTAILLÉE PAR COMMUNE -->
            <div id="boxVueCom" style="max-height: 380px; overflow-y: auto; border: 1px solid #0C447C; border-radius: 6px; background: white; display: none;">
            <table class="vraisem-table">
                <thead>
                    <tr>
                        <th>Jour</th>
                        <th>Département</th>
                        <th>Commune</th>
                        <th>Niveau Brut</th>
                        <th>Vraisemblance</th>
                        <th>Échéance</th>
                        <th>Cohérence</th>
                        <th>Niveau Ajusté</th>
                        <th>Diagnostic</th>
                    </tr>
                </thead>
                <tbody>
            """

            prev_dept_com = None
            for row in rapport_rows:
                key_dept = f"{row['Jour']}_{row['Département']}"
                border_top = "border-top: 1.5px solid #0C447C;" if (prev_dept_com and prev_dept_com != key_dept) else ""
                prev_dept_com = key_dept

                tr_style = f"background-color: #fff8e1; {border_top}" if row['_divergent'] else f"{border_top}"
                diag_action = '⚠️ Plafonné' if row['_divergent'] else 'Conforme'
                diag_badge = f"<span class='badge-alert'>{diag_action}</span>" if row['_divergent'] else f"<span class='badge-ok'>{diag_action}</span>"
                cfg_brut = COLOR_MAP.get(row['Brut ECMWF'], {'bg':'#fff','txt':'#000','border':'#ccc'})
                cfg_ajust = COLOR_MAP.get(row['Niveau Proposé'], {'bg':'#fff','txt':'#000','border':'#ccc'})

                html_code += f"""
                <tr style="{tr_style}">
                    <td><b>{row['Jour']}</b></td>
                    <td><b style="color:#0C447C;">{row['Département']}</b></td>
                    <td><b>{row['Commune']}</b></td>
                    <td><span class="cell-vig" style="background-color: {cfg_brut['bg']}; color: {cfg_brut['txt']}; border: 1px solid {cfg_brut['border']}">{row['Brut ECMWF'].capitalize()}</span></td>
                    <td><b>{row['Vraisemblance']}</b></td>
                    <td>{row['Échéance']}</td>
                    <td>{row['Cohérence']}</td>
                    <td><span class="cell-vig" style="background-color: {cfg_ajust['bg']}; color: {cfg_ajust['txt']}; border: 1px solid {cfg_ajust['border']}">{row['Niveau Proposé'].capitalize()}</span></td>
                    <td>{diag_badge}</td>
                </tr>
                """

            html_code += """
                </tbody>
            </table>
            </div>

            <script>
            function switchVue(mode) {
                var boxDept = document.getElementById('boxVueDept');
                var boxCom = document.getElementById('boxVueCom');
                var btnDept = document.getElementById('btnVueDept');
                var btnCom = document.getElementById('btnVueCom');

                if (mode === 'dept') {
                    boxDept.style.display = 'block';
                    boxCom.style.display = 'none';
                    btnDept.classList.add('active');
                    btnCom.classList.remove('active');
                } else {
                    boxDept.style.display = 'none';
                    boxCom.style.display = 'block';
                    btnCom.classList.add('active');
                    btnDept.classList.remove('active');
                }
            }
            </script>
            """

            display(HTML(html_code))
        else:
            print("ℹ️ Aucune commune n'atteint le niveau 'Modéré' (Jaune) sur ce run.")

Mounted at /content/drive
💾 Dashboard sauvegardé sur Drive : diagnostic_vraisemblance_20260924.csv


Jour,Département,Communes Concernées,Brut Max,Niveau Proposé Max,Diagnostic Matrice
J+1 (24/09),Alibori,"6 (Banikoara, Gogounou, Kandi, Karimama, Malanville, Segbana)",Modéré,Faible,⚠️ Plafonné
J+1 (24/09),Atacora,"4 (Kerou, Kobli, Materi, Tanguieta)",Modéré,Faible,⚠️ Plafonné
J+1 (24/09),Borgou,"4 (Bembereke, Kalale, Nikki, Sinende)",Modéré,Faible,⚠️ Plafonné
J+2 (25/09),Alibori,"6 (Banikoara, Gogounou, Kandi, Karimama, Malanville, Segbana)",Grave,Grave,⚠️ Plafonné
J+2 (25/09),Atacora,"7 (Boukombe, Kerou, Kobli, Kouande, Materi, Pehunco, Tanguieta)",Modéré,Faible,⚠️ Plafonné
J+2 (25/09),Borgou,"7 (Bembereke, Kalale, N'Dali, Nikki, Parakou, Sinende, Tchaourou)",Modéré,Faible,⚠️ Plafonné
J+2 (25/09),Collines,"5 (Bante, Glazoue, Ouesse, Savalou, Save)",Modéré,Faible,⚠️ Plafonné
J+2 (25/09),Donga,"3 (Bassila, Djougou, Ouake)",Modéré,Faible,⚠️ Plafonné
J+3 (26/09),Alibori,"6 (Banikoara, Gogounou, Kandi, Karimama, Malanville, Segbana)",Modéré,Faible,⚠️ Plafonné
J+3 (26/09),Atacora,"7 (Boukombe, Kerou, Kobli, Kouande, Materi, Pehunco, Tanguieta)",Modéré,Faible,⚠️ Plafonné


### 🖱️ C8 — Ajustement interactif & Visualisation (Cartes & Météorogrammes)

Cette cellule vous permet d'ajuster interactivement les niveaux de vigilance et les pictogrammes par département (et potentiellement par commune) et de visualiser les cartes et les météorogrammes résultants. Vous pouvez d'abord ajuster les prévisions, puis générer les cartes et météorogrammes.

In [13]:
# ════════════════════════════════════════════════════════════════
# C8 — CARTES JOURNALIÈRES (J1-J7) + HELPERS CARTOGRAPHIQUES PARTAGÉS
#      + MÉTÉOROGRAMMES + INTERFACE INTERACTIVE DE VALIDATION
# ════════════════════════════════════════════════════════════════

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch, Circle, Wedge
from shapely.geometry import MultiPolygon, Polygon
import numpy as np, os, datetime, copy, re
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import io as _io

os.makedirs(CONFIG['output_dir'], exist_ok=True)

date_lundi_local = PERIODE_DEBUT
date_fin_local = PERIODE_FIN

date_str = date_lundi_local.strftime('%d/%m/%Y')
date0    = date_lundi_local

print(f'Période : {date_lundi_local.strftime("%d/%m %Hh")} -> '
      f'{date_fin_local.strftime("%d/%m %Hh")} (heure locale du Bénin)')

# ── Listes strictes & ordonnées ────────────────────────────────────
VILLES_METEO_BASE = [
    ('Cotonou',        'Littoral'),
    ('Allada',         'Atlantique'),
    ('Abomey-Calavi',  'Atlantique'),
    ('Porto-Novo',     'Ouémé'),
    ('Pobè',           'Plateau'),
    ('Lokossa',        'Mono'),
    ('Aplahoué',       'Couffo'),
    ('Bohicon',        'Zou'),
    ('Savè',           'Collines'),
    ('Djougou',        'Donga'),
    ('Parakou',        'Borgou'),
    ('Natitingou',     'Atacora'),
    ('Kandi',          'Alibori'),
]

if 'ALL_COMMUNES_GEODATA' in globals() and ALL_COMMUNES_GEODATA:
    VILLES_METEO = [(nom, dept) for nom, dept, *_ in ALL_COMMUNES_GEODATA]
else:
    VILLES_METEO = VILLES_METEO_BASE

_NOMS_DEPT = sorted(list(set([d for _, d in VILLES_METEO_BASE] + (list(gdf_dept['name'].unique()) if 'gdf_dept' in globals() else []))))
_NOMS_COMMUNE = sorted(list(set([v for v, _ in VILLES_METEO] + (list(gdf_com_clean['commune'].unique()) if 'gdf_com_clean' in globals() else []))))

HEURES_AFFICHAGE = [0, 6, 12, 18]

# ================================================================
# PARTIE 1 — CARTES DE VIGILANCE
# ================================================================

VIG_LBL     = {k: v['label'] for k, v in VIG.items()} if 'VIG' in globals() else {0:'Vert', 1:'Vert', 2:'Jaune', 3:'Orange', 4:'Rouge'}
VIG_CLR_LBL = {0:'#555555', 1:'#2D6A27', 2:'#7A5800', 3:'#7F3C00', 4:'#7F0000'}

JOURS_HDR = ['#2E7D32','#B71C1C','#E65100','#1565C0']
JOURS_CFG = [
    {'idx': i,
     'label':   j['label'],
     'debut':   j['debut'],
     'fin':     j['fin'],
     'hdr':     JOURS_HDR[i % len(JOURS_HDR)],
     'hdr_txt':'#FFFFFF',
     'alert': i == 1}
    for i,j in enumerate(JOURS)
]

from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image as PILImage

_ICON_DARK_CACHE = {}

def _icone_noir_fonce(path, couleur=(26, 26, 26)):
    if path in _ICON_DARK_CACHE: return _ICON_DARK_CACHE[path]
    img = PILImage.open(path).convert('RGBA')
    arr = np.array(img)
    arr[..., 0] = couleur[0]; arr[..., 1] = couleur[1]; arr[..., 2] = couleur[2]
    _ICON_DARK_CACHE[path] = arr
    return arr

def load_icon_png(path, zoom=0.06):
    try:
        arr = _icone_noir_fonce(path)
        return OffsetImage(arr, zoom=zoom)
    except Exception: return None

def _trouver_icone(nom):
    candidats = [
        os.path.join(CONFIG.get('icones_dir', '/content'), f'icone_{nom}.png'),
        f'/content/icone_{nom}.png',
        os.path.join(CONFIG['output_dir'], f'icone_{nom}.png'),
        f'icone_{nom}.png',
    ]
    for p in candidats:
        if os.path.exists(p): return p
    return None

ICON_FILES = {c: _trouver_icone(n) for c, n in [('orage','orage'),('pluie','orage'),('chaleur','chaleur'),('vent','vent')]}
ICON_FILES[None] = None
ICON_ZOOM = 0.07

def fill_geom(ax, geom, color, ec='#888888', lw=0.3, zorder=2):
    polys = list(geom.geoms) if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        if isinstance(poly, Polygon) and not poly.is_empty:
            x, y = poly.exterior.xy
            ax.fill(x, y, color=color, zorder=zorder, linewidth=0, antialiased=True)

def draw_boundary(ax, geom, color, lw, zorder=5):
    polys = list(geom.geoms) if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        if isinstance(poly, Polygon) and not poly.is_empty:
            x, y = poly.exterior.xy
            ax.plot(x, y, color=color, lw=lw, zorder=zorder, solid_capstyle='round', solid_joinstyle='round')

def draw_icon_png(ax, cx, cy, picto_code, zoom=ICON_ZOOM):
    path = ICON_FILES.get(picto_code)
    if path is None: return
    ico = load_icon_png(path, zoom=zoom)
    if ico is None: return
    ab = AnnotationBbox(ico, (cx, cy), frameon=False, zorder=9, box_alignment=(0.5, 0.5))
    ax.add_artist(ab)

def draw_north_arrow(ax, x, y, size=0.25, color='#111111'):
    ax.annotate('', xy=(x, y + size), xytext=(x, y), arrowprops=dict(arrowstyle='-|>', color=color, lw=1.6), zorder=50)
    ax.text(x, y + size * 1.18, 'N', ha='center', va='bottom', fontsize=8, fontweight='bold', color=color, zorder=50)

def draw_scale_bar(ax, x, y, lat_ref, km=100, color='#111111'):
    deg = km / (111.32 * np.cos(np.radians(lat_ref)))
    ax.plot([x, x + deg], [y, y], color=color, lw=2.2, zorder=50, solid_capstyle='butt')
    for val in (x, x + deg):
        ax.plot([val, val], [y - 0.03, y + 0.03], color=color, lw=1.2, zorder=50)
    ax.text(x + deg / 2, y - 0.10, f'{km} km', ha='center', va='top', fontsize=6.5, color=color, zorder=50)

_LOGO_TRANSPARENT_CACHE = {}

def _logo_fond_transparent(logo_path, seuil=240):
    if logo_path in _LOGO_TRANSPARENT_CACHE: return _LOGO_TRANSPARENT_CACHE[logo_path]
    img = PILImage.open(logo_path).convert('RGBA')
    arr = np.array(img)
    quasi_blanc = (arr[..., 0] >= seuil) & (arr[..., 1] >= seuil) & (arr[..., 2] >= seuil)
    arr[..., 3] = np.where(quasi_blanc, 0, arr[..., 3])
    _LOGO_TRANSPARENT_CACHE[logo_path] = arr
    return arr

def draw_logo_badge(ax, x, y, zoom=None):
    if zoom is None: zoom = CONFIG.get('logo_zoom_general', 0.035)
    logo_path = CONFIG.get('logo_path')
    if not logo_path or not os.path.exists(logo_path): return
    try:
        arr = _logo_fond_transparent(logo_path)
        ob = OffsetImage(arr, zoom=zoom)
        ab = AnnotationBbox(ob, (x, y), frameon=False, zorder=50)
        ax.add_artist(ab)
    except Exception: pass

def draw_commune_labels(ax, gdf, fontsize=3.2, color='#444444'):
    for _, row in gdf.iterrows():
        if 'VILLES_CARTE' in globals() and row['commune'] in VILLES_CARTE: continue
        try: cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
        except Exception: continue
        ax.text(cx, cy, row['commune'], fontsize=fontsize, ha='center', va='center',
                color=color, zorder=9, alpha=0.85,
                path_effects=[pe.withStroke(linewidth=1.0, foreground='white')])

MAX_VILLES_PAR_DEPT = {'Littoral': 1, 'Alibori': 4, 'Atacora': 5, 'Borgou': 3, 'Collines': 3}
VILLES_CARTE = {}
CHEF_LIEU = {}
_dept_from_config = {}

for nom, dept, lat, lon in CONFIG.get('villes', []):
    VILLES_CARTE[nom] = (lon, lat)
    CHEF_LIEU[canon_dept(dept)] = nom
    _dept_from_config[nom] = canon_dept(dept)

for nom, dept, lat, lon in CONFIG.get('villes_carte_supplementaires', []):
    VILLES_CARTE[nom] = (lon, lat)
    _dept_from_config[nom] = canon_dept(dept)

def draw_villes_carte(ax):
    ville_dept_map = {row['commune']: canon_dept(row['dept']) for _, row in gdf_com_clean.iterrows()}
    ville_dept_map.update(_dept_from_config)

    dept_count = {}; villes_ok = set()
    for ville,(vx,vy) in VILLES_CARTE.items():
        if ville in villes_ok: continue
        dept = ville_dept_map.get(ville, '')
        max_v = MAX_VILLES_PAR_DEPT.get(canon_dept(dept), 2)
        if dept_count.get(dept, 0) >= max_v: continue
        villes_ok.add(ville)
        dept_count[dept] = dept_count.get(dept, 0) + 1
        ax.plot(vx, vy, 'o', ms=2.8, color='#1A1A1A', markeredgecolor='white', markeredgewidth=0.7, zorder=10)
        ha  = 'left' if vx < 2.5 else 'right'
        xof = 0.05 if ha=='left' else -0.05
        ax.text(vx+xof, vy, ville, fontsize=5.2, ha=ha, va='center', color='#111111',
                fontweight='bold', zorder=11,
                path_effects=[pe.withStroke(linewidth=1.8, foreground='white')])

from scipy.ndimage import zoom as _zoom

def lisser_grille(arr, facteur=4, categorique=True):
    masque_nan = np.isnan(arr)
    arr_rempli = np.where(masque_nan, 0, arr)
    arr_zoom = _zoom(arr_rempli, facteur, order=1)
    masque_zoom = _zoom(masque_nan.astype(float), facteur, order=1) > 0.5
    if categorique: arr_zoom = np.round(arr_zoom)
    arr_zoom = arr_zoom.astype(float)
    arr_zoom[masque_zoom] = np.nan
    return arr_zoom

def generer_carte_jour(j_idx):
    jour = JOURS_CFG[j_idx]
    fig = plt.figure(figsize=(5.5, 8.5), dpi=250)
    fig.patch.set_facecolor('white')
    gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[0.045, 0.955],
                           left=0.01, right=0.99, top=0.99, bottom=0.01, hspace=0.01)

    ax_hdr = fig.add_subplot(gs[0])
    ax_hdr.set_facecolor(jour['hdr']); ax_hdr.set_axis_off()
    title = f"VIGILANCE — {jour['label'].upper()}"
    if jour['alert']: title += '  ⚠️'
    ax_hdr.text(0.5, 0.5, title + f"   |   {date_str}",
                transform=ax_hdr.transAxes, fontsize=7.5,
                fontweight='bold', color=jour['hdr_txt'],
                va='center', ha='center')

    ax = fig.add_subplot(gs[1])
    ax.set_facecolor('#C8E6F5')
    ax.set_xlim(0.55, 4.05); ax.set_ylim(5.90, 12.55)
    ax.set_axis_off()

    if 'GRID_VIG' in globals() and GRID_VIG is not None:
        from matplotlib.colors import ListedColormap
        cmap_vig = ListedColormap([VIG_HEX[i] for i in range(5)])
        grille_val = grille_vig_ajustee(j_idx) if 'grille_vig_ajustee' in globals() else GRID_VIG[j_idx]
        grille_jour = np.ma.masked_invalid(lisser_grille(grille_val, facteur=4, categorique=True))
        ax.imshow(grille_jour, extent=GRID_EXTENT, origin='upper',
                  cmap=cmap_vig, vmin=-0.5, vmax=4.5, zorder=2,
                  interpolation='nearest', aspect='auto')
    else:
        for _, row in gdf_com_clean.iterrows():
            com = row['commune']
            n = max(0, min(4, COM_VIG.get(com, [0]*7)[j_idx])) if 'COM_VIG' in globals() else 0
            fill_geom(ax, row.geometry, VIG_HEX[n] if 'VIG_HEX' in globals() else '#2E7D32', zorder=2)

    # ── Contours géographiques ───────────────────────────────────────
    if gdf_arr is not None:
        for _, arow in gdf_arr.iterrows(): draw_boundary(ax, arow.geometry, '#CFCFCF', lw=0.12, zorder=3)
    for _, row in gdf_com_clean.iterrows(): draw_boundary(ax, row.geometry, '#AAAAAA', lw=0.25, zorder=3)
    for _, drow in gdf_dept.iterrows(): draw_boundary(ax, drow.geometry, '#333333', lw=0.90, zorder=5)
    draw_boundary(ax, gdf_dept.dissolve().geometry.iloc[0], '#000000', lw=1.80, zorder=6)

    # ── Tracé des Pictogrammes (Interface Manuelle + Auto C7) ────────
    for _, row in gdf_com_clean.iterrows():
        com = row['commune']
        dept = canon_dept(row.get('dept', '')) if 'canon_dept' in globals() else row.get('dept', '')
        cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
        chef = CHEF_LIEU.get(dept, '')

        # 1. Vérification des ajustements manuels de l'interface
        picto_manuel = None
        if com in AJUST_PICTO_COMMUNE and j_idx < len(AJUST_PICTO_COMMUNE[com]):
            picto_manuel = AJUST_PICTO_COMMUNE[com][j_idx]
        elif dept in AJUST_PICTO_DEPT and j_idx < len(AJUST_PICTO_DEPT[dept]):
            picto_manuel = AJUST_PICTO_DEPT[dept][j_idx]

        # 2. Résolution du pictogramme final
        if picto_manuel and picto_manuel != 'auto':
            picto_final = None if picto_manuel == 'aucun' else picto_manuel
        else:
            # Repli sur les pictogrammes auto calculés par C7
            pictos_com = COM_PICTO.get(com, [None]*7) if 'COM_PICTO' in globals() else [None]*7
            picto_final = pictos_com[j_idx] if j_idx < len(pictos_com) else None

        # 3. Dessin sur la carte
        if picto_final:
            draw_icon_png(ax, cx, cy + 0.04, picto_final, zoom=ICON_ZOOM * 0.85)

    # ── Éléments d'habillage et Sauvegarde ───────────────────────────
    draw_villes_carte(ax)
    draw_commune_labels(ax, gdf_com_clean)
    draw_north_arrow(ax, 0.75, 11.9, size=0.28)
    draw_scale_bar(ax, 0.75, 6.35, lat_ref=9.3, km=100)
    draw_logo_badge(ax, 3.75, 12.05, zoom=CONFIG.get('logo_zoom_general', 0.035))

    fname = f"carte_J{j_idx+1}_{jour['label'].split()[0]}.png"
    path   = f"{CONFIG['output_dir']}/{fname}"
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    return path

print('✅ Génération des cartes de vigilance...')
CARTE_PATHS = [generer_carte_jour(_j_idx) for _j_idx in range(len(JOURS_CFG))]

# ================================================================
# PARTIE 2 — EXTRACTION DES CUMULS ET MÉTÉOROGRAMMES
# ================================================================
_PALETTE_JOURS = ['#2E7D32','#B71C1C','#E65100','#1565C0']
JOURS_NM  = [j['label'] for j in JOURS_CFG]
JOURS_CLR = [_PALETTE_JOURS[i % len(_PALETTE_JOURS)] for i in range(len(JOURS_CFG))]

# ── Extraction des cumuls journaliers 24h (Lien direct C7) ────────
def extraire_totaux(nom):
    """Extrait les totaux 24h calculés par C7 de façon souple et robuste."""
    rows = []
    nom_target = norm_txt(nom)

    details_source = None
    if 'VILLE_DETAILS' in globals() and VILLE_DETAILS:
        details_source = next((v for k, v in VILLE_DETAILS.items() if norm_txt(k) == nom_target), None)

    if details_source is None and 'COMMUNE_REF_CITY_DETAILS' in globals() and COMMUNE_REF_CITY_DETAILS:
        details_source = next((v for k, v in COMMUNE_REF_CITY_DETAILS.items() if norm_txt(k) == nom_target), None)

    for j_idx in range(CONFIG.get('nb_jours_cartes', 7)):
        jour_date = date_lundi_local + datetime.timedelta(days=j_idx)
        label_j = JOURS_NM[j_idx] if j_idx < len(JOURS_NM) else f'J+{j_idx+1}'

        pluie_val, tmax_val, tmin_val = 0.0, 30.0, 24.0

        if details_source and j_idx < len(details_source):
            det = details_source[j_idx]
            p = det.get('pluie_24h', 0.0)
            if p == p and p is not None: pluie_val = float(p)
            tm = det.get('tmax', 30.0)
            if tm == tm and tm is not None: tmax_val = float(tm)
            tn = det.get('tmin', tmax_val - 6.0)
            if tn == tn and tn is not None: tmin_val = float(tn)

        rows.append({
            'jour':  jour_date.strftime('%d/%m'),
            'label': label_j,
            'tmax':  round(tmax_val, 1),
            'tmin':  round(tmin_val, 1),
            'pluie': round(max(0.0, pluie_val), 1),
        })
    return rows

def extraire_serie_3h_full(nom):
    if 'data' not in globals() or not data: return None
    nom_target = norm_txt(nom)
    data_key = next((k for k in data.keys() if norm_txt(k) == nom_target), None)
    if not data_key: return None

    df = data[data_key].copy()
    if 'time' not in df.columns: return None
    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values('time').reset_index(drop=True)
    df = df[(df['time'] >= PERIODE_DEBUT) & (df['time'] < PERIODE_FIN)].copy()
    if df.empty: return None

    if nom in AJUST_METEOGRAMMES:
        ajust_dict = AJUST_METEOGRAMMES[nom]
        for timestamp_str_raw, adjustments in ajust_dict.items():
            match = re.search(r'>([^<]+)<', str(timestamp_str_raw))
            timestamp_str = match.group(1).strip() if match else str(timestamp_str_raw).strip()
            ts = pd.Timestamp(timestamp_str)
            idx = df[df['time'] == ts].index
            if not idx.empty:
                for param, value in adjustments.items():
                    if param == 'temperature_c': df.loc[idx, 't2m_C'] = value
                    elif param == 'pluie_mm':    df.loc[idx, 'tp_mm'] = value
                    elif param == 'humidite_pct': df.loc[idx, 'rh_pct'] = value

    return df if not df.empty else None

def generer_un_meteogramme(nom, dept, meteo_data_for_plot=None):
    df_full = extraire_serie_3h_full(nom)

    fig, ax = plt.subplots(figsize=(10, 4), dpi=200)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax2 = ax.twinx()

    if df_full is not None and not df_full.empty:
        times = df_full['time'].tolist()
        t2m   = df_full['t2m_C'].tolist() if 't2m_C' in df_full else [np.nan]*len(times)
        tp    = df_full['tp_mm'].tolist()  if 'tp_mm' in df_full else [0.0]*len(times)

        x_num = mdates.date2num(times)
        w = (3 / 24) * 0.65

        bars = ax.bar(x_num, tp, width=w, color='#90CAF9', alpha=0.80, zorder=2, label='Pluie 3h (mm)', edgecolor='#1565C0', linewidth=0.5)
        for bar, val in zip(bars, tp):
            if val > 0.5:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.15, f'{val:.1f}', ha='center', va='bottom', fontsize=7, color='#1565C0', fontweight='bold')

        ax2.plot(x_num, t2m, 'o-', color='#C62828', lw=2.0, ms=5, markeredgecolor='white', markeredgewidth=0.7, zorder=5, label='T2m (°C)')
        for i,(t,tp_v) in enumerate(zip(times, t2m)):
            if not np.isnan(tp_v):
                ax2.text(mdates.date2num(t), tp_v+0.3, f'{tp_v:.0f}°', ha='center', va='bottom', fontsize=7, color='#C62828', fontweight='bold')

        ax.xaxis.set_major_locator(mdates.HourLocator(byhour=HEURES_AFFICHAGE))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Hh'))
        plt.setp(ax.xaxis.get_majorticklabels(), fontsize=8.5, rotation=45)

        for j,(jnm,clr) in enumerate(zip(JOURS_NM,JOURS_CLR)):
            j_start = pd.Timestamp(date_lundi_local+datetime.timedelta(days=j))
            j_mid   = j_start + datetime.timedelta(hours=9)
            ax.axvline(mdates.date2num(j_start), color=clr, lw=1.2, ls='-', alpha=0.55, zorder=4)
            ax.axvspan(mdates.date2num(j_start), mdates.date2num(j_start+datetime.timedelta(hours=24)), color=clr, alpha=0.04, zorder=0)
            ax.text(mdates.date2num(j_mid), -0.14, f"{jnm.split()[0]}\n{(date0+datetime.timedelta(days=j)).strftime('%d/%m')}", fontsize=8, color=clr, fontweight='bold', va='top', ha='center', zorder=10, transform=ax.get_xaxis_transform())

        ax.set_ylim(0, max(max(tp)*2.8, 10))
        t2m_clean = [t for t in t2m if not np.isnan(t)]
        if t2m_clean:
            tr = max(t2m_clean)-min(t2m_clean)
            ax2.set_ylim(min(t2m_clean)-tr*0.5, max(t2m_clean)+tr*0.6)

        ax.set_xlim(mdates.date2num(df_full['time'].min()), mdates.date2num(df_full['time'].max()))

    ax.set_ylabel('Pluie (mm)', fontsize=9, color='#1565C0', labelpad=3)
    ax.tick_params(axis='y', labelsize=8, colors='#1565C0')
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', lw=0.3, alpha=0.4, color='#90CAF9')
    ax2.set_ylabel('Température (°C)', fontsize=9, color='#C62828', labelpad=3)
    ax2.tick_params(axis='y', labelsize=8, colors='#C62828')
    ax2.spines[['top']].set_visible(False)
    ax2.spines[['left','bottom']].set_visible(False)

    ax.set_title(f'{nom}  ({dept})', fontsize=10, fontweight='bold', color='#0C447C', pad=14)

    l1,lb1 = ax.get_legend_handles_labels()
    l2,lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lb1+lb2, fontsize=8, loc='upper left', framealpha=0.85, ncol=3, handlelength=1.2, borderpad=0.3, labelspacing=0.2)

    ax.spines[['left']].set_color('#90CAF9')
    plt.tight_layout(pad=0.6)

    buf = _io.BytesIO()
    plt.savefig(buf, format='png', dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    buf.seek(0)
    return buf.read()

# ================================================================
# INTERFACE INTERACTIVE MULTI-PANNEAUX
# ================================================================

# ── PANNEAU 1 : RELECTURE DES CARTES ──────────────────────────────
w_titre_carte = widgets.HTML(value="""<h3 style="color:#0C447C;margin:4px 0">1. Cartes de Vigilance — Relecture & Correction</h3>""")
w_jour_carte = widgets.ToggleButtons(
    options=[(j['label'], i) for i, j in enumerate(JOURS_CFG)],
    value=0, description='Jour :', style={'button_width': 'auto', 'description_width': '48px'})

w_output_carte = widgets.Output()

def afficher_carte_jour(*args):
    with w_output_carte:
        clear_output(wait=True)
        j_idx = w_jour_carte.value
        if j_idx < len(CARTE_PATHS) and os.path.exists(CARTE_PATHS[j_idx]):
            with open(CARTE_PATHS[j_idx], 'rb') as f:
                display(widgets.Image(value=f.read(), format='png', width=360))

w_jour_carte.observe(afficher_carte_jour, names='value')

w_type_ajust = widgets.Dropdown(options=[('Département', 'dept'), ('Commune', 'commune')], value='dept', description='Zone :', style={'description_width': '48px'})
w_nom_ajust = widgets.Dropdown(options=_NOMS_DEPT, value=_NOMS_DEPT[0], description='Nom :', style={'description_width': '48px'}, layout=widgets.Layout(width='260px'))
w_niveau_ajust = widgets.Dropdown(
    options=[('(ne pas changer)', None)] + [(VIG[i]['label'], i) for i in range(5)],
    value=None, description='Niveau :', style={'description_width': '60px'},
    layout=widgets.Layout(width='220px')
)

# Carreau Symbole élargi et libellé ajusté
w_picto_ajust = widgets.Dropdown(
    options=[('(ne pas changer)', None), ('Auto', 'auto'), ('Pluie', 'pluie'), ('Orage', 'orage'), ('Chaleur', 'chaleur'), ('Vent', 'vent'), ('Aucun', 'aucun')],
    value=None, description='Symbole :', style={'description_width': '65px'},
    layout=widgets.Layout(width='220px')
)
w_appliquer_carte = widgets.Button(description='Appliquer sur la carte', button_style='success')
# BOUTON POUR GÉNÉRER / ACTUALISER TOUTES LES CARTES
w_generer_cartes = widgets.Button(description='🔄 Actualiser toutes les cartes', button_style='primary')
w_msg_ajust_carte = widgets.Output()

def _maj_options_nom(*args):
    w_nom_ajust.options = _NOMS_DEPT if w_type_ajust.value == 'dept' else _NOMS_COMMUNE
    w_nom_ajust.value = w_nom_ajust.options[0]

w_type_ajust.observe(_maj_options_nom, names='value')

def _regenerer_apres_correction(j_idx):
    if 'recalculer_com_vig_picto' in globals(): recalculer_com_vig_picto()
    CARTE_PATHS[j_idx] = generer_carte_jour(j_idx)
    afficher_carte_jour()

def generer_toutes_les_cartes(button=None):
    global CARTE_PATHS
    if 'recalculer_com_vig_picto' in globals(): recalculer_com_vig_picto()
    with w_msg_ajust_carte:
        clear_output(wait=True)
        print("ℹ️ Régénération complète des 7 cartes de vigilance...")
    CARTE_PATHS = [generer_carte_jour(_j_idx) for _j_idx in range(len(JOURS_CFG))]
    afficher_carte_jour()
    with w_msg_ajust_carte:
        clear_output(wait=True)
        print("✅ Toutes les cartes de vigilance ont été actualisées avec succès !")

def appliquer_correction_carte(button):
    type_sel, nom_sel, j_idx = w_type_ajust.value, w_nom_ajust.value, w_jour_carte.value
    niveau_sel, picto_sel = w_niveau_ajust.value, w_picto_ajust.value

    cible_niveau = AJUST_DEPT if type_sel == 'dept' else AJUST_COMMUNE
    cible_picto = AJUST_PICTO_DEPT if type_sel == 'dept' else AJUST_PICTO_COMMUNE

    if nom_sel not in cible_niveau: cible_niveau[nom_sel] = [None] * CONFIG.get('nb_jours_cartes', 7)
    if nom_sel not in cible_picto:  cible_picto[nom_sel]  = [None] * CONFIG.get('nb_jours_cartes', 7)

    if niveau_sel is not None: cible_niveau[nom_sel][j_idx] = niveau_sel
    if picto_sel is not None:  cible_picto[nom_sel][j_idx] = picto_sel

    _regenerer_apres_correction(j_idx)
    with w_msg_ajust_carte:
        clear_output(wait=True)
        print(f"✅ Correction enregistrée pour {nom_sel} ({type_sel}). Carte J{j_idx+1} actualisée.")

w_appliquer_carte.on_click(appliquer_correction_carte)
w_generer_cartes.on_click(generer_toutes_les_cartes)

# ── PANNEAU 2 : CUMULS JOURNALIERS 24H ─────────────────────────────
w_titre_cumul = widgets.HTML(value="""<h3 style="color:#0C447C;margin:16px 0 4px 0">2. Cumuls Journaliers (24h) — Ajustement par Ville/Commune</h3>""")

w_ville_cumul = widgets.Dropdown(
    options=_NOMS_COMMUNE,
    value=_NOMS_COMMUNE[0],
    description='Ville :', style={'description_width':'55px'}, layout=widgets.Layout(width='240px'))

w_tableau_cumul = widgets.VBox([])
w_appliquer_cumul = widgets.Button(description='Sauvegarder les Cumuls 24h', button_style='info')
w_msg_cumul = widgets.Output()

def update_cumul_widgets(*args):
    ville_sel = w_ville_cumul.value
    rows = extraire_totaux(ville_sel)
    widgets_list = []
    for j_idx, r in enumerate(rows):
        widgets_list.append(widgets.HBox([
            widgets.HTML(f"<b style='width:75px;display:inline-block;'>{r['label']} ({r['jour']})</b>"),
            widgets.FloatText(description='Pluie 24h (mm)', value=r['pluie'], step=0.5, layout=widgets.Layout(width='160px'), style={'description_width':'90px'}),
            widgets.FloatText(description='Tmax °C', value=r['tmax'], step=0.5, layout=widgets.Layout(width='140px'), style={'description_width':'60px'}),
            widgets.FloatText(description='Tmin °C', value=r['tmin'], step=0.5, layout=widgets.Layout(width='140px'), style={'description_width':'60px'}),
        ]))
    w_tableau_cumul.children = widgets_list

def appliquer_cumul_ajustements(button):
    ville_sel = w_ville_cumul.value
    nom_target = norm_txt(ville_sel)

    target_dict = None
    if 'VILLE_DETAILS' in globals() and VILLE_DETAILS:
        k_found = next((k for k in VILLE_DETAILS.keys() if norm_txt(k) == nom_target), None)
        if k_found: target_dict = VILLE_DETAILS[k_found]

    if target_dict is None and 'COMMUNE_REF_CITY_DETAILS' in globals() and COMMUNE_REF_CITY_DETAILS:
        k_found = next((k for k in COMMUNE_REF_CITY_DETAILS.keys() if norm_txt(k) == nom_target), None)
        if k_found: target_dict = COMMUNE_REF_CITY_DETAILS[k_found]

    if target_dict:
        for j_idx, hbox in enumerate(w_tableau_cumul.children):
            if j_idx < len(target_dict):
                target_dict[j_idx]['pluie_24h'] = float(hbox.children[1].value)
                target_dict[j_idx]['tmax']      = float(hbox.children[2].value)
                target_dict[j_idx]['tmin']      = float(hbox.children[3].value)

    if 'recalculer_com_vig_picto' in globals():
        recalculer_com_vig_picto()

    with w_msg_cumul:
        clear_output(wait=True)
        print(f"✅ Cumuls 24h enregistrés pour {ville_sel}. Vigilances recalculées.")

w_ville_cumul.observe(update_cumul_widgets, names='value')
w_appliquer_cumul.on_click(appliquer_cumul_ajustements)

# ── PANNEAU 3 : MÉTÉOROGRAMMES PAS 6H & GÉNÉRATION ─────────────────────
w_titre_meteo = widgets.HTML(value="""<h3 style="color:#0C447C;margin:16px 0 4px 0">3. Météorogrammes (Pas 6h) & Génération Finale</h3>""")

w_ville_meteo = widgets.Dropdown(
    options=_NOMS_COMMUNE,
    value=_NOMS_COMMUNE[0],
    description='Ville :', style={'description_width':'55px'}, layout=widgets.Layout(width='240px'))

w_jour_meteo = widgets.ToggleButtons(
    options=[(f'J{i+1}', i) for i in range(CONFIG.get('nb_jours_cartes', 7))],
    value=0, description='Jour :', style={'button_width':'45px','description_width':'48px'})

w_output_meteo = widgets.Output()
w_valeurs_heure = widgets.VBox([])
w_appliquer_3h = widgets.Button(description='Appliquer ajustements 6h', button_style='success')
w_generer_tout = widgets.Button(description='✅ GÉNÉRER TOUS LES MÉTÉOROGRAMMES (77 COMMUNES)', button_style='primary')

def update_meteogram_plot():
    with w_output_meteo:
        clear_output(wait=True)
        selected_ville = w_ville_meteo.value
        dept_candidates = [d for v,d in VILLES_METEO if v==selected_ville]
        selected_dept = dept_candidates[0] if dept_candidates else 'Littoral'
        img_data = generer_un_meteogramme(selected_ville, selected_dept)
        display(widgets.Image(value=img_data, format='png', width=680))

def get_hourly_data_for_display(ville, j_idx):
    df_full = extraire_serie_3h_full(ville)
    if df_full is None or df_full.empty:
        return pd.DataFrame()

    # Délimitation stricte de la journée (ex: du J_idx à 00h au lendemain 00h)
    day_start_local = date_lundi_local + datetime.timedelta(days=j_idx)
    day_end_local = day_start_local + datetime.timedelta(days=1)

    df_day = df_full[(df_full['time'] >= day_start_local) & (df_full['time'] < day_end_local)].copy()
    if df_day.empty:
        return pd.DataFrame()

    # Filtrage strict sur les 4 créneaux principaux : 00h, 06h, 12h, 18h
    df_4h = df_day[df_day['time'].dt.hour.isin([0, 6, 12, 18])].copy()

    # Sécurité : au cas où les heures locales ont un décalage, on prend au maximum 4 pas espacés de 6h
    if df_4h.empty:
        df_4h = df_day.iloc[::2].head(4)

    return df_4h[['time', 't2m_C', 'tp_mm', 'rh_pct']].copy()

def update_input_widgets(*args):
    selected_ville = w_ville_meteo.value
    selected_jour_idx = w_jour_meteo.value
    df_day_hourly = get_hourly_data_for_display(selected_ville, selected_jour_idx)

    new_widgets = []
    if df_day_hourly is not None and not df_day_hourly.empty:
        for idx, row in df_day_hourly.iterrows():
            time_str_local = row['time'].strftime('%Y-%m-%d %H:%M')
            t_val = row['t2m_C'] if not pd.isna(row['t2m_C']) else 0.0
            p_val = row['tp_mm'] if not pd.isna(row['tp_mm']) else 0.0
            h_val = row['rh_pct'] if not pd.isna(row['rh_pct']) else 0.0

            new_widgets.append(widgets.HBox([
                widgets.HTML(f"<b style='width:50px;display:inline-block;'>{row['time'].strftime('%H:%M')}</b>"),
                widgets.FloatText(description='T°C', value=round(t_val, 1), step=0.1, layout=widgets.Layout(width='120px')),
                widgets.FloatText(description='Pluie', value=round(p_val, 1), step=0.1, layout=widgets.Layout(width='120px')),
                widgets.FloatText(description='Hum %', value=round(h_val, 1), step=0.1, layout=widgets.Layout(width='120px')),
                widgets.HTML(f"<span style='display:none;'>{time_str_local}</span>")
            ]))
    else:
        new_widgets.append(widgets.HTML("<i style='color:gray;'>Aucune donnée disponible pour cette journée.</i>"))

    w_valeurs_heure.children = new_widgets
    update_meteogram_plot()

def appliquer_ajustements_3h(button):
    selected_ville = w_ville_meteo.value
    if selected_ville not in AJUST_METEOGRAMMES:
        AJUST_METEOGRAMMES[selected_ville] = {}

    for hbox_widget in w_valeurs_heure.children:
        if isinstance(hbox_widget, widgets.HBox) and len(hbox_widget.children) > 4:
            html_span_value = hbox_widget.children[4].value
            match = re.search(r'>([^<]+)<', str(html_span_value))
            time_str_local = match.group(1).strip() if match else str(html_span_value).strip()

            t_val = hbox_widget.children[1].value
            p_val = hbox_widget.children[2].value
            h_val = hbox_widget.children[3].value

            AJUST_METEOGRAMMES[selected_ville][time_str_local] = {
                'temperature_c': t_val,
                'pluie_mm': p_val,
                'humidite_pct': h_val
            }
    update_meteogram_plot()

def generer_tous_meteogrammes(button):
    global METEO_PATHS
    METEO_PATHS = []
    cibles = VILLES_METEO
    print(f"ℹ️ Génération des météorogrammes pour {len(cibles)} communes...")
    for nom, dept in cibles:
        img_data = generer_un_meteogramme(nom, dept)
        fname = f"meteo_{nom.replace(' ', '_').replace('-', '_')}.png"
        path = f"{CONFIG['output_dir']}/{fname}"
        with open(path, 'wb') as f:
            f.write(img_data)
        METEO_PATHS.append(path)
    print(f"\n✅ {len(METEO_PATHS)} météorogrammes générés avec succès.")

w_ville_meteo.observe(update_input_widgets, names='value')
w_jour_meteo.observe(update_input_widgets, names='value')
w_appliquer_3h.on_click(appliquer_ajustements_3h)
w_generer_tout.on_click(generer_tous_meteogrammes)

# Composition finale des 3 panneaux (avec le bouton d'actualisation des cartes)
layout_final = widgets.VBox([
    w_titre_carte, w_jour_carte, w_output_carte,
    widgets.HBox([w_type_ajust, w_nom_ajust]),
    widgets.HBox([w_niveau_ajust, w_picto_ajust]),
    widgets.HBox([w_appliquer_carte, w_generer_cartes]), # <--- Boutons côte à côte
    w_msg_ajust_carte,
    w_titre_cumul, w_ville_cumul, w_tableau_cumul, w_appliquer_cumul, w_msg_cumul,
    w_titre_meteo, widgets.HBox([w_ville_meteo, w_jour_meteo]),
    w_valeurs_heure,
    widgets.HBox([w_appliquer_3h, w_generer_tout]),
    w_output_meteo
])

# Chargement et affichage
afficher_carte_jour()
update_cumul_widgets()
update_input_widgets()

display(layout_final)

Période : 24/09 00h -> 01/10 00h (heure locale du Bénin)
✅ Génération des cartes de vigilance...


ℹ️ Génération des météorogrammes pour 77 communes...

✅ 77 météorogrammes générés avec succès.


In [14]:
import numpy as np
import pandas as pd
import math
import os, datetime
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from shapely.geometry import MultiPolygon, Polygon
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── 1. SÉCURITÉ VARIABLES GLOBALES & CONFIGURATION ───────────────
os.makedirs(CONFIG['output_dir'], exist_ok=True)

NB_JOURS = CONFIG.get('nb_jours_cartes', len(JOURS_CFG) if 'JOURS_CFG' in globals() else 7)
SEUIL_ALERTE_CUMUL = 3  # V3 = Grave. Mettre 4 pour ne garder que Sévère.

if 'SEUILS_CHALEUR' not in globals():
    SEUILS_CHALEUR = (35.0, 37.0, 40.0, 42.0)

if 'VIG_HEX' not in globals():
    VIG_HEX = {0: '#2E7D32', 1: '#2E7D32', 2: '#ECE12B', 3: '#FF9800', 4: '#E53935'}

if 'niveau_vigilance' not in globals():
    def niveau_vigilance(pluie_mm, tmax_c=0.0, vent_kmh=0.0):
        pluie_mm = 0.0 if pd.isna(pluie_mm) else max(float(pluie_mm), 0.0)
        if pluie_mm < 10.0:
            lvl_p = 0
        elif pluie_mm < 25.0:
            lvl_p = 1
        elif pluie_mm < 50.0:
            lvl_p = 2
        elif pluie_mm < 75.0:
            lvl_p = 3
        else: # pluie_mm >= 75.0
            lvl_p = 4
        lvl_t = 3 if tmax_c >= 40 else (2 if tmax_c >= 37 else (1 if tmax_c >= 35 else 0))
        return max(lvl_p, lvl_t)

if 'canon_dept' not in globals():
    def canon_dept(d): return str(d).strip().capitalize()

if 'lisser_grille' not in globals():
    from scipy.ndimage import zoom as _zoom
    def lisser_grille(arr, facteur=4, categorique=True):
        masque_nan = np.isnan(arr)
        arr_rempli = np.where(masque_nan, 0, arr)
        arr_zoom = _zoom(arr_rempli, facteur, order=1)
        masque_zoom = _zoom(masque_nan.astype(float), facteur, order=1) > 0.5
        if categorique: arr_zoom = np.round(arr_zoom)
        arr_zoom = arr_zoom.astype(float)
        arr_zoom[masque_zoom] = np.nan
        return arr_zoom

if 'fill_geom' not in globals():
    def fill_geom(ax, geom, color, ec='#888888', lw=0.3, zorder=2):
        polys = list(geom.geoms) if isinstance(geom, MultiPolygon) else [geom]
        for poly in polys:
            if isinstance(poly, Polygon) and not poly.is_empty:
                x, y = poly.exterior.xy
                ax.fill(x, y, color=color, zorder=zorder, linewidth=0, antialiased=True)

if 'draw_boundary' not in globals():
    def draw_boundary(ax, geom, color, lw, zorder=5):
        polys = list(geom.geoms) if isinstance(geom, MultiPolygon) else [geom]
        for poly in polys:
            if isinstance(poly, Polygon) and not poly.is_empty:
                x, y = poly.exterior.xy
                ax.plot(x, y, color=color, lw=lw, zorder=zorder, solid_capstyle='round', solid_joinstyle='round')

if 'draw_north_arrow' not in globals():
    def draw_north_arrow(ax, x, y, size=0.25, color='#111111'):
        ax.annotate('', xy=(x, y + size), xytext=(x, y), arrowprops=dict(arrowstyle='-|>', color=color, lw=1.6), zorder=50)
        ax.text(x, y + size * 1.18, 'N', ha='center', va='bottom', fontsize=8, fontweight='bold', color=color, zorder=50)

if 'draw_scale_bar' not in globals():
    def draw_scale_bar(ax, x, y, lat_ref, km=100, color='#111111'):
        deg = km / (111.32 * np.cos(np.radians(lat_ref)))
        ax.plot([x, x + deg], [y, y], color=color, lw=2.2, zorder=50, solid_capstyle='butt')
        for val in (x, x + deg):
            ax.plot([val, val], [y - 0.03, y + 0.03], color=color, lw=1.2, zorder=50)
        ax.text(x + deg / 2, y - 0.10, f'{km} km', ha='center', va='top', fontsize=6.5, color=color, zorder=50)

if 'draw_logo_badge' not in globals():
    def draw_logo_badge(ax, x, y, zoom=None): pass

if 'draw_villes_carte' not in globals():
    def draw_villes_carte(ax): pass

if 'draw_commune_labels' not in globals():
    def draw_commune_labels(ax, gdf): pass

_GRID_OK = ('GRID_VIG' in globals() and GRID_VIG is not None
            and 'tp_pas' in globals() and 'at_stack' in globals()
            and 'ws_stack' in globals() and 'temps_tries' in globals())

# ── 2. FONCTIONS DE CALCUL DU CUMUL ───────────────────────────────
def _indices_jour_c8bis(j_idx):
    day_start = PERIODE_DEBUT + pd.Timedelta(days=j_idx)
    day_end = PERIODE_DEBUT + pd.Timedelta(days=j_idx + 1)
    return [i for i, t in enumerate(temps_tries) if day_start <= t < day_end]

def _grille_niveau_cumul(jours_fenetre):
    """Niveau IBF/OMM (0-4) par maille, pluie/Tmax/vent cumulés sur la fenêtre."""
    sel = sorted({i for j in jours_fenetre for i in _indices_jour_c8bis(j)})
    if not sel:
        if 'n_lat' in globals() and 'n_lon' in globals():
            return np.full((n_lat, n_lon), np.nan, dtype=float)
        else:
            return np.array([], dtype=float).reshape(0,0)

    pluie_cumul = tp_pas[sel].sum(axis=0)
    tmax_max = at_stack[sel].max(axis=0)
    vent_max_kmh = (ws_stack[sel] * 3.6).max(axis=0)

    niv_pluie = np.zeros(pluie_cumul.shape, dtype=int)
    niv_pluie[pluie_cumul >= 10.0] = 1
    niv_pluie[pluie_cumul >= 25.0] = 2
    niv_pluie[pluie_cumul >= 50.0] = 3
    niv_pluie[pluie_cumul >= 75.0] = 4

    S1, S2, S3, S4 = SEUILS_CHALEUR
    niv_temp = np.zeros(tmax_max.shape, dtype=int)
    niv_temp[tmax_max >= S1] = 1
    niv_temp[tmax_max >= S2] = 2
    niv_temp[tmax_max >= S3] = 3
    niv_temp[tmax_max >= S4] = 4

    niv_vent = np.zeros(vent_max_kmh.shape, dtype=int)
    niv_vent[vent_max_kmh > 30.0] = 1
    niv_vent[vent_max_kmh >= 45.0] = 2
    niv_vent[vent_max_kmh >= 60.0] = 3
    niv_vent[vent_max_kmh >= 75.0] = 4

    niv = np.maximum(np.maximum(niv_pluie, niv_temp), niv_vent).astype(float)
    if 'GRID_DEPT' in globals() and GRID_DEPT is not None:
        niv[GRID_DEPT == None] = np.nan
    return niv

def pluie_tmax_vent_commune_jour(nom, j_idx):
    det = COMMUNE_REF_CITY_DETAILS.get(nom) if 'COMMUNE_REF_CITY_DETAILS' in globals() else None
    if not det or j_idx >= len(det):
        return 0.0, np.nan, 0.0
    d = det[j_idx]
    return float(d.get('pluie_24h', 0.0) or 0.0), d.get('tmax', np.nan), float(d.get('wind_max', 0.0) or 0.0)

def niveau_individuel_commune_jour(nom, j_idx):
    if 'COM_VIG' in globals():
        return COM_VIG.get(nom, [0] * NB_JOURS)[j_idx]
    return 0

# ── 3. GÉNÉRATION DE LA CARTE DE CUMUL ────────────────────────────
def generer_carte_cumul(jours_fenetre, fenetre_h):
    label_periode = f"{JOURS_CFG[jours_fenetre[0]]['label']} → {JOURS_CFG[jours_fenetre[-1]]['label']}"
    fig = plt.figure(figsize=(5.5, 8.5), dpi=250)
    fig.patch.set_facecolor('white')
    gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[0.045, 0.955],
                           left=0.01, right=0.99, top=0.99, bottom=0.01, hspace=0.01)

    ax_hdr = fig.add_subplot(gs[0])
    ax_hdr.set_facecolor('#4A148C')
    ax_hdr.set_axis_off()
    ax_hdr.text(0.5, 0.5, f"CUMUL {fenetre_h}H — {label_periode}",
                transform=ax_hdr.transAxes, fontsize=7.5,
                fontweight='bold', color='white', va='center', ha='center')

    ax = fig.add_subplot(gs[1])
    ax.set_facecolor('#C8E6F5')
    ax.set_xlim(0.55, 4.05); ax.set_ylim(5.90, 12.55)
    ax.set_axis_off()

    risque_cache = False
    niveau_max_zone = 0
    cmap_vig = ListedColormap([VIG_HEX[i] for i in range(5)])

    if _GRID_OK:
        niv_grid = _grille_niveau_cumul(jours_fenetre)
        if niv_grid is not None and niv_grid.size > 0:
            grille_aff = np.ma.masked_invalid(lisser_grille(niv_grid, facteur=4, categorique=True))
            ax.imshow(grille_aff, extent=GRID_EXTENT, origin='upper',
                      cmap=cmap_vig, vmin=-0.5, vmax=4.5, zorder=2,
                      interpolation='nearest', aspect='auto')

            niv_max_indiv_grid = None
            for j in jours_fenetre:
                g = GRID_VIG[j]
                niv_max_indiv_grid = g if niv_max_indiv_grid is None else np.fmax(niv_max_indiv_grid, g)
            zone_valide = ~np.isnan(niv_grid)
            if zone_valide.any():
                niveau_max_zone = int(np.nanmax(niv_grid[zone_valide]))
                niv_indiv_max_zone = np.nanmax(niv_max_indiv_grid[zone_valide])
                if niveau_max_zone >= SEUIL_ALERTE_CUMUL and niveau_max_zone > niv_indiv_max_zone:
                    risque_cache = True
        else:
            print(f"   ⚠️ Données de grille indisponibles pour le cumul {fenetre_h}h sur {label_periode}.")

    else:
        for _, row in gdf_com_clean.iterrows():
            com = row['commune']
            vals = [pluie_tmax_vent_commune_jour(com, j) for j in jours_fenetre]
            pluie_cumul_com = sum(v[0] for v in vals)
            tmax_vals = [v[1] for v in vals if not np.isnan(v[1])]
            tmax_max_com = max(tmax_vals) if tmax_vals else 0.0
            vent_max_com = max((v[2] for v in vals), default=0.0)
            niv_com = niveau_vigilance(pluie_cumul_com, tmax_max_com, vent_max_com)
            fill_geom(ax, row.geometry, VIG_HEX[niv_com], zorder=2)
            niveau_max_zone = max(niveau_max_zone, niv_com)
            niv_indiv_max_com = max(niveau_individuel_commune_jour(com, j) for j in jours_fenetre)
            if niv_com >= SEUIL_ALERTE_CUMUL and niv_com > niv_indiv_max_com:
                risque_cache = True

    if 'gdf_arr' in globals() and gdf_arr is not None:
        for _, arow in gdf_arr.iterrows():
            draw_boundary(ax, arow.geometry, '#CFCFCF', lw=0.12, zorder=3)
    if 'gdf_com_clean' in globals():
        for _, row in gdf_com_clean.iterrows():
            draw_boundary(ax, row.geometry, '#AAAAAA', lw=0.25, zorder=3)
    if 'gdf_dept' in globals():
        for _, drow in gdf_dept.iterrows():
            draw_boundary(ax, drow.geometry, '#333333', lw=0.90, zorder=5)
        draw_boundary(ax, gdf_dept.dissolve().geometry.iloc[0], '#000000', lw=1.80, zorder=6)

    draw_villes_carte(ax)
    draw_commune_labels(ax, gdf_com_clean if 'gdf_com_clean' in globals() else None)
    draw_north_arrow(ax, 0.75, 11.9, size=0.28)
    draw_scale_bar(ax, 0.75, 6.35, lat_ref=9.3, km=100)
    draw_logo_badge(ax, 3.75, 12.05, zoom=CONFIG.get('logo_zoom_general', 0.035))

    if risque_cache:
        ax.text(0.5, 0.02, "⚠️ Niveau non visible sur les cartes journalières individuelles",
                transform=ax.transAxes, ha='center', va='bottom', fontsize=6.5,
                color='#B71C1C', fontweight='bold',
                path_effects=[pe.withStroke(linewidth=2, foreground='white')])

    fname = f"carte_cumul_{fenetre_h}h_J{jours_fenetre[0]+1}-J{jours_fenetre[-1]+1}.png"
    path = f"{CONFIG['output_dir']}/{fname}"
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    return path, risque_cache, niveau_max_zone

# ── 4. EXECUTION ET BOUCLE DE GÉNÉRATION ──────────────────────────
print('🗺️ Génération des cartes de cumul glissant 48h/72h...')
CARTE_CUMUL_INFO = []
for fenetre in (2, 3):
    for j0 in range(NB_JOURS - fenetre + 1):
        jours_fenetre = list(range(j0, j0 + fenetre))
        path, risque_cache, niveau_max = generer_carte_cumul(jours_fenetre, fenetre * 24)
        label = f"{JOURS_CFG[jours_fenetre[0]]['label']} → {JOURS_CFG[jours_fenetre[-1]]['label']}"
        if path is not None:
            CARTE_CUMUL_INFO.append({
                'path': path, 'fenetre_h': fenetre * 24, 'label': label,
                'risque_cache': risque_cache, 'niveau_cumul': niveau_max
            })
            marque = ' ⚠️ RISQUE CACHÉ (≥ V3)' if risque_cache else ''
            print(f"   ✅ {os.path.basename(path)}  ({label}){marque}")

# ── 5. PANNEAU INTERACTIF DE RELECTURE ────────────────────────────
w_choix_cumul = widgets.Dropdown(
    options=[(f"{c['fenetre_h']}h — {c['label']}" + (' ⚠️' if c['risque_cache'] else ''), i)
             for i, c in enumerate(CARTE_CUMUL_INFO)],
    description='Fenêtre :', style={'description_width': '70px'}, layout=widgets.Layout(width='420px'))
w_output_cumul = widgets.Output()

def afficher_carte_cumul(*args):
    with w_output_cumul:
        clear_output(wait=True)
        if not CARTE_CUMUL_INFO:
            print("Aucune carte de cumul glissant à afficher.")
            return
        info = CARTE_CUMUL_INFO[w_choix_cumul.value]
        with open(info['path'], 'rb') as f:
            display(widgets.Image(value=f.read(), format='png', width=360))
        if info['risque_cache']:
            print("⚠️ Ce cumul atteint le niveau Grave/Sévère, non visible sur les cartes journalières.")

w_choix_cumul.observe(afficher_carte_cumul, names='value')
display(widgets.VBox([
    widgets.HTML("<h3 style='color:#0C447C;margin:4px 0'>Cartes de cumul glissant (48h / 72h)</h3>"),
    w_choix_cumul, w_output_cumul
]))

if CARTE_CUMUL_INFO:
    afficher_carte_cumul()
else:
    with w_output_cumul:
        print("Aucune carte de cumul glissant à afficher car aucune donnée n'était disponible.")

# ── 6. SYNTHÈSE POUR CANEVAS POWERPOINT ───────────────────────────
FENETRES_A_RISQUE = [c for c in CARTE_CUMUL_INFO if c['risque_cache']]

if FENETRES_A_RISQUE:
    n = len(FENETRES_A_RISQUE)
    n_cols = min(3, n)
    n_rows = math.ceil(n / n_cols)
    fig_s, axes_s = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 4.6 * n_rows), dpi=200)
    fig_s.patch.set_facecolor('white')
    axes_s = np.atleast_1d(axes_s).flatten()
    for i, info in enumerate(FENETRES_A_RISQUE):
        img = plt.imread(info['path'])
        axes_s[i].imshow(img)
        axes_s[i].set_axis_off()
        axes_s[i].set_title(f"{info['fenetre_h']}h — {info['label']}", fontsize=9, fontweight='bold', color='#B71C1C')
    for k in range(n, len(axes_s)):
        axes_s[k].set_axis_off()
    plt.tight_layout()
    CARTE_CUMUL_SYNTHESE_PATH = f"{CONFIG['output_dir']}/carte_synthese_cumul_glissant.png"
    plt.savefig(CARTE_CUMUL_SYNTHESE_PATH, dpi=250, bbox_inches='tight', facecolor='white')
    plt.close()
    noms_fenetres = ', '.join(f"{i['fenetre_h']}h ({i['label']})" for i in FENETRES_A_RISQUE)
    CUMUL_GLISSANT_COMMENTAIRE = (f"{len(FENETRES_A_RISQUE)} fenêtre(s) cumulée(s) atteignent le niveau Grave/Sévère "
                                   f"non visible jour par jour : {noms_fenetres}.")
    print(f"⚠️ {CUMUL_GLISSANT_COMMENTAIRE}")
else:
    CARTE_CUMUL_SYNTHESE_PATH = None
    CUMUL_GLISSANT_COMMENTAIRE = "Aucun risque cumulatif caché (Grave/Sévère) détecté sur cette échéance (48h/72h)."
    print(f"✅ {CUMUL_GLISSANT_COMMENTAIRE}")

🗺️ Génération des cartes de cumul glissant 48h/72h...
   ✅ carte_cumul_48h_J1-J2.png  (Jeudi 24 septembre → Vendredi 25 septembre)
   ✅ carte_cumul_48h_J2-J3.png  (Vendredi 25 septembre → Samedi 26 septembre)
   ✅ carte_cumul_48h_J3-J4.png  (Samedi 26 septembre → Dimanche 27 septembre) ⚠️ RISQUE CACHÉ (≥ V3)
   ✅ carte_cumul_48h_J4-J5.png  (Dimanche 27 septembre → Lundi 28 septembre)
   ✅ carte_cumul_48h_J5-J6.png  (Lundi 28 septembre → Mardi 29 septembre)
   ✅ carte_cumul_48h_J6-J7.png  (Mardi 29 septembre → Mercredi 30 septembre) ⚠️ RISQUE CACHÉ (≥ V3)
   ✅ carte_cumul_72h_J1-J3.png  (Jeudi 24 septembre → Samedi 26 septembre)
   ✅ carte_cumul_72h_J2-J4.png  (Vendredi 25 septembre → Dimanche 27 septembre) ⚠️ RISQUE CACHÉ (≥ V3)
   ✅ carte_cumul_72h_J3-J5.png  (Samedi 26 septembre → Lundi 28 septembre) ⚠️ RISQUE CACHÉ (≥ V3)
   ✅ carte_cumul_72h_J4-J6.png  (Dimanche 27 septembre → Mardi 29 septembre) ⚠️ RISQUE CACHÉ (≥ V3)
   ✅ carte_cumul_72h_J5-J7.png  (Lundi 28 septembre → Mercredi 

⚠️ 6 fenêtre(s) cumulée(s) atteignent le niveau Grave/Sévère non visible jour par jour : 48h (Samedi 26 septembre → Dimanche 27 septembre), 48h (Mardi 29 septembre → Mercredi 30 septembre), 72h (Vendredi 25 septembre → Dimanche 27 septembre), 72h (Samedi 26 septembre → Lundi 28 septembre), 72h (Dimanche 27 septembre → Mardi 29 septembre), 72h (Lundi 28 septembre → Mercredi 30 septembre).


# C9 — Carte de synthèse nationale (jour par jour)

S'exécute juste après C8. Produit une grille compacte avec une mini-carte du pays par jour et un commentaire généré automatiquement à partir de `COM_VIG`/`COM_PICTO`, réutilisée en page 1 du bulletin Word (C11).

In [15]:
# ════════════════════════════════════════════════════════════════
# C9 — CARTE DE SYNTHÈSE NATIONALE (vue d'ensemble jour par jour)
#
# S'exécute juste après C8 (dont elle réutilise gdf_com_clean,
# COM_VIG, COM_PICTO, VIG_HEX, fill_geom, draw_boundary).
#
# Produit :
#   - carte_synthese_nationale.png : une grille compacte avec une
#     mini-carte du pays par jour (communes colorées par niveau IBF)
#   - SYNTHESE_COMMENTAIRES : liste de phrases (une par jour),
#     générées automatiquement à partir de COM_VIG/COM_PICTO,
#     réutilisable telle quelle dans le bulletin Word (C11).
# ════════════════════════════════════════════════════════════════

import math
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# Define draw_vigilance_legend function here
def draw_vigilance_legend(ax, x, y, width=0.55, height=1.5, orientation='vertical', fontsize=7, border_lw=0.5):
    # Assumes VIG_HEX and VIG_LBL are globally accessible (from C3)
    num_levels = len(VIG_HEX) # Typically 0-4, so 5 levels

    if orientation == 'vertical':
        box_height = height / num_levels
        for i in range(num_levels -1, -1, -1): # Draw from 4 down to 0 for proper stacking
            color = VIG_HEX[i]
            label = VIG_LBL[i]
            rect = mpatches.Rectangle((x, y + i * box_height), width, box_height,
                                      facecolor=color, edgecolor='black', linewidth=0.5, zorder=10)
            ax.add_patch(rect)
            ax.text(x + width / 2, y + i * box_height + box_height / 2, label,
                    ha='center', va='center', fontsize=fontsize, color='black', zorder=11,
                    path_effects=[pe.withStroke(linewidth=1, foreground='white')])
    elif orientation == 'horizontal':
        box_width = width / num_levels
        for i in range(num_levels): # Draw from 0 up to 4
            color = VIG_HEX[i]
            label = VIG_LBL[i]
            rect = mpatches.Rectangle((x + i * box_width, y), box_width, height,
                                      facecolor=color, edgecolor='black', linewidth=0.5, zorder=10)
            ax.add_patch(rect)
            ax.text(x + i * box_width + box_width / 2, y + height / 2, label,
                    ha='center', va='center', fontsize=fontsize, color='black', zorder=11,
                    path_effects=[pe.withStroke(linewidth=1, foreground='white')])


PHENOMENE_LBL = {
    'orage': 'pluies/orages', 'pluie': 'pluies/orages',
    'chaleur': 'chaleur marquée', 'vent': 'vent fort', None: None,
}

def _departements_a_niveau(jour_idx, seuil=3):
    """Renvoie {niveau: set(départements)} pour les communes atteignant
    au moins `seuil` ce jour-là, et le niveau max atteint dans le pays."""
    par_dept = {}
    niveau_max = 0
    for _, row in gdf_com_clean.iterrows():
        com = row['commune']
        dept = canon_dept(row['dept'])
        niveaux = COM_VIG.get(com, [0] * CONFIG['nb_jours_cartes'])
        n = niveaux[jour_idx] if jour_idx < len(niveaux) else 0
        niveau_max = max(niveau_max, n)
        if n >= seuil:
            par_dept.setdefault(n, set()).add(dept)
    return par_dept, niveau_max


def _phenomene_dominant(jour_idx, depts_concernes):
    """Phénomène le plus fréquent parmi les communes des départements concernés."""
    compte = {}
    for _, row in gdf_com_clean.iterrows():
        dept = canon_dept(row['dept'])
        if dept not in depts_concernes:
            continue
        com = row['commune']
        pictos = COM_PICTO.get(com, [None] * CONFIG['nb_jours_cartes'])
        p = pictos[jour_idx] if jour_idx < len(pictos) else None
        lbl = PHENOMENE_LBL.get(p)
        if lbl:
            compte[lbl] = compte.get(lbl, 0) + 1
    if not compte:
        return None
    return max(compte, key=compte.get)


def generer_commentaire_jour(jour_idx, jour_label):
    """Phrase de synthèse pour un jour donné, à partir de COM_VIG/COM_PICTO."""
    par_dept, niveau_max = _departements_a_niveau(jour_idx, seuil=3)

    if niveau_max < 2:
        return f"{jour_label} : pas de vigilance particulière sur le territoire national."

    if niveau_max < 3:
        return f"{jour_label} : vigilance jaune diffuse ; soyez attentif sur l'ensemble du pays."

    depts_max = sorted(par_dept.get(niveau_max, []))
    phenomene = _phenomene_dominant(jour_idx, set(depts_max))
    lbl_niveau = VIG_LBL.get(niveau_max, '').lower()

    depts_txt = ', '.join(depts_max[:5])
    if len(depts_max) > 5:
        depts_txt += f' (+{len(depts_max) - 5} autres)'

    phrase = f"{jour_label} : vigilance {lbl_niveau} sur {depts_txt}"
    if phenomene:
        phrase += f" ({phenomene})"
    phrase += " ; reste du pays en vigilance jaune ou verte."
    return phrase


print('📊 Génération des commentaires automatiques par jour...')
SYNTHESE_COMMENTAIRES = [
    generer_commentaire_jour(j, JOURS_CFG[j]['label'])
    for j in range(CONFIG['nb_jours_cartes'])
]
for c in SYNTHESE_COMMENTAIRES:
    print('  •', c)


# ── Grille de mini-cartes, une par jour ───────────────────────────
print('🗺️  Génération de la carte de synthèse nationale...')

n_jours = CONFIG['nb_jours_cartes']
n_cols = 2 if n_jours > 1 else 1
n_rows = math.ceil(n_jours / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(9, 5.5 * n_rows), dpi=200)
fig.patch.set_facecolor('white')
axes = np.atleast_1d(axes).flatten()

for j_idx in range(n_jours):
    ax = axes[j_idx]
    jour = JOURS_CFG[j_idx]
    ax.set_facecolor('#C8E6F5')
    ax.set_xlim(0.55, 4.05)
    ax.set_ylim(5.90, 12.55)
    ax.set_axis_off()

    if GRID_VIG is not None:
        from matplotlib.colors import ListedColormap
        cmap_vig = ListedColormap([VIG_HEX[i] for i in range(5)])
        # grille_vig_ajustee() (C7) : mêmes ajustements manuels que sur la
        # carte C8 (avant, la mini-carte de synthèse montrait la grille brute).
        grille_jour = np.ma.masked_invalid(lisser_grille(grille_vig_ajustee(j_idx), facteur=4, categorique=True))
        ax.imshow(grille_jour, extent=GRID_EXTENT, origin='upper',
                  cmap=cmap_vig, vmin=-0.5, vmax=4.5, zorder=2,
                  interpolation='nearest', aspect='auto')
    else:
        for _, row in gdf_com_clean.iterrows():
            com = row['commune']
            n = max(0, min(4, COM_VIG.get(com, [0] * n_jours)[j_idx]))
            fill_geom(ax, row.geometry, VIG_HEX[n], zorder=2)

    if gdf_arr is not None:
        for _, arow in gdf_arr.iterrows():
            draw_boundary(ax, arow.geometry, '#D8D8D8', lw=0.08, zorder=3)

    for _, row in gdf_com_clean.iterrows():
        draw_boundary(ax, row.geometry, '#AAAAAA', lw=0.2, zorder=3)
    for _, drow in gdf_dept.iterrows():
        draw_boundary(ax, drow.geometry, '#333333', lw=0.6, zorder=4)
    draw_boundary(ax, gdf_dept.dissolve().geometry.iloc[0], '#000000', lw=1.2, zorder=5)

    ax.set_title(jour['label'].upper() + (' ⚠' if jour['alert'] else ''),
                 fontsize=10, fontweight='bold', color=jour['hdr'], pad=4)

    # Commentaire sous chaque mini-carte, découpé pour tenir en largeur
    import textwrap
    texte = textwrap.fill(SYNTHESE_COMMENTAIRES[j_idx], width=42)
    ax.text(0.5, -0.03, texte, transform=ax.transAxes, fontsize=6.8,
            ha='center', va='top', color='#333333', wrap=True)

# Masquer les axes en trop si n_jours impair
for k in range(n_jours, len(axes)):
    axes[k].set_axis_off()

# Éléments cartographiques standards : une seule fois pour toute la page
# (logo + nord + échelle + légende vigilance), sur le premier panneau —
# à cette taille de mini-carte, les répéter sur chaque panneau ou ajouter
# les 77 noms de commune rendrait l'ensemble illisible.
if n_jours > 0:
    ax0 = axes[0]
    draw_north_arrow(ax0, 0.75, 11.9, size=0.30)
    draw_scale_bar(ax0, 0.75, 6.35, lat_ref=9.3, km=100)
    draw_logo_badge(ax0, 3.75, 12.05, zoom=CONFIG.get('logo_zoom_synthese', 0.028))

# Légende vigilance partagée, sous la grille
if n_jours > 1:
    ax_leg = axes[1]
    draw_vigilance_legend(ax_leg, 3.35, 12.3, width=0.55, height=1.5)

fig.suptitle(f"VIGILANCE MÉTÉO — SYNTHÈSE NATIONALE  |  {date_str}",
             fontsize=12, fontweight='bold', color='#0C447C', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])

SYNTHESE_IMG_PATH = f"{CONFIG['output_dir']}/carte_synthese_nationale.png"
plt.savefig(SYNTHESE_IMG_PATH, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()

print(f'✅ Carte de synthèse : {SYNTHESE_IMG_PATH}')


📊 Génération des commentaires automatiques par jour...
  • Jeudi 24 septembre : vigilance jaune diffuse ; soyez attentif sur l'ensemble du pays.
  • Vendredi 25 septembre : vigilance risque élevé (grave) sur Alibori (pluies/orages) ; reste du pays en vigilance jaune ou verte.
  • Samedi 26 septembre : vigilance jaune diffuse ; soyez attentif sur l'ensemble du pays.
  • Dimanche 27 septembre : vigilance risque élevé (grave) sur Borgou, Collines, Donga, Plateau (pluies/orages) ; reste du pays en vigilance jaune ou verte.
  • Lundi 28 septembre : vigilance jaune diffuse ; soyez attentif sur l'ensemble du pays.
  • Mardi 29 septembre : pas de vigilance particulière sur le territoire national.
  • Mercredi 30 septembre : vigilance risque élevé (grave) sur Collines (pluies/orages) ; reste du pays en vigilance jaune ou verte.
🗺️  Génération de la carte de synthèse nationale...
✅ Carte de synthèse : /content/cartes_vigilance/carte_synthese_nationale.png


# C9bis — Carte de cumul pluviométrique attendu

S'exécute après C6bis (`GRID_CUMUL_PLUIE`) et C8/C9 (fonctions de décoration, `gdf_com_clean`). Légende continue (colorbar en mm), à la différence des cartes de vigilance qui utilisent la légende à 5 niveaux.

In [16]:
# ════════════════════════════════════════════════════════════════
# C9bis — CARTE DE CUMUL PLUVIOMÉTRIQUE ATTENDU (période complète)
#
# S'exécute après C6bis (GRID_CUMUL_PLUIE) et C8 (gdf_com_clean,
# fonctions de décoration cartographique, gdf_dept).
# Style aligné sur Annexe1_Inondations : titre, flèche nord, échelle,
# logo, et colorbar continue (mm) plutôt que la légende à 5 niveaux
# utilisée pour les cartes de vigilance.
# ════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.colorbar as colorbar

CUMUL_IMG_PATH = None

def draw_colorbar_cumul(fig, ax, im, label='', pad=0.01, width=0.02, y0_offset=0.0, height_scale=1.0):
    """Dessine une colorbar verticale avec un label et un style uniforme.
    pad          : espace horizontal entre la carte et la colorbar (fraction de figure).
    width        : largeur de la colorbar (fraction de figure).
    y0_offset    : décale la colorbar verticalement (positif = vers le haut).
    height_scale : raccourcit/allonge la colorbar (1.0 = pleine hauteur de la carte).
    """
    pos = ax.get_position()
    h = pos.height * height_scale
    y0 = pos.y0 + y0_offset + (pos.height - h) / 2
    cax = fig.add_axes([pos.x1 + pad, y0, width, h])
    cbar = colorbar.ColorbarBase(cax, cmap=im.cmap, norm=im.norm, orientation='vertical')
    cbar.set_ticks(CUMUL_BORNES_MM)   # ← la correction : sans ça, matplotlib ajoute
                                        #   automatiquement un repère à 1200 (borne
                                        #   supérieure du dernier palier ouvert).
    cbar.set_label(label, fontsize=8, color='#333333', labelpad=4)
    cbar.ax.tick_params(labelsize=7, colors='#333333')
    cbar.outline.set_edgecolor('#333333')
    cbar.outline.set_linewidth(0.5)


if GRID_VIG is None or 'GRID_CUMUL_PLUIE' not in globals():
    print('⚠️ Pas de grille disponible (repli Open Data/simulation) — carte de cumul non générée.')
else:
    print('🗺️  Génération de la carte de cumul pluviométrique...')

    fig, ax = plt.subplots(figsize=(6.5, 9.5), dpi=250)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#C8E6F5')
    ax.set_xlim(0.55, 4.05)
    ax.set_ylim(5.90, 12.55)
    ax.set_axis_off()

    cmap_cumul, norm_cumul = cumul_cmap_norm()
    im = ax.imshow(np.ma.masked_invalid(lisser_grille(GRID_CUMUL_PLUIE, facteur=4, categorique=False)), extent=GRID_EXTENT,
                    origin='upper', cmap=cmap_cumul, norm=norm_cumul,
                    zorder=2, interpolation='nearest', aspect='auto')

    if gdf_arr is not None:
        for _, arow in gdf_arr.iterrows():
            draw_boundary(ax, arow.geometry, '#CFCFCF', lw=0.12, zorder=3)
    for _, row in gdf_com_clean.iterrows():
        draw_boundary(ax, row.geometry, '#AAAAAA', lw=0.25, zorder=3)
    for _, drow in gdf_dept.iterrows():
        draw_boundary(ax, drow.geometry, '#333333', lw=0.90, zorder=5)
    draw_boundary(ax, gdf_dept.dissolve().geometry.iloc[0], '#000000', lw=1.80, zorder=6)

    draw_commune_labels(ax, gdf_com_clean)
    draw_villes_carte(ax)
    draw_north_arrow(ax, 0.75, 11.9, size=0.28)
    draw_scale_bar(ax, 0.75, 6.35, lat_ref=9.3, km=100)
    draw_logo_badge(ax, 3.75, 12.05, zoom=CONFIG.get('logo_zoom_general', 0.035))
    draw_colorbar_cumul(fig, ax, im, label='Cumul attendu (mm)', pad=0.08, width=0.020, height_scale=0.85)

    ax.set_title(f"CUMUL PLUVIOMÉTRIQUE ATTENDU\nValide du {PERIODE_DEBUT:%d/%m} au "
                 f"{(PERIODE_DEBUT + pd.Timedelta(days=CONFIG['nb_jours_cartes'])):%d/%m/%Y}",
                 fontsize=11, fontweight='bold', color='#0C447C', pad=8)

    plt.tight_layout()
    CUMUL_IMG_PATH = f"{CONFIG['output_dir']}/carte_cumul_pluviometrique.png"
    plt.savefig(CUMUL_IMG_PATH, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    print(f'✅ Carte de cumul : {CUMUL_IMG_PATH}')

🗺️  Génération de la carte de cumul pluviométrique...
✅ Carte de cumul : /content/cartes_vigilance/carte_cumul_pluviometrique.png


# C9quater — Cumul observé IMERG (7 derniers jours, automatisé)

S'exécute après C3 (`PERIODE_DEBUT`) et C8/C9bis (fonctions de décoration, `gdf_com_clean`). Télécharge automatiquement les précipitations observées via `earthaccess` (NASA Earthdata, compte `makondjou90`) — produit Late Run demi-horaire (GPM_3IMERGHHL, ~14h de latence), le seul assez rapide pour couvrir les 7 derniers jours au moment de l'émission du bulletin.

In [17]:
# ════════════════════════════════════════════════════════════════
# C9quater — CUMUL OBSERVÉ IMERG (7 derniers jours, automatisé)
#
# Réutilise la méthode déjà validée sur un projet précédent
# (certificats météorologiques) : téléchargement via `earthaccess`
# (authentification NASA Earthdata native), avec repli en cascade
# du produit le plus rapide (Late Run demi-horaire, ~14h de latence)
# — le seul assez rapide pour couvrir les 7 derniers jours au
# moment où le bulletin est émis (vendredi/mercredi matin).
#
# S'exécute après C3 (PERIODE_DEBUT) et C8/C9bis (fonctions de
# décoration cartographique, gdf_com_clean, gdf_dept).
# ════════════════════════════════════════════════════════════════

import sys, subprocess
try:
    import earthaccess
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "earthaccess", "xarray", "h5netcdf"])
    import earthaccess

import xarray as xr
import glob as _glob

CUMUL_OBSERVE_IMERG_PATH = None
DOSSIER_IMERG = os.path.join(CONFIG['output_dir'], 'imerg')
os.makedirs(DOSSIER_IMERG, exist_ok=True)

# ── 1. Fenêtre observée : les 7 jours qui précèdent la période de prévision ──
FENETRE_IMERG_DEBUT = PERIODE_DEBUT - pd.Timedelta(days=7)
FENETRE_IMERG_FIN = PERIODE_DEBUT
print(f"🛰️  Cumul observé IMERG : {FENETRE_IMERG_DEBUT:%d/%m/%Y} → {FENETRE_IMERG_FIN:%d/%m/%Y}")

# ── 2. Authentification NASA Earthdata (compte déjà enregistré : makondjou90) ──
# strategy="interactive" persiste les identifiants (~/.netrc) — demandé une
# seule fois, réutilisé automatiquement les fois suivantes.
try:
    _auth = earthaccess.login(strategy="interactive", persist=True)
except Exception as e:
    _auth = None
    print(f"⚠️ Authentification Earthdata impossible ({e}).")

if _auth is None or not getattr(_auth, 'authenticated', False):
    print("⚠️ Pas d'accès IMERG cette fois — la carte de cumul observé restera indisponible.")
else:
    print("✅ Authentifié auprès de NASA Earthdata.")

    # ── 3. Recherche + téléchargement (Late Run demi-horaire, le plus rapide) ──
    periode = (FENETRE_IMERG_DEBUT.strftime('%Y-%m-%d'), FENETRE_IMERG_FIN.strftime('%Y-%m-%d'))
    print(f"🔎 Recherche GPM_3IMERGHHL (Late Run demi-horaire) du {periode[0]} au {periode[1]}...")

    try:
        resultats = earthaccess.search_data(
            short_name="GPM_3IMERGHHL", version="07", temporal=periode,
            bounding_box=(-3.9, -12.6, 3.9, 12.6),  # (ouest, sud, est, nord) — marge Bénin
        )
    except Exception as e:
        resultats = []
        print(f"⚠️ Recherche IMERG échouée : {e}")

    if not resultats:
        print("⚠️ Aucun granule IMERG trouvé pour cette fenêtre — carte de cumul observé non générée.")
    else:
        deja_presents = _glob.glob(os.path.join(DOSSIER_IMERG, '*.HDF5')) + \
                        _glob.glob(os.path.join(DOSSIER_IMERG, '*.nc4'))
        if len(deja_presents) < len(resultats):
            print(f"⬇️  Téléchargement de {len(resultats)} fichier(s) demi-horaires...")
            fichiers = earthaccess.download(resultats, DOSSIER_IMERG)
        else:
            fichiers = deja_presents
            print(f"✅ {len(fichiers)} fichier(s) déjà présents — non retéléchargés.")

        fichiers = [str(f) for f in fichiers]  # earthaccess renvoie parfois des Path

        # ── 4. Ouverture + cumul sur la fenêtre, recadré sur le Bénin ──────
        print("📐 Cumul des précipitations sur la fenêtre (recadrage Bénin)...")
        try:
            cumul_total = None
            for f in fichiers:
                ds = xr.open_dataset(f, group='Grid', engine='h5netcdf' if f.endswith('.HDF5') else None,
                              decode_times=False)
                var = 'precipitation' if 'precipitation' in ds else 'precipitationCal'
                sous_grille = ds[var].sel(lon=slice(-3.9, 3.9), lat=slice(6.0, 12.6))
                # precipitation en mm/h, chaque pas demi-horaire = 0.5h
                lame_mm = sous_grille.squeeze() * 0.5
                cumul_total = lame_mm if cumul_total is None else cumul_total + lame_mm
                ds.close()

            # IMERG : (lon, lat) -> on transpose en (lat, lon) pour cohérence avec nos autres grilles
            cumul_arr = cumul_total.transpose('lat', 'lon').values
            lats_imerg = cumul_total['lat'].values
            lons_imerg = cumul_total['lon'].values
            cumul_arr = np.flipud(cumul_arr) if lats_imerg[0] < lats_imerg[-1] else cumul_arr
            lats_imerg_sorted = np.sort(lats_imerg)[::-1]

            # ── Masquage hors territoire national ───────────────────────
            # La fenêtre de téléchargement/découpage IMERG ci-dessus est
            # volontairement large (marge de recherche autour du Bénin), donc
            # le rectangle lon/lat déborde largement des frontières (Togo,
            # Burkina Faso, Niger, Nigeria dans les coins). Contrairement à
            # la grille ECMWF (déjà masquée par département via GRID_DEPT en
            # C6bis), rien ici ne limite l'affichage au polygone national :
            # sans ce masque, la carte affiche des couleurs bien au-delà de
            # la frontière noire (débordement visible sur le rendu PPT).
            # Même technique de masquage point-dans-polygone que C6bis/C9ter.
            from matplotlib.path import Path as _MplPath
            from shapely.geometry import MultiPolygon as _MultiPolygon, Polygon as _Polygon

            def _mask_polygone_imerg(geom, points):
                polys = list(geom.geoms) if isinstance(geom, _MultiPolygon) else [geom]
                masque = np.zeros(len(points), dtype=bool)
                for poly in polys:
                    if not isinstance(poly, _Polygon) or poly.is_empty:
                        continue
                    chemin = _MplPath(np.asarray(poly.exterior.coords))
                    masque |= chemin.contains_points(points)
                return masque

            LON2D_IMERG, LAT2D_IMERG = np.meshgrid(lons_imerg, lats_imerg_sorted)
            points_imerg = np.column_stack([LON2D_IMERG.ravel(), LAT2D_IMERG.ravel()])
            masque_benin_imerg = _mask_polygone_imerg(
                gdf_dept.dissolve().geometry.iloc[0], points_imerg
            ).reshape(len(lats_imerg_sorted), len(lons_imerg))
            cumul_arr[~masque_benin_imerg] = np.nan

            print(f"✅ Cumul calculé : min {np.nanmin(cumul_arr):.0f} mm, max {np.nanmax(cumul_arr):.0f} mm.")

            # ── 5. Carte (même style que C9bis, mais teinte différente pour
            # bien distinguer "observé" de "prévu") ────────────────────────
            fig, ax = plt.subplots(figsize=(6.5, 9.5), dpi=250)
            fig.patch.set_facecolor('white')
            ax.set_facecolor('#FFFFFF')
            ax.set_xlim(0.55, 4.05)
            ax.set_ylim(5.90, 12.55)
            ax.set_axis_off()

            extent_imerg = [lons_imerg.min(), lons_imerg.max(), lats_imerg_sorted.min(), lats_imerg_sorted.max()]
            cmap_obs, norm_obs = cumul_cmap_norm()
            im = ax.imshow(np.ma.masked_invalid(lisser_grille(cumul_arr, facteur=3, categorique=False)),
                            extent=extent_imerg, origin='upper', cmap=cmap_obs, norm=norm_obs,
                            zorder=2, interpolation='nearest', aspect='auto')

            if gdf_arr is not None:
                for _, arow in gdf_arr.iterrows():
                    draw_boundary(ax, arow.geometry, '#CFCFCF', lw=0.12, zorder=3)
            for _, row in gdf_com_clean.iterrows():
                draw_boundary(ax, row.geometry, '#AAAAAA', lw=0.25, zorder=3)
            for _, drow in gdf_dept.iterrows():
                draw_boundary(ax, drow.geometry, '#333333', lw=0.90, zorder=5)
            draw_boundary(ax, gdf_dept.dissolve().geometry.iloc[0], '#000000', lw=1.80, zorder=6)

            draw_commune_labels(ax, gdf_com_clean)
            draw_villes_carte(ax)
            draw_north_arrow(ax, 0.75, 11.9, size=0.28)
            draw_scale_bar(ax, 0.75, 6.35, lat_ref=9.3, km=100)
            draw_logo_badge(ax, 3.75, 12.05, zoom=CONFIG.get('logo_zoom_general', 0.035))
            draw_colorbar_cumul(fig, ax, im, label='Cumul observé (mm)', pad=0.08, width=0.020, height_scale=0.85)

            ax.set_title(f"CUMUL PLUVIOMÉTRIQUE OBSERVÉ (IMERG)\n"
                         f"{FENETRE_IMERG_DEBUT:%d/%m} au {FENETRE_IMERG_FIN:%d/%m/%Y}",
                         fontsize=11, fontweight='bold', color='#0C447C', pad=8)

            plt.tight_layout()
            CUMUL_OBSERVE_IMERG_PATH = f"{CONFIG['output_dir']}/carte_cumul_observe_imerg.png"
            plt.savefig(CUMUL_OBSERVE_IMERG_PATH, dpi=300, bbox_inches='tight', facecolor='white')
            plt.show()
            plt.close()
            print(f'✅ Carte de cumul observé : {CUMUL_OBSERVE_IMERG_PATH}')

        except Exception as e:
            print(f"⚠️ Traitement IMERG échoué ({e}) — carte de cumul observé non générée.")
            CUMUL_OBSERVE_IMERG_PATH = None


🛰️  Cumul observé IMERG : 17/09/2026 → 24/09/2026
Enter your Earthdata Login username: makondjou90
Enter your Earthdata password: ··········
✅ Authentifié auprès de NASA Earthdata.
🔎 Recherche GPM_3IMERGHHL (Late Run demi-horaire) du 2026-09-17 au 2026-09-24...
⬇️  Téléchargement de 324 fichier(s) demi-horaires...


QUEUEING TASKS | :   0%|          | 0/324 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/324 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/324 [00:00<?, ?it/s]

📐 Cumul des précipitations sur la fenêtre (recadrage Bénin)...
✅ Cumul calculé : min 4 mm, max 217 mm.
✅ Carte de cumul observé : /content/cartes_vigilance/carte_cumul_observe_imerg.png


# C9ter — Cartes zoom par commune >= seuil (arrondissements en évidence)

S'exécute après C6bis (`GRID_VIG_MAX`), C4 (`gdf_arr`) et C8/C9 (`gdf_com_clean`, fonctions de décoration). Génère une carte zoomée par commune ayant atteint au moins CONFIG['seuil_focus_commune'] (3 = Élevé/Grave par défaut) sur la période, avec les arrondissements de cette commune mis en évidence (jointure spatiale centroïde-dans-polygone).

In [18]:
import numpy as np
import pandas as pd
import math
import os, datetime
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from shapely.geometry import MultiPolygon, Polygon
import ipywidgets as widgets
from IPython.display import display, clear_output

# ================================================================
# BLOC INTÉGRÉ — FOCUS COMMUNES & MÉTÉOROGRAMME ANCRÉ SUR LES RÉSULTATS VALIDÉS
# ================================================================

# Utilisation d'un dictionnaire temporaire pour gérer les doublons et les niveaux de vigilance
communes_focus_dict = {}
ZOOM_IMG_PATHS = {}  # {(nom_commune, jour_idx): chemin_png}

if GRID_VIG is None:
    print('⚠️ Pas de grille disponible (repli Open Data) — cartes zoom non générées.')
elif 'GRID_META' not in globals() or GRID_META is None:
    print('⚠️ GRID_META indisponible — cartes zoom non générées.')
else:
    print('📐 Attribution des mailles aux communes (jointure spatiale)...')

    lats = GRID_META['lats']; lons = GRID_META['lons']
    LON2D, LAT2D = np.meshgrid(lons, lats)
    n_lat, n_lon = LAT2D.shape
    points_grille = np.column_stack([LON2D.ravel(), LAT2D.ravel()])

    from matplotlib.path import Path as _MplPath
    from shapely.geometry import MultiPolygon as _MultiPolygon, Polygon as _Polygon

    def _mask_polygone(geom, points):
        polys = list(geom.geoms) if isinstance(geom, _MultiPolygon) else [geom]
        masque = np.zeros(len(points), dtype=bool)
        for poly in polys:
            if not isinstance(poly, _Polygon) or poly.is_empty:
                continue
            masque |= _MplPath(np.asarray(poly.exterior.coords)).contains_points(points)
        return masque

    GRID_COMMUNE = np.full((n_lat, n_lon), None, dtype=object)
    for _, crow in gdf_com_clean.iterrows():
        m = _mask_polygone(crow.geometry, points_grille).reshape(n_lat, n_lon)
        GRID_COMMUNE[m] = crow['commune']

    SEUIL_FOCUS = CONFIG.get('seuil_focus_commune', 3)
    _dept_par_commune = {crow['commune']: canon_dept(crow['dept']) for _, crow in gdf_com_clean.iterrows()}

    # ---------------------------------------------------------------------------
    # 1. Ajout des alertes quotidiennes (COM_VIG)
    # ---------------------------------------------------------------------------
    if 'COM_VIG' in globals() and COM_VIG:
        for nom, niveaux_jours in COM_VIG.items():
            for j_idx, niv in enumerate(niveaux_jours):
                if niv is not None and niv >= SEUIL_FOCUS:
                    # Ajouter ou mettre à jour si le niveau actuel est plus élevé
                    key = (nom, j_idx)
                    current_entry = communes_focus_dict.get(key, {'niveau': -1, 'is_cumul': False, 'cumul_label': ''})
                    if float(niv) > current_entry['niveau']:
                        communes_focus_dict[key] = {'dept': _dept_par_commune.get(nom, ''), 'niveau': float(niv), 'is_cumul': False, 'cumul_label': ''}
    else:
        print('⚠️ COM_VIG indisponible — vérifie que C7 a bien tourné avant cette cellule.')

    # ---------------------------------------------------------------------------
    # 2. Ajout des alertes cumulatives (CARTE_CUMUL_INFO)
    #    COMMENTÉ pour ne garder que le focus jour, selon la demande utilisateur.
    # ---------------------------------------------------------------------------
    # if 'CARTE_CUMUL_INFO' in globals() and CARTE_CUMUL_INFO:
    #     for cumul_info in CARTE_CUMUL_INFO:
    #         if cumul_info['risque_cache'] and cumul_info['niveau_cumul'] >= SEUIL_FOCUS:
    #             fenetre_h = cumul_info['fenetre_h']
    #             jours_fenetre_labels = cumul_info['label'].split('→')
    #             start_date_label = jours_fenetre_labels[0].strip()
    #             end_date_label = jours_fenetre_labels[-1].strip()
    #             jours_fenetre_indices = []
    #             for j_iter in JOURS_CFG:
    #                 j_label_cleaned = j_iter['label'].split(' ')[0] + ' ' + ' '.join(j_iter['label'].split(' ')[-2:])
    #                 if start_date_label.split(' ')[0] in j_label_cleaned and start_date_label.split(' ')[-2] in j_label_cleaned:
    #                     start_j_idx = j_iter['idx']
    #                 if end_date_label.split(' ')[0] in j_label_cleaned and end_date_label.split(' ')[-2] in j_label_cleaned:
    #                     end_j_idx = j_iter['idx']
    #
    #             if 'start_j_idx' in locals() and 'end_j_idx' in locals():
    #                 jours_fenetre_indices = list(range(start_j_idx, end_j_idx + 1))
    #
    #             if jours_fenetre_indices:
    #                 niv_grid_cumul = _grille_niveau_cumul(jours_fenetre_indices)
    #                 if niv_grid_cumul is not None and niv_grid_cumul.size > 0:
    #                     for nom_com in gdf_com_clean['commune'].unique():
    #                         masque_com = (GRID_COMMUNE == nom_com)
    #                         if masque_com.any():
    #                             niv_cumul_per_commune = float(np.nanmax(niv_grid_cumul[masque_com]))
    #
    #                             if niv_cumul_per_commune >= SEUIL_FOCUS:
    #                                 for j_idx in jours_fenetre_indices:
    #                                     key = (nom_com, j_idx)
    #                                     current_entry = communes_focus_dict.get(key, {'niveau': -1, 'is_cumul': False, 'cumul_label': ''})
    #                                     if niv_cumul_per_commune > current_entry['niveau'] or (niv_cumul_per_commune >= SEUIL_FOCUS and current_entry['niveau'] < SEUIL_FOCUS):
    #                                         communes_focus_dict[key] = {
    #                                             'dept': _dept_par_commune.get(nom_com, ''),
    #                                             'niveau': niv_cumul_per_commune,
    #                                             'is_cumul': True,
    #                                             'cumul_label': f"{cumul_info['fenetre_h']}h sur {cumul_info['label']}"
    #                                         }

    # Conversion du dictionnaire en liste de tuples pour compatibilité avec le reste du code
    COMMUNES_FOCUS = [
        (com, data['dept'], data['niveau'], j_idx, data['is_cumul'], data['cumul_label'])
        for (com, j_idx), data in communes_focus_dict.items()
    ]
    COMMUNES_FOCUS.sort(key=lambda t: (t[3], t[0])) # Tri par jour, puis par commune

    print(f'✅ {len(COMMUNES_FOCUS)} page(s) focus (commune × jour) ≥ V{SEUIL_FOCUS} sur la période.')

    # ---------------------------------------------------------------------------
    # Météorogramme ancré strictement sur les niveaux validés (COMMUNES_FOCUS / Grille validée)
    # ---------------------------------------------------------------------------
    _communes_focus_uniques = {}
    for nom, dept, niveau, jour_idx, is_cumul, cumul_label in COMMUNES_FOCUS:
        _communes_focus_uniques.setdefault(nom, dept)

    if GRID_SERIES is not None and GRID_META is not None:
        lat0_local = float(GRID_META['lats'][0])
        lon0_local = float(GRID_META['lons'][0])
        dlat_local = float(GRID_META['dlat'])

        # Seuils de pluie minimaux par niveau (alignés sur C3)
        SEUILS_PLUIE_MIN = {0: 0.0, 1: 10.0, 2: 25.0, 3: 50.0, 4: 75.0}

        for nom_com, dept_com in _communes_focus_uniques.items():
            row_com_meteo = gdf_com_clean[gdf_com_clean['commune'] == nom_com].iloc[0]
            geom_com = row_com_meteo.geometry
            masque_com = (GRID_COMMUNE == nom_com) if 'GRID_COMMUNE' in globals() and GRID_COMMUNE is not None else _mask_polygone(geom_com, points_grille).reshape(n_lat, n_lon)

            rows_com = []
            for valid_time in sorted(GRID_SERIES):
                g = GRID_SERIES[valid_time]

                if masque_com.any() and 'tp' in g:
                    tp_grid_mm = np.maximum(g['tp'] * 1000, 0)
                    tp_val = float(np.nanmax(tp_grid_mm[masque_com]))
                else:
                    cx, cy = geom_com.centroid.x, geom_com.centroid.y
                    # Fallback for _interp_simple which is not defined, should likely use _grille_bilineaire
                    if '_grille_bilineaire' in globals():
                        tp_val = max(_grille_bilineaire(g['tp'], lat0_local, lon0_local, dlat_local, cy, cx) * 1000, 0) if 'tp' in g else 0.0
                    else:
                        tp_val = 0.0 # Default to 0 if no interpolation function is available

                cx, cy = geom_com.centroid.x, geom_com.centroid.y
                def _interp(param):
                    if param not in g:
                        return np.nan
                    return _grille_bilineaire(g[param], lat0_local, lon0_local, dlat_local, cy, cx)

                t2 = _interp('2t') - 273.15
                td2 = _interp('2d') - 273.15
                u10, v10 = _interp('10u'), _interp('10v')
                wind = np.sqrt(u10**2 + v10**2) * 3.6 if not (np.isnan(u10) or np.isnan(v10)) else np.nan
                rh = calc_rh_from_t_td(t2, td2)
                at_ressenti = calc_temp_ressentie(t2, td2, wind)

                rows_com.append({
                    'time': valid_time, 't2m_C': round(t2, 1),
                    'td2_C': round(td2, 1) if not np.isnan(td2) else np.nan,
                    'at_C': round(at_ressenti, 1) if not np.isnan(at_ressenti) else np.nan,
                    'tp_cumul_mm': round(tp_val, 2),
                    'wind_kmh': round(wind, 1) if not np.isnan(wind) else np.nan,
                    'rh_pct': round(rh, 1) if not np.isnan(rh) else np.nan
                })

            df_com_meteo = pd.DataFrame(rows_com).sort_values('time').reset_index(drop=True)
            df_com_meteo['tp_mm'] = df_com_meteo['tp_cumul_mm'].diff().fillna(df_com_meteo['tp_cumul_mm'])
            df_com_meteo.loc[df_com_meteo['tp_mm'] < 0, 'tp_mm'] = df_com_meteo.loc[df_com_meteo['tp_mm'] < 0, 'tp_cumul_mm']

            # --- ANCRAGE SUR LE NIVEAU VALIDÉ (QUI A DÉCLENCHÉ LE FOCUS) ---
            # Utilise le communes_focus_dict pour récupérer le niveau de vigilance effectif par jour.
            for j_idx in range(CONFIG['nb_jours_cartes']):
                # Cherche le niveau de vigilance le plus élevé pour ce commune et ce jour
                entry = communes_focus_dict.get((nom_com, j_idx))
                if entry and entry['niveau'] >= 2: # Modéré ou plus (seuils d'ancrage)
                    niv_valide = entry['niveau']

                    day_start = PERIODE_DEBUT + pd.Timedelta(days=j_idx)
                    day_end = PERIODE_DEBUT + pd.Timedelta(days=j_idx + 1)
                    mask_jour = (df_com_meteo['time'] >= day_start) & (df_com_meteo['time'] < day_end)

                    if mask_jour.any():
                        sum_jour = df_com_meteo.loc[mask_jour, 'tp_mm'].sum()
                        min_attendu = SEUILS_PLUIE_MIN.get(int(niv_valide), 0.0)

                        # Si le cumul brut est en deçà du seuil validé par le prévisionniste,
                        # on ajuste proportionnellement pour refléter fidèlement le niveau validé
                        if sum_jour < min_attendu and sum_jour > 0:
                            facteur = min_attendu / sum_jour
                            df_com_meteo.loc[mask_jour, 'tp_mm'] *= facteur
                        elif sum_jour == 0 and min_attendu > 0:
                            # S'il n'y avait pas de pluie brute mais un niveau validé, on place une impulsion sur le pic du jour
                            # Note: ceci est une simplification, un ajustement plus précis pourrait distribuer la pluie.
                            # Pour l'instant, on ajoute juste la quantité manquante au point max.
                            idx_max = df_com_meteo.loc[mask_jour, 'tp_mm'].idxmax()
                            df_com_meteo.loc[idx_max, 'tp_mm'] = min_attendu # Directement au seuil minimal

            df_com_meteo['tp_mm'] = df_com_meteo['tp_mm'].clip(lower=0).round(1)
            data[nom_com] = df_com_meteo

        for nom_com, dept_com in _communes_focus_uniques.items():
            fname_meteo = f"meteo_{nom_com.replace(' ', '_').replace('-', '_')}.png"
            fpath_meteo = os.path.join(CONFIG['output_dir'], fname_meteo)
            try:
                img_data_com = generer_un_meteogramme(nom_com, dept_com)
                with open(fpath_meteo, 'wb') as f:
                    f.write(img_data_com)
                print(f'  ✅ Météorogramme ancré sur le niveau validé pour {nom_com}')
            except Exception as e:
                print(f'  ⚠️ Météorogramme impossible pour {nom_com} ({e})')

    # ---------------------------------------------------------------------------
    # Jointure spatiale arrondissement → commune
    # ---------------------------------------------------------------------------
    gdf_arr_com = None
    if gdf_arr is not None and 'shapeName' in gdf_arr.columns:
        try:
            centroids = gdf_arr.copy()
            centroids['geometry'] = centroids.geometry.centroid
            joined = gpd.sjoin(centroids, gdf_com_clean[['commune', 'geometry']], predicate='within', how='left')
            gdf_arr_com = gdf_arr.copy()
            gdf_arr_com['commune_parent'] = joined['commune'].values
        except Exception:
            gdf_arr_com = None

    # ---------------------------------------------------------------------------
    # Génération des cartes zoom
    # ---------------------------------------------------------------------------
    def generer_carte_zoom(nom_commune, dept, niveau, jour_idx, is_cumul=False, cumul_label='', marge_deg=0.18):
        row_com = gdf_com_clean[gdf_com_clean['commune'] == nom_commune].iloc[0]
        minx, miny, maxx, maxy = row_com.geometry.bounds
        xlim = (minx - marge_deg, maxx + marge_deg)
        ylim = (miny - marge_deg, maxy + marge_deg)
        larg, haut = xlim[1] - xlim[0], ylim[1] - ylim[0]

        fig, ax = plt.subplots(figsize=(7.2, 6.5), dpi=250)
        fig.patch.set_facecolor('white')
        ax.set_facecolor('#FFFFFF')
        ax.set_xlim(xlim); ax.set_ylim(ylim)
        ax.set_axis_off()

        from matplotlib.colors import ListedColormap
        cmap_vig = ListedColormap([VIG_HEX[i] for i in range(5)])
        grille_jour_validee = grille_vig_ajustee(jour_idx) if 'grille_vig_ajustee' in globals() else GRID_VIG[jour_idx]
        grille_commune = np.where(GRID_COMMUNE == nom_commune, grille_jour_validee, np.nan)
        grille = np.ma.masked_invalid(lisser_grille(grille_commune, facteur=4, categorique=True), )
        ax.imshow(grille, extent=GRID_EXTENT, origin='upper', cmap=cmap_vig,
                  vmin=-0.5, vmax=4.5, zorder=2, interpolation='nearest', aspect='auto')

        for _, r in gdf_com_clean.iterrows():
            draw_boundary(ax, r.geometry, '#BBBBBB', lw=0.35, zorder=3)

        if gdf_arr_com is not None:
            arr_commune = gdf_arr_com[gdf_arr_com['commune_parent'] == nom_commune]
            for _, arow in arr_commune.iterrows():
                draw_boundary(ax, arow.geometry, '#0C447C', lw=1.3, zorder=6)
            arr_autres = gdf_arr_com[gdf_arr_com['commune_parent'] != nom_commune]
            for _, arow in arr_autres.iterrows():
                draw_boundary(ax, arow.geometry, '#EEEEEE', lw=0.2, zorder=3)

        if gdf_arr_com is None:
            draw_boundary(ax, row_com.geometry, '#000000', lw=2.2, zorder=7)

        draw_north_arrow(ax, xlim[0] + 0.06 * larg, ylim[1] - 0.10 * haut, size=0.06 * haut)
        draw_logo_badge(ax, xlim[1] - 0.10 * larg, ylim[1] - 0.10 * haut, zoom=CONFIG.get('logo_zoom_focus', 0.03))
        draw_scale_bar(ax, xlim[1] - 0.22 * larg, ylim[1] - 0.20 * haut, lat_ref=(miny + maxy) / 2, km=10)
        draw_vigilance_legend(ax, xlim[0] + 0.06 * larg, ylim[0] + 0.03 * haut, width=0.88 * larg, height=0.045 * haut, orientation='horizontal')

        d0_titre = JOURS_CFG[jour_idx]['debut'] if jour_idx < len(JOURS_CFG) else None
        date_complete = f"{d0_titre.day} {MOIS_FR[d0_titre.month - 1]} {d0_titre.year}" if d0_titre else ''

        if is_cumul:
            title_vig = f"Vigilance CUMULATIVE {VIG_LBL.get(int(niveau), '')} (V{{int(niveau)}}) sur {cumul_label}"
        else:
            title_vig = f"Vigilance {VIG_LBL.get(int(niveau), '')} (V{{int(niveau)}}) au {date_complete}"

        ax.set_title(f"{nom_commune.upper()} ({dept})\n{title_vig}",
                     fontsize=10, fontweight='bold', color='#0C447C', pad=10, loc='center')

        plt.tight_layout()
        slug = nom_commune.lower().replace(' ', '_').replace("'", '')
        path = f"{CONFIG['output_dir']}/carte_zoom_{slug}_j{jour_idx + 1}.png"
        plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        return path

    print(f'🗺️ Génération des cartes zoom ({len(COMMUNES_FOCUS)} page(s))...')
    for nom, dept, niveau, jour_idx, is_cumul, cumul_label in COMMUNES_FOCUS:
        ZOOM_IMG_PATHS[(nom, jour_idx)] = generer_carte_zoom(nom, dept, niveau, jour_idx, is_cumul, cumul_label)
        print(f'  ✅ carte_zoom_{nom.lower().replace(" ", "_")}_j{jour_idx + 1}.png')

    print(f'✅ {len(ZOOM_IMG_PATHS)} carte(s) zoom générée(s).')

📐 Attribution des mailles aux communes (jointure spatiale)...
✅ 10 page(s) focus (commune × jour) ≥ V3 sur la période.
  ✅ Météorogramme ancré sur le niveau validé pour Karimama
  ✅ Météorogramme ancré sur le niveau validé pour Malanville
  ✅ Météorogramme ancré sur le niveau validé pour Bassila
  ✅ Météorogramme ancré sur le niveau validé pour Ifangni
  ✅ Météorogramme ancré sur le niveau validé pour Ketou
  ✅ Météorogramme ancré sur le niveau validé pour Save
  ✅ Météorogramme ancré sur le niveau validé pour Tchaourou
  ✅ Météorogramme ancré sur le niveau validé pour Glazoue
  ✅ Météorogramme ancré sur le niveau validé pour Ouesse
🗺️ Génération des cartes zoom (10 page(s))...
  ✅ carte_zoom_karimama_j2.png
  ✅ carte_zoom_malanville_j2.png
  ✅ carte_zoom_bassila_j4.png
  ✅ carte_zoom_ifangni_j4.png
  ✅ carte_zoom_ketou_j4.png
  ✅ carte_zoom_save_j4.png
  ✅ carte_zoom_tchaourou_j4.png
  ✅ carte_zoom_glazoue_j7.png
  ✅ carte_zoom_ouesse_j7.png
  ✅ carte_zoom_save_j7.png
✅ 10 carte(s) zo

In [19]:
# ════════════════════════════════════════════════════════════════
# C10 — TÉLÉCHARGEMENT : Cartes + Météorogrammes → ZIP
#   Scan direct du dossier output — indépendant des variables
# ════════════════════════════════════════════════════════════════
import zipfile, os, glob

out_dir = CONFIG['output_dir']

# ── Scanner tous les PNG générés ─────────────────────────────────
cartes  = sorted(glob.glob(os.path.join(out_dir, 'carte_*.png')))  # carte_J*, carte_cumul_*, carte_synthese_*
meteos  = sorted(glob.glob(os.path.join(out_dir, 'meteo_*.png')))

all_files = cartes + meteos

print(f'📁 Dossier : {out_dir}')
print(f'\n🗺️  Cartes ({len(cartes)}) :')
for p in cartes:
    sz = os.path.getsize(p)//1024
    print(f'  ✓ {os.path.basename(p):<45} {sz} Ko')

print(f'\n📊 Météorogrammes ({len(meteos)}) :')
for p in meteos:
    sz = os.path.getsize(p)//1024
    print(f'  ✓ {os.path.basename(p):<45} {sz} Ko')

if not all_files:
    print('\n⚠️  Aucun fichier trouvé.')
    print('   → Exécuter C9 et cliquer ✅ GÉNÉRER avant de lancer C10.')
else:
    # ── Créer le ZIP ─────────────────────────────────────────────
    zip_path = os.path.join(out_dir, 'sorties_vigilance.zip')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for p in all_files:
            z.write(p, arcname=os.path.basename(p))

    sz_zip = os.path.getsize(zip_path)//1024
    print(f'\n📦 ZIP : {os.path.basename(zip_path)} ({sz_zip} Ko)')
    print(f'   {len(all_files)} fichiers inclus')

    # ── Téléchargement ────────────────────────────────────────────
    try:
        from google.colab import files
        files.download(zip_path)
        print('\n✅ Téléchargement lancé → vérifier le dossier Téléchargements')
    except Exception as e:
        print(f'\nTéléchargement manuel : {zip_path}')

📁 Dossier : /content/cartes_vigilance

🗺️  Cartes (32) :
  ✓ carte_J1_Jeudi.png                            731 Ko
  ✓ carte_J2_Vendredi.png                         741 Ko
  ✓ carte_J3_Samedi.png                           728 Ko
  ✓ carte_J4_Dimanche.png                         766 Ko
  ✓ carte_J5_Lundi.png                            745 Ko
  ✓ carte_J6_Mardi.png                            731 Ko
  ✓ carte_J7_Mercredi.png                         722 Ko
  ✓ carte_cumul_48h_J1-J2.png                     752 Ko
  ✓ carte_cumul_48h_J2-J3.png                     742 Ko
  ✓ carte_cumul_48h_J3-J4.png                     799 Ko
  ✓ carte_cumul_48h_J4-J5.png                     786 Ko
  ✓ carte_cumul_48h_J5-J6.png                     758 Ko
  ✓ carte_cumul_48h_J6-J7.png                     778 Ko
  ✓ carte_cumul_72h_J1-J3.png                     755 Ko
  ✓ carte_cumul_72h_J2-J4.png                     804 Ko
  ✓ carte_cumul_72h_J3-J5.png                     809 Ko
  ✓ carte_cumul_72h_J4-J6.png  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Téléchargement lancé → vérifier le dossier Téléchargements


In [20]:
import sys, subprocess, copy, os
from google.colab import files # <--- ADDED THIS IMPORT
try:
    from pptx import Presentation
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-pptx"])
    from pptx import Presentation

from pptx.util import Emu
from pptx.dml.color import RGBColor
from pptx.oxml.ns import qn
from pathlib import Path
from PIL import Image as _PILImage

print(f"✅ Source de données avant génération du PPT : {globals().get('SOURCE_DONNEES', 'inconnue')} .")

CANEVAS_PATH = CONFIG.get('canevas_path')
_canevas_candidats = sorted(Path('/content').glob('CANEVAS*.pptx'), key=lambda p: p.stat().st_mtime)

# If CANEVAS_PATH is not explicitly set in CONFIG, try to find an existing one
if not CANEVAS_PATH and _canevas_candidats:
    CANEVAS_PATH = str(_canevas_candidats[0])
    CONFIG['canevas_path'] = CANEVAS_PATH # Update CONFIG for future runs

# If still no CANEVAS_PATH, prompt for upload
if not CANEVAS_PATH:
    print("📂 Aucun fichier 'CANEVAS.pptx' n'a été trouvé. Veuillez le télécharger pour générer le bulletin.")
    try:
        uploaded_canevas = files.upload()
        if uploaded_canevas:
            # Assume the user uploads one file and it's the correct one
            fname = list(uploaded_canevas.keys())[0]
            if fname.endswith('.pptx'):
                with open(Path('/content') / fname, 'wb') as f:
                    f.write(uploaded_canevas[fname])
                CANEVAS_PATH = str(Path('/content') / fname)
                CONFIG['canevas_path'] = CANEVAS_PATH
                print(f"✅ Canevas '{fname}' chargé avec succès !")
            else:
                raise ValueError("Le fichier téléchargé n'est pas un fichier PowerPoint (.pptx).")
        else:
            # User closed the upload dialog without selecting a file
            raise RuntimeError("Aucun fichier n'a été sélectionné pour le canevas.")
    except Exception as e:
        # Catch any exceptions during upload/processing and provide a clearer error
        raise RuntimeError(f"Erreur lors du téléchargement du canevas : {e}. Impossible de générer le bulletin.")

if not CANEVAS_PATH:
    # This final check should theoretically not be hit if the above logic works,
    # but it's a good fail-safe.
    raise RuntimeError("Aucun canevas disponible — impossible de générer le bulletin.")

print(f'✅ Canevas : {CANEVAS_PATH}')

def _fit_dims(img_path, max_w_emu, max_h_emu):
    with _PILImage.open(img_path) as im:
        w_px, h_px = im.size
    ratio = w_px / h_px
    w, h = max_w_emu, max_w_emu / ratio
    if h > max_h_emu:
        h = max_h_emu
        w = h * ratio
    return int(w), int(h)

def replace_text(shape, new_text):
    tf = shape.text_frame
    ref_font = tf.paragraphs[0].runs[0].font if tf.paragraphs[0].runs else None
    lines = new_text.split('\n')
    for p in list(tf.paragraphs[1:]):
        p._p.getparent().remove(p._p)
    p0 = tf.paragraphs[0]
    for r in list(p0.runs[1:]):
        r._r.getparent().remove(r._r)
    if not p0.runs:
        p0.add_run()
    p0.runs[0].text = lines[0]
    for line in lines[1:]:
        p = tf.add_paragraph()
        r = p.add_run()
        r.text = line
        if ref_font is not None:
            r.font.size = ref_font.size
            r.font.bold = ref_font.bold
            r.font.italic = ref_font.italic
            r.font.name = ref_font.name
            try:
                r.font.color.rgb = ref_font.color.rgb
            except Exception:
                pass
        p.alignment = p0.alignment

def replace_picture(slide, shape, new_path):
    if not new_path or not os.path.exists(new_path):
        return None
    left, top, width, height = shape.left, shape.top, shape.width, shape.height
    shape._element.getparent().remove(shape._element)
    w, h = _fit_dims(new_path, width, height)
    dx, dy = (width - w) // 2, (height - h) // 2
    return slide.shapes.add_picture(new_path, left + dx, top + dy, w, h)

def find_text_shape(slide, substr):
    for shp in slide.shapes:
        if shp.has_text_frame and substr in shp.text_frame.text:
            return shp
    return None

def picture_below(slide, anchor_shape, min_width=None):
    ax0, ax1 = anchor_shape.left, anchor_shape.left + anchor_shape.width
    ay_ref = anchor_shape.top + anchor_shape.height / 2
    marge = Emu(200000)
    candidats = []
    for shp in slide.shapes:
        if shp.shape_type != 13:
            continue
        if min_width and shp.width < min_width:
            continue
        cx = shp.left + shp.width / 2
        if (ax0 - marge) <= cx <= (ax1 + marge) and shp.top >= (ay_ref - marge):
            candidats.append(shp)
    candidats.sort(key=lambda s: s.top)
    return candidats

def duplicate_slide(prs, slide_index):
    source = prs.slides[slide_index]
    dest = prs.slides.add_slide(source.slide_layout)
    for shp in list(dest.shapes):
        shp._element.getparent().remove(shp._element)
    for shp in source.shapes:
        dest.shapes._spTree.append(copy.deepcopy(shp._element))

    rid_map = {}
    for old_rid, rel in source.part.rels.items():
        if "image" in rel.reltype:
            new_rid = dest.part.relate_to(
                rel.target_part,
                "http://schemas.openxmlformats.org/officeDocument/2006/relationships/image")
            rid_map[old_rid] = new_rid
    for blip in dest.shapes._spTree.iter(qn('a:blip')):
        embed = blip.get(qn('r:embed'))
        if embed in rid_map:
            blip.set(qn('r:embed'), rid_map[embed])
    return dest

def remove_slide(prs, slide_index):
    xml_slides = prs.slides._sldIdLst
    slides = list(xml_slides)
    rId = slides[slide_index].get(qn('r:id'))
    prs.part.drop_rel(rId)
    xml_slides.remove(slides[slide_index])

def date_complete_fr(ts):
    return f"{ts.day} {MOIS_FR[ts.month - 1]} {ts.year}"

def date_jour_complete_fr(ts):
    return f"{JOURS_FR[ts.weekday()].lower()} {ts.day:02d} {MOIS_FR[ts.month - 1]} {ts.year}"

prs = Presentation(CANEVAS_PATH)
n_slides = len(prs.slides._sldIdLst)

EXPECTED_SLIDES = 11
if n_slides > EXPECTED_SLIDES:
    n_en_trop = n_slides - EXPECTED_SLIDES
    xml_slides = prs.slides._sldIdLst
    for _ in range(n_en_trop):
        slides_restantes = list(xml_slides)
        rId_dernier = slides_restantes[-1].get(qn('r:id'))
        prs.part.drop_rel(rId_dernier)
        xml_slides.remove(slides_restantes[-1])
    n_slides = len(prs.slides._sldIdLst)

# ── 1. PAGE DE GARDE ───────────────────────────────────────────────
s0 = prs.slides[0]
shp_dates = find_text_shape(s0, 'Émis le')
if shp_dates:
    _jour_emission = JOURS_FR[datetime.date.today().weekday()].lower()
    replace_text(shp_dates,
        f"Émis le : {_jour_emission} {date_complete_fr(pd.Timestamp(datetime.date.today()))}\n"
        f"Période de validité : {date_complete_fr(PERIODE_DEBUT)} au {date_complete_fr(PERIODE_FIN)}")

# ── 2. SYNTHÈSE ────────────────────────────────────────────────────
s1 = prs.slides[1]
shp_titre_synth = find_text_shape(s1, 'SYNTHÈSE HEBDOMADAIRE')
if shp_titre_synth:
    replace_text(shp_titre_synth, f"SYNTHÈSE HEBDOMADAIRE — VALIDE DU {date_complete_fr(PERIODE_DEBUT)} AU {date_complete_fr(PERIODE_FIN)}")

shp_cumul_attendu_lbl = find_text_shape(s1, 'CUMUL ATTENDU')
if shp_cumul_attendu_lbl:
    pics = picture_below(s1, shp_cumul_attendu_lbl, min_width=Emu(2000000))
    if pics and 'CUMUL_IMG_PATH' in globals() and CUMUL_IMG_PATH:
        replace_picture(s1, pics[0], CUMUL_IMG_PATH)

shp_cumul_observe_lbl = find_text_shape(s1, 'CUMUL OBSERVÉ')
if shp_cumul_observe_lbl:
    pics_obs = picture_below(s1, shp_cumul_observe_lbl, min_width=Emu(2000000))
    if pics_obs and 'CUMUL_OBSERVE_IMERG_PATH' in globals() and CUMUL_OBSERVE_IMERG_PATH:
        replace_picture(s1, pics_obs[0], CUMUL_OBSERVE_IMERG_PATH)

shp_commentaire = find_text_shape(s1, 'Vigilance de la semaine')
if shp_commentaire and 'SYNTHESE_COMMENTAIRES' in globals() and SYNTHESE_COMMENTAIRES:
    replace_text(shp_commentaire, "Vigilance de la semaine : " + " ".join(SYNTHESE_COMMENTAIRES))

# ── 3. PAGES JOUR ──────────────────────────────────────────────────
if 'JOURS_CFG' in globals() and 'CARTE_PATHS' in globals():
    n_jour_slides = min(len(JOURS_CFG), 7)
    for j_idx in range(n_jour_slides):
        s = prs.slides[2 + j_idx]
        jour = JOURS_CFG[j_idx]
        shp_titre = find_text_shape(s, 'PRÉVISION DU JOUR')
        if shp_titre:
            replace_text(shp_titre, f"PRÉVISION DU JOUR — {date_jour_complete_fr(jour['debut'])}")

        shp_highlight = find_text_shape(s, 'À RETENIR')
        commentaire_jour = SYNTHESE_COMMENTAIRES[j_idx] if 'SYNTHESE_COMMENTAIRES' in globals() and j_idx < len(SYNTHESE_COMMENTAIRES) else ""
        if shp_highlight:
            pics = picture_below(s, shp_highlight, min_width=Emu(2000000))
            replace_text(shp_highlight, f"À RETENIR : {commentaire_jour}")
            if pics and j_idx < len(CARTE_PATHS):
                replace_picture(s, pics[0], CARTE_PATHS[j_idx])

        shp_impacts = find_text_shape(s, 'Impacts potentiels')
        if shp_impacts:
            niv_max_jour = int(np.nanmax(GRID_VIG[j_idx])) if 'GRID_VIG' in globals() and GRID_VIG is not None else 0
            niveau_lbl = VIG_LBL.get(niv_max_jour, 'Pas de risque')
            replace_text(shp_impacts, f"Impacts potentiels\nNiveau du jour : {niveau_lbl}.\n\nMesures ABPC / DRM\nSuivre les instructions associées au niveau atteint.")

# ── 3bis. RISQUE CUMULATIF 48H/72H (SUPPRESSION DE LA LIGNE D'ICÔNES CASSÉES) ──
IDX_CUMUL_TEMPLATE = 9
FENETRES_A_RISQUE = [c for c in CARTE_CUMUL_INFO if c['fenetre_h'] == 72 or c['risque_cache']] if 'CARTE_CUMUL_INFO' in globals() else []

if FENETRES_A_RISQUE:
    n_risque = len(FENETRES_A_RISQUE)
    cumul_slide_indices = [IDX_CUMUL_TEMPLATE]
    for k in range(1, n_risque):
        duplicate_slide(prs, IDX_CUMUL_TEMPLATE)
        new_idx_end = len(prs.slides._sldIdLst) - 1
        target_idx = IDX_CUMUL_TEMPLATE + k
        xml_slides = prs.slides._sldIdLst
        slides_list = list(xml_slides)
        el = slides_list[new_idx_end]
        xml_slides.remove(el)
        xml_slides.insert(target_idx, el)
        cumul_slide_indices.append(target_idx)

    for info, s_idx in zip(FENETRES_A_RISQUE, cumul_slide_indices):
        s = prs.slides[s_idx]
        niv_i = int(info['niveau_cumul'])

        # --- NETTOYAGE PROPRE : SUPPRESSION DE LA LIGNE D'ICÔNES CASSÉES DU TABLEAU ---
        for shp in s.shapes:
            if shp.has_table:
                table = shp.table
                if len(table.rows) >= 6:
                    tr = table.rows[0]._tr
                    tr.getparent().remove(tr)
                break

        shp_titre_c = find_text_shape(s, 'RISQUE CUMULATIF')
        if shp_titre_c:
            replace_text(shp_titre_c, f"RISQUE CUMULATIF {info['fenetre_h']}H — {info['label']}")

        shp_retenir_c = find_text_shape(s, 'À RETENIR')
        if shp_retenir_c:
            pics_c = picture_below(s, shp_retenir_c, min_width=Emu(2000000))
            replace_text(shp_retenir_c, f"À RETENIR : le cumul {info['fenetre_h']}h sur {info['label']} atteint la vigilance {VIG_LBL.get(niv_i, '')}, non visible en regardant chaque jour séparément.")
            if pics_c:
                replace_picture(s, pics_c[0], info['path'])

        shp_impacts_c = find_text_shape(s, 'Impacts potentiels')
        if shp_impacts_c:
            replace_text(shp_impacts_c, f"Impacts potentiels\nNiveau du cumul {info['fenetre_h']}h : {VIG_LBL.get(niv_i, '')}.\n\nMesures ABPC / DRM\nSuivre les instructions associées au niveau atteint.")
    n_cumul_slides_final = n_risque
else:
    remove_slide(prs, IDX_CUMUL_TEMPLATE)
    n_cumul_slides_final = 0

DELTA_CUMUL = n_cumul_slides_final - 1

# ── 4. PAGE(S) FOCUS COMMUNE ───────────────────────────────────────
IDX_FOCUS_TEMPLATE = 10 + DELTA_CUMUL
if 'COMMUNES_FOCUS' in globals() and COMMUNES_FOCUS:
    n_focus = len(COMMUNES_FOCUS)
    _seuil_focus_i = int(CONFIG.get('seuil_focus_commune', 3))
    _seuil_focus_lbl = VIG.get(_seuil_focus_i, {}).get('label', '').replace('Risque ', '').upper()

    shp_titre_focus = find_text_shape(prs.slides[IDX_FOCUS_TEMPLATE], 'FOCUS COMMUNES')
    if shp_titre_focus and _seuil_focus_lbl:
        replace_text(shp_titre_focus, f"FOCUS COMMUNES — VIGILANCE ≥ {_seuil_focus_lbl}")

    focus_slide_indices = [IDX_FOCUS_TEMPLATE]
    for _ in range(n_focus - 1):
        duplicate_slide(prs, IDX_FOCUS_TEMPLATE)
        focus_slide_indices.append(len(prs.slides._sldIdLst) - 1)

    for (nom_com, dept_com, niveau_com, jour_idx_com, is_cumul_val, cumul_label_val), s_idx in zip(COMMUNES_FOCUS, focus_slide_indices):
        s = prs.slides[s_idx]
        niv_i = int(niveau_com)
        jour_lbl_com = JOURS_CFG[jour_idx_com]['label'] if jour_idx_com < len(JOURS_CFG) else ''

        # Nettoyage identique de la ligne d'icônes sur les pages focus si le tableau y est présent
        for shp in s.shapes:
            if shp.has_table:
                table = shp.table
                if len(table.rows) >= 6:
                    tr = table.rows[0]._tr
                    tr.getparent().remove(tr)
                break

        # Mise à jour du bandeau de titre commune & département
        for shp in s.shapes:
            if shp.has_text_frame and ('DJOUGOU' in shp.text_frame.text.upper() or 'Vigilance' in shp.text_frame.text):
                bg_hex = VIG_HEX.get(niv_i, '#FFFFFF').replace('#', '')
                try:
                    shp.fill.solid()
                    shp.fill.fore_color.rgb = RGBColor.from_string(bg_hex)
                except Exception:
                    pass
                replace_text(shp, f"{nom_com.upper()} ({dept_com}) — Vigilance {VIG_LBL.get(niv_i, '')} au {jour_lbl_com}")
                break

        # Sélection stricte du corps pour les cartes zoom et météo
        corps_pics = [p for p in s.shapes if p.shape_type == 13 and p.top > Emu(1500000) and p.width > Emu(2000000)]
        corps_pics.sort(key=lambda p: p.left)

        if len(corps_pics) >= 2:
            img_zoom_path = ZOOM_IMG_PATHS.get((nom_com, jour_idx_com))
            if img_zoom_path and os.path.exists(img_zoom_path):
                replace_picture(s, corps_pics[0], img_zoom_path)

            meteo_fname = f"meteo_{nom_com.replace(' ', '_').replace('-', '_')}.png"
            meteo_path = os.path.join(CONFIG['output_dir'], meteo_fname)
            if os.path.exists(meteo_path):
                replace_picture(s, corps_pics[1], meteo_path)

    print(f"✅ {n_focus} page(s) focus commune remplie(s) avec succès.")
else:
    remove_slide(prs, IDX_FOCUS_TEMPLATE)
    print("ℹ️ Aucune commune >= seuil — page focus retirée.")

# ── Sauvegarde et Téléchargement ───────────────────────────────────
out_pptx = os.path.join(CONFIG['output_dir'], 'Prevision_Hebdomadaire_METEO-BENIN_ABPC.pptx')
prs.save(out_pptx)
print(f'\n✅ Bulletin généré : {out_pptx} ({len(prs.slides._sldIdLst)} diapos)')

try:
    from google.colab import files
    files.download(out_pptx)
    print('✅ Téléchargement lancé.')
except Exception as e:
    print(f'⚠️ Téléchargement manuel requis : {out_pptx}')

✅ Source de données avant génération du PPT : ECMWF Dissémination ECPDS (2026-09-24 00Z, 0.1°) + Open Data 0.25° (repli pour 27/09 18h→01/10 00h) .
📂 Aucun fichier 'CANEVAS.pptx' n'a été trouvé. Veuillez le télécharger pour générer le bulletin.


Saving CANEVAS_v7.pptx to CANEVAS_v7.pptx
✅ Canevas 'CANEVAS_v7.pptx' chargé avec succès !
✅ Canevas : /content/CANEVAS_v7.pptx
✅ 10 page(s) focus commune remplie(s) avec succès.

✅ Bulletin généré : /content/cartes_vigilance/Prevision_Hebdomadaire_METEO-BENIN_ABPC.pptx (26 diapos)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Téléchargement lancé.
